In [13]:
!pip install chromadb
!pip install sentence-transformers

     ---------------------------------------- 0.0/52.0 kB ? eta -:--:--
     ----------------------- ---------------- 30.7/52.0 kB ? eta -:--:--
     -------------------------------------- 52.0/52.0 kB 888.2 kB/s eta 0:00:00
   ---------------------------------------- 0.0/23.5 MB ? eta -:--:--
   ---------------------------------------- 0.1/23.5 MB 1.7 MB/s eta 0:00:15
   ---------------------------------------- 0.1/23.5 MB 2.1 MB/s eta 0:00:11
   ---------------------------------------- 0.2/23.5 MB 2.1 MB/s eta 0:00:12
   ---------------------------------------- 0.3/23.5 MB 1.8 MB/s eta 0:00:13
   ---------------------------------------- 0.3/23.5 MB 1.8 MB/s eta 0:00:13
    --------------------------------------- 0.3/23.5 MB 1.3 MB/s eta 0:00:18
    --------------------------------------- 0.4/23.5 MB 1.3 MB/s eta 0:00:18
    --------------------------------------- 0.5/23.5 MB 1.4 MB/s eta 0:00:17
    --------------------------------------- 0.6/23.5 MB 1.4 MB/s eta 0:00:17
   - -------


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# digital_public_safety_platform.py
# ET AI Hackathon 2026 - Digital Public Safety Platform (PS6)
# Notebook 8 - Master Orchestrator (Revision 5)
#
# Mission (one sentence):
# Take a single citizen-reported case from intake to a merged, multi-
# audience intelligence package by calling every specialized engine
# (Notebooks 2-7) in the right order, fusing their outputs into one
# threat score, validating them against each other, and assembling one
# Digital Public Safety Intelligence Package.
#
# REVISION 5 - implements the code-review feedback that is achievable
# from inside Notebook 8 (i.e. without editing Notebooks 2-7, which are
# not available in this repository). What changed vs Revision 4:
#
#   1. Engine Registry - every external engine (Notebook 2-7) is tracked
#      as an EngineMeta record: name, version, loaded, status, health,
#      last inference time. Printed as a table at startup and exposed on
#      the administrator dashboard, instead of a bare boolean.
#   2. Richer failure diagnostics - _run_stage now records status,
#      failure reason, whether a fallback recovered the stage, and CPU
#      time, instead of only "Failed -> stub".
#   3. Standard internal contracts - a CaseContext dataclass carries case
#      + all engine outputs through the pipeline in one object, and a
#      StandardEngineResult dataclass gives every stage the same
#      (success, engine, version, processing_time, confidence, result)
#      shape in master["engine_results"], alongside the existing
#      per-engine keys (kept so nothing downstream breaks).
#   4. Dot-access - process_case() now returns a DigitalPublicSafetyPackage,
#      a dict subclass, so both master["fraud_intelligence"] (old code)
#      and master.engines.fraud (new code) work on the same object.
#   5. Dynamic threat-fusion weights - if counterfeit or geospatial
#      evidence is absent for a case, that slice of the weight is
#      redistributed across the remaining signals instead of being
#      silently zeroed out of a fixed-weight sum.
#   6. Confidence fusion now also folds in an evidence-quality signal
#      (entity density, evidence-type diversity, text length) alongside
#      fraud/network/geo/counterfeit confidence.
#   7. Risk breakdown - financial_risk, victim_risk, national_risk and
#      priority are now reported alongside the single overall_threat_score.
#   8. PII masking - phone numbers, UPI IDs, emails and bank accounts are
#      masked on the citizen- and telecom-facing dashboards. The police
#      and administrator dashboards stay unmasked, since investigators
#      need the real values.
#   9. Concurrency - Notebook 5 (counterfeit) and Notebook 6 (network) run
#      concurrently in a thread pool, since neither depends on the
#      other's output; Notebook 7 (geospatial) still runs after Notebook 6
#      because it needs the network package's campaign/community/mule
#      links.
#  10. SQLite-backed case registry - master packages are persisted to
#      /tmp/notebook8_registry.db so total-case counts survive a process
#      restart. Falls back to an in-memory-only registry (with a logged
#      warning) if SQLite is unavailable for any reason.
#  11. Administrator dashboard - adds CPU time, peak memory, slowest/
#      fastest stage, and failure/fallback counts.
#  12. Explainability flowchart - a compact arrow-chain string
#      (Evidence -> Entities -> Fraud -> Network -> Geo -> Fusion ->
#      Decision) is generated alongside the existing detailed chain.
#  13. Startup banner + live checklist, plain-text markers only (no
#      emoji), for a cleaner terminal demo.
#  14. Report gauge - a threat-score gauge is added to both the plain-
#      text report and, when reportlab's graphics module is available,
#      the PDF report.
#
# What this revision deliberately does NOT do, and why:
#   - It does not change Notebook 2-7's own input/output contracts,
#     versioning, or internals, since those files are not part of this
#     repository. Where the review asked for something inside those
#     files (e.g. "every notebook should expose ENGINE_VERSION"), this
#     file reads that attribute if present (getattr(module,
#     "ENGINE_VERSION", "unknown")) and degrades gracefully to
#     "unknown" if it is not, rather than guessing.
#   - It does not render an actual network graph / geo map image inside
#     the PDF, since those visualizations are Notebook 6/7's job and are
#     not exposed to Notebook 8 in a re-renderable form here. A threat
#     gauge is added instead, which Notebook 8 can build honestly from
#     numbers it already owns.
#   - It does not run Notebook 2/3/4 concurrently with anything, since
#     each is a genuine dependency of the next stage (evidence -> fraud
#     -> decision) and running them concurrently would be fake
#     parallelism.
#
# What this notebook is NOT:
#   - It is not an AI reasoning engine. It performs no fraud
#     classification, no computer vision, no graph analytics, and no
#     geospatial clustering itself - it calls the notebooks that do.
#   - It is not a replacement for Notebooks 2-7. It depends on them.
#
# Design approach - engine adapters:
# Each of Notebooks 2-7 is treated as an independent module. This file
# tries to import the real module first; if that import fails, or if the
# real module raises at call time, it automatically falls back to a
# small, deterministic, clearly-labeled stand-in so the pipeline still
# runs end-to-end and every stage is honestly labeled "real_engine" or
# "stub_adapter" (or "stub_fallback_after_real_engine_error") in the
# audit trail and in the engine registry. Nothing here re-implements
# fraud classification, network analytics, or geospatial clustering as a
# permanent design choice - the stand-ins exist only for graceful
# degradation.

import dataclasses
import hashlib
import json
import logging
import os
import re
import sqlite3
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Dict, List, Optional, Tuple

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)


class PlatformLogger:
    '''
    Central logger (review item #10). Thin wrapper over the standard
    logging module so every stage logs through one place with a
    consistent vocabulary (INFO / WARNING / ERROR / SUCCESS), instead of
    ad hoc logger.warning(...) calls scattered through the file. No
    emoji, no color codes - plain text markers only, so this reads the
    same in a terminal, a CI log, or a judge's screen share.
    '''

    def __init__(self, name: str) -> None:
        self._logger = logging.getLogger(name)

    def info(self, msg: str, *args: Any) -> None:
        self._logger.info(msg, *args)

    def warning(self, msg: str, *args: Any) -> None:
        self._logger.warning(msg, *args)

    def error(self, msg: str, *args: Any) -> None:
        self._logger.error(msg, *args)

    def success(self, msg: str, *args: Any) -> None:
        self._logger.info("[SUCCESS] " + msg, *args)


PLOG = PlatformLogger("digital_public_safety_platform")
logger = PLOG._logger  # kept so any external code importing `logger` directly still works

# ============================================================================
# 1. Engine Availability + Engine Registry - import real Notebook 2-7 modules
# ============================================================================
#
# Every notebook is imported the same way: try the real module first, and
# only fall back to a stand-in if the import itself fails. A second,
# independent fallback also exists at CALL time (see _run_stage and each
# adapter below) in case the real module imports fine but raises during
# execution (e.g. Notebook 2 needs OPENROUTER_API_KEY, Notebook 5 needs an
# actual image file on disk). Both fallback paths are logged, and both
# update the Engine Registry, so nothing ever silently pretends a stub
# result came from a real engine.
#
# NOTE: the warnings you see on a fresh run ("fraud_intelligence_engine.py
# not found", etc.) are expected and harmless if those .py files are not
# sitting next to this file in the same working directory / on sys.path.
# The orchestrator is DESIGNED to keep working in that case (stub mode).
# To get "real_engine" instead of "stub_adapter" in the audit trail,
# place the actual notebook .py files (matching the import names below)
# in the same directory as this file, or on your PYTHONPATH.


@dataclass
class EngineMeta:
    '''Review item #2/#5: replaces a bare boolean with a structured engine record.'''
    key: str
    name: str
    version: str
    loaded: bool
    status: str = "Not run yet"
    health: str = "Unknown"
    last_inference_seconds: Optional[float] = None
    last_updated: Optional[str] = None


ENGINE_REGISTRY: Dict[str, EngineMeta] = {}


def _register_engine(key: str, name: str, module: Any, available: bool) -> None:
    version = getattr(module, "ENGINE_VERSION", "unknown") if module is not None else "unknown"
    ENGINE_REGISTRY[key] = EngineMeta(
        key=key,
        name=name,
        version=str(version),
        loaded=available,
        status="Loaded" if available else "Not installed (stub mode)",
        health="Unknown",
    )


try:
    import fraud_intelligence_engine as notebook2
    _NOTEBOOK2_AVAILABLE = True
except ImportError:
    notebook2 = None
    _NOTEBOOK2_AVAILABLE = False
    PLOG.warning("fraud_intelligence_engine.py not found; Notebook 2 will run in stub mode.")

try:
    import decision_intelligence_engine as notebook3
    _NOTEBOOK3_AVAILABLE = True
except ImportError:
    notebook3 = None
    _NOTEBOOK3_AVAILABLE = False
    PLOG.warning("decision_intelligence_engine.py not found; Notebook 3 will run in stub mode.")

try:
    import digital_evidence_intelligence_engine as notebook4
    _NOTEBOOK4_AVAILABLE = True
except ImportError:
    notebook4 = None
    _NOTEBOOK4_AVAILABLE = False
    PLOG.warning("digital_evidence_intelligence_engine.py not found; Notebook 4 will run in stub mode.")

try:
    import counterfeit_currency_intelligence_engine as notebook5
    _NOTEBOOK5_AVAILABLE = True
except ImportError:
    notebook5 = None
    _NOTEBOOK5_AVAILABLE = False
    PLOG.warning("counterfeit_currency_intelligence_engine.py not found; Notebook 5 will run in stub mode.")

try:
    import fraud_network_intelligence_engine as notebook6
    _NOTEBOOK6_AVAILABLE = True
except ImportError:
    notebook6 = None
    _NOTEBOOK6_AVAILABLE = False
    PLOG.warning("fraud_network_intelligence_engine.py not found; Notebook 6 will run in stub mode.")

try:
    import geospatial_crime_pattern_intelligence_engine as notebook7
    _NOTEBOOK7_AVAILABLE = True
except ImportError:
    notebook7 = None
    _NOTEBOOK7_AVAILABLE = False
    PLOG.warning("geospatial_crime_pattern_intelligence_engine.py not found; Notebook 7 will run in stub mode.")

_register_engine("notebook2", "Fraud Intelligence Engine", notebook2, _NOTEBOOK2_AVAILABLE)
_register_engine("notebook3", "Decision Intelligence Engine", notebook3, _NOTEBOOK3_AVAILABLE)
_register_engine("notebook4", "Digital Evidence Intelligence Engine", notebook4, _NOTEBOOK4_AVAILABLE)
_register_engine("notebook5", "Counterfeit Currency Intelligence Engine", notebook5, _NOTEBOOK5_AVAILABLE)
_register_engine("notebook6", "Fraud Network Intelligence Engine", notebook6, _NOTEBOOK6_AVAILABLE)
_register_engine("notebook7", "Geospatial Crime Pattern Intelligence Engine", notebook7, _NOTEBOOK7_AVAILABLE)

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    from reportlab.lib import colors
    _REPORTLAB_AVAILABLE = True
except ImportError:
    _REPORTLAB_AVAILABLE = False

try:
    from reportlab.graphics.shapes import Drawing, Rect, String
    _REPORTLAB_GRAPHICS_AVAILABLE = _REPORTLAB_AVAILABLE
except ImportError:
    _REPORTLAB_GRAPHICS_AVAILABLE = False

try:
    import resource
    _RESOURCE_AVAILABLE = True
except ImportError:
    _RESOURCE_AVAILABLE = False


class EngineSource(str, Enum):
    '''Marks where a stage's output actually came from.'''
    REAL = "real_engine"
    STUB = "stub_adapter"
    STUB_FALLBACK = "stub_fallback_after_real_engine_error"
    SKIPPED = "skipped"


# ============================================================================
# 2. Configuration
# ============================================================================


class Config:
    NOTEBOOK_VERSION = "v5.0"

    # --- Decision thresholds (stub Notebook 3 fallback only) ---
    DECISION_EMERGENCY_RISK_MIN = 90.0
    DECISION_URGENT_RISK_MIN = 70.0
    DECISION_HUMAN_REVIEW_RISK_MIN = 40.0

    # --- Fraud stub (fallback Notebook 2) ---
    FRAUD_KEYWORD_SCORE_CAP = 60.0
    FRAUD_BASE_SCORE = 20.0

    # --- Severity bands, shared by the fraud stub and the fusion engine
    # so severity labels stay consistent whether Notebook 2 or the stub
    # produced the underlying risk score. ---
    SEVERITY_LOW_MAX = 30
    SEVERITY_MEDIUM_MAX = 60
    SEVERITY_HIGH_MAX = 85

    # --- Counterfeit stub (fallback Notebook 5) ---
    COUNTERFEIT_SUSPICION_THRESHOLD = 0.55

    # --- Threat Fusion Engine BASE weights (must sum to 1.0). These are
    # the weights used when every signal is present; fuse_threat_score()
    # redistributes weight away from any signal that has no evidence for
    # a given case (review item #5), so a case with no location data
    # never gets a free pass on the geo slice, it gets redistributed. ---
    FUSION_WEIGHT_NETWORK_RISK = 0.45
    FUSION_WEIGHT_FRAUD_RISK = 0.20
    FUSION_WEIGHT_GEO_SIGNAL = 0.20
    FUSION_WEIGHT_COUNTERFEIT_SIGNAL = 0.15

    # --- Confidence Fusion BASE weights (must sum to 1.0). Rebalanced
    # from Revision 4 to make room for the new evidence-quality signal. ---
    CONFIDENCE_WEIGHT_FRAUD = 0.30
    CONFIDENCE_WEIGHT_NETWORK = 0.25
    CONFIDENCE_WEIGHT_GEO = 0.15
    CONFIDENCE_WEIGHT_COUNTERFEIT = 0.10
    CONFIDENCE_WEIGHT_EVIDENCE_QUALITY = 0.20

    # --- Hotspot / district priority -> numeric signal (0-100) used only
    # inside the Threat Fusion Engine, never shown to the user directly. ---
    PRIORITY_TO_SCORE = {"Critical": 95.0, "High": 75.0, "Medium": 50.0, "Low": 20.0}

    # --- Feature toggles ---
    ENABLE_CONCURRENT_STAGES = True
    ENABLE_PII_MASKING = True
    SQLITE_REGISTRY_PATH = "/tmp/notebook8_registry.db"

    # --- Field-name constants (review item #9: avoid magic strings for
    # the handful of fields that get read/written in more than one
    # place). A full sweep of every dict-key literal in this file is a
    # mechanical follow-up; these are the highest-traffic ones. ---
    FIELD_FRAUD_TYPE = "fraud_type"
    FIELD_RISK_SCORE = "risk_score"
    FIELD_SEVERITY = "severity"
    FIELD_CONFIDENCE = "confidence"


CONFIG = Config()
assert abs(
    CONFIG.FUSION_WEIGHT_NETWORK_RISK + CONFIG.FUSION_WEIGHT_FRAUD_RISK
    + CONFIG.FUSION_WEIGHT_GEO_SIGNAL + CONFIG.FUSION_WEIGHT_COUNTERFEIT_SIGNAL - 1.0
) < 1e-6, "Threat fusion base weights must sum to 1.0."
assert abs(
    CONFIG.CONFIDENCE_WEIGHT_FRAUD + CONFIG.CONFIDENCE_WEIGHT_NETWORK
    + CONFIG.CONFIDENCE_WEIGHT_GEO + CONFIG.CONFIDENCE_WEIGHT_COUNTERFEIT
    + CONFIG.CONFIDENCE_WEIGHT_EVIDENCE_QUALITY - 1.0
) < 1e-6, "Confidence fusion base weights must sum to 1.0."

PLOG.info(
    "Notebook 8 configuration loaded. version=%s notebook2=%s notebook3=%s notebook4=%s notebook5=%s notebook6=%s notebook7=%s",
    CONFIG.NOTEBOOK_VERSION, _NOTEBOOK2_AVAILABLE, _NOTEBOOK3_AVAILABLE,
    _NOTEBOOK4_AVAILABLE, _NOTEBOOK5_AVAILABLE, _NOTEBOOK6_AVAILABLE, _NOTEBOOK7_AVAILABLE,
)


class OrchestrationError(Exception):
    '''Raised when Notebook 8 cannot produce a valid Digital Public Safety Intelligence Package.'''


# ============================================================================
# 3. Module 1 - Case Intake (input contract)
# ============================================================================


@dataclass
class EvidenceItem:
    '''
    One piece of citizen-submitted evidence.

    evidence_type - a broad label such as "call_recording",
        "whatsapp_screenshot", "document", "currency_image", "text".
    content       - extractable text (a transcript, OCR text, a pasted
        message) where available, OR an image/document path when the
        real file lives on disk (see metadata["file_path"]).
    metadata      - anything evidence-type-specific: file_path,
        original_filename, source_channel, citizen_reported_suspicious,
        override_extracted_text (pre-computed OCR/ASR text from an
        upstream service), etc.
    '''
    evidence_type: str
    content: str = ""
    metadata: Dict[str, Any] = field(default_factory=dict)


@dataclass
class CaseIntake:
    case_id: str
    citizen_name: Optional[str] = None
    victim_id: Optional[str] = None
    timestamp: Optional[str] = None
    city: Optional[str] = None
    state: Optional[str] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    evidence: List[EvidenceItem] = field(default_factory=list)
    priority: str = "Normal"        # citizen/call-center-reported urgency; advisory only
    source: str = "citizen_app"
    amount_involved: float = 0.0


def intake_case(raw: CaseIntake) -> CaseIntake:
    '''Module 1 entry point. Validates minimum required fields and fills sensible defaults.'''
    if not raw.case_id:
        raise OrchestrationError("CaseIntake is missing a case_id; cannot proceed.")
    if not raw.timestamp:
        raw.timestamp = datetime.now(timezone.utc).isoformat()
    if not raw.victim_id:
        raw.victim_id = f"VICTIM-{raw.case_id}"
    PLOG.info("Case intake complete. case_id=%s evidence_items=%d", raw.case_id, len(raw.evidence))
    return raw


@dataclass
class CaseContext:
    '''
    Review item #3 (standard input contract). Rather than passing case,
    evidence_package, fraud_package, network_package, etc. as five
    separate positional arguments between functions, this bundle carries
    everything one case has produced so far through the pipeline. Note:
    Notebooks 2-7 themselves still receive whatever shape they were
    actually written for (we do not control that code); this contract
    standardizes the *internal* handoffs inside Notebook 8.
    '''
    case: CaseIntake
    evidence_package: Optional[Dict[str, Any]] = None
    fraud_package: Optional[Dict[str, Any]] = None
    counterfeit_package: Optional[Dict[str, Any]] = None
    network_package: Optional[Dict[str, Any]] = None
    geo_package: Optional[Dict[str, Any]] = None


@dataclass
class StandardEngineResult:
    '''
    Review item #4 (standard output contract). Every pipeline stage is
    additionally summarized into this uniform shape, stored under
    master["engine_results"][stage_key], alongside the existing
    per-engine dict keys (kept so every downstream reader that already
    expects master["fraud_intelligence"] etc. keeps working unchanged).
    '''
    success: bool
    engine: str
    version: str
    processing_time: float
    confidence: float
    result: Dict[str, Any]
    error_reason: Optional[str] = None
    recovered: bool = False
    fallback_used: bool = False


# ============================================================================
# 4. Module 9 - Case Registry (SQLite-backed, in-memory cache per process)
# ============================================================================
#
# Review item #7: master packages now persist to a small SQLite database
# so the "total cases processed" figure survives a process restart,
# instead of resetting to zero every run. Notebook 6 and 7 still need
# their own native Python record objects (CaseRecord / GeoCaseRecord),
# which are defined inside those notebooks and are not safely
# JSON-serializable in a generic way here, so those two collections stay
# in-memory-only, exactly as before, scoped to a single running process.
# If SQLite is unavailable for any reason (read-only filesystem, etc.)
# the registry degrades to in-memory-only and logs a warning once,
# rather than crashing case processing.


class CaseRegistry:
    def __init__(self, db_path: str = CONFIG.SQLITE_REGISTRY_PATH) -> None:
        self.network_cases: Dict[str, Any] = {}     # case_id -> notebook6.CaseRecord (in-memory only)
        self.geo_cases: Dict[str, Any] = {}          # case_id -> notebook7.GeoCaseRecord (in-memory only)
        self.master_packages: Dict[str, Dict[str, Any]] = {}   # case_id -> final master package (this-run cache)
        self.db_path = db_path
        self._db_available = False
        self._init_db()

    def _init_db(self) -> None:
        try:
            conn = sqlite3.connect(self.db_path)
            conn.execute(
                "CREATE TABLE IF NOT EXISTS master_packages ("
                "case_id TEXT PRIMARY KEY, package_json TEXT, created_at TEXT)"
            )
            conn.commit()
            conn.close()
            self._db_available = True
        except Exception as exc:
            PLOG.warning("SQLite registry unavailable (%s); using in-memory-only registry for this run.", exc)
            self._db_available = False

    def register_network_case(self, case_id: str, record: Any) -> None:
        self.network_cases[case_id] = record

    def register_geo_case(self, case_id: str, record: Any) -> None:
        self.geo_cases[case_id] = record

    def register_master_package(self, case_id: str, package: Dict[str, Any]) -> None:
        self.master_packages[case_id] = package
        if self._db_available:
            try:
                conn = sqlite3.connect(self.db_path)
                conn.execute(
                    "INSERT OR REPLACE INTO master_packages (case_id, package_json, created_at) VALUES (?, ?, ?)",
                    (case_id, json.dumps(package, default=str), datetime.now(timezone.utc).isoformat()),
                )
                conn.commit()
                conn.close()
            except Exception as exc:
                PLOG.warning("Failed to persist case %s to SQLite registry: %s", case_id, exc)

    def all_network_cases(self) -> List[Any]:
        return list(self.network_cases.values())

    def all_geo_cases(self) -> List[Any]:
        return list(self.geo_cases.values())

    def total_cases(self) -> int:
        if self._db_available:
            try:
                conn = sqlite3.connect(self.db_path)
                count = conn.execute("SELECT COUNT(*) FROM master_packages").fetchone()[0]
                conn.close()
                return int(count)
            except Exception as exc:
                PLOG.warning("Failed to read case count from SQLite registry: %s", exc)
        return len(self.master_packages)


CASE_REGISTRY = CaseRegistry()

# ============================================================================
# 5. Module 2 - Evidence Router (smarter, per-entity routing)
# ============================================================================


def route_evidence(case: CaseIntake) -> Dict[str, Any]:
    '''
    Module 2 entry point. Decides which downstream engines need to run
    and WHY, so the routing decision itself is explainable rather than a
    bare set of booleans.
    '''
    types_present = {item.evidence_type for item in case.evidence}
    has_location = case.city is not None or (case.latitude is not None and case.longitude is not None)

    reasons: List[str] = []
    plan = {
        "run_evidence_engine": True,
        "run_fraud_intelligence": True,
        "run_counterfeit_check": "currency_image" in types_present,
        "run_network_intelligence": True,
        "run_geospatial_intelligence": has_location,
        "run_decision_engine": True,
    }

    reasons.append("Evidence engine always runs to normalize whatever evidence was submitted.")
    reasons.append("Fraud intelligence always runs to classify the case.")
    if plan["run_counterfeit_check"]:
        reasons.append("Currency image evidence detected; routing to the Counterfeit Currency Engine.")
    else:
        reasons.append("No currency image evidence present; Counterfeit Currency Engine skipped.")
    reasons.append("Network intelligence always runs so this case joins the shared fraud graph.")
    if plan["run_geospatial_intelligence"]:
        reasons.append("City or GPS coordinates present; routing to the Geospatial Intelligence Engine.")
    else:
        reasons.append("No location data present; Geospatial Intelligence Engine skipped for this case.")
    reasons.append("Decision engine always runs to produce the final stakeholder actions.")

    plan["reasons"] = reasons
    PLOG.info("Evidence routing complete. case_id=%s plan=%s", case.case_id, {k: v for k, v in plan.items() if k != "reasons"})
    return plan


# ============================================================================
# 6. Module 3 - Run Evidence Engine (real Notebook 4 call, stub fallback)
# ============================================================================

_ENTITY_PATTERNS: Dict[str, re.Pattern] = {
    "phone_numbers": re.compile(r"(?:\+91[\-\s]?)?[6-9]\d{9}\b"),
    "upi_ids": re.compile(r"\b[\w.\-]{2,}@[a-zA-Z]{2,}\b"),
    "emails": re.compile(r"\b[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}\b"),
    "bank_accounts": re.compile(r"\b\d{9,18}\b"),
    "urls": re.compile(r"https?://[^\s]+"),
}


def _extract_entities_from_text_stub(text: str) -> Dict[str, List[str]]:
    '''Fallback-only regex entity extraction, used when Notebook 4 is unavailable or errors.'''
    entities: Dict[str, List[str]] = defaultdict(list)
    for entity_type, pattern in _ENTITY_PATTERNS.items():
        for match in pattern.findall(text):
            if match not in entities[entity_type]:
                entities[entity_type].append(match)
    entities["emails"] = [e for e in entities["emails"] if "." in e.split("@", 1)[-1]]
    entities["upi_ids"] = [u for u in entities["upi_ids"] if u not in entities["emails"]]
    return dict(entities)


def _to_notebook4_inputs(case: CaseIntake) -> List[Any]:
    '''Converts our EvidenceItem list into Notebook 4's EvidenceInput contract.'''
    inputs = []
    for item in case.evidence:
        file_path = item.metadata.get("file_path")
        inputs.append(notebook4.EvidenceInput(
            file_path=file_path,
            raw_text=item.content if not file_path else None,
            source_channel=item.metadata.get("source_channel", item.evidence_type),
            submitted_at=item.metadata.get("submitted_at", case.timestamp),
            original_filename=item.metadata.get("original_filename"),
            override_extracted_text=item.metadata.get("override_extracted_text"),
        ))
    return inputs


def run_evidence_engine(case: CaseIntake) -> Dict[str, Any]:
    '''Module 3 entry point (Notebook 4 adapter, real-first with stub fallback).'''
    if _NOTEBOOK4_AVAILABLE:
        try:
            inputs = _to_notebook4_inputs(case)
            package = notebook4.package_case_evidence(inputs, case_id=case.case_id)
            package["engine_source"] = EngineSource.REAL.value
            return package
        except Exception as exc:
            PLOG.warning("Notebook 4 raised at call time; falling back to stub evidence extraction. error=%s", exc)

    combined_text = " ".join(item.content for item in case.evidence if item.content)
    entities = _extract_entities_from_text_stub(combined_text)
    evidence_types_seen = sorted({item.evidence_type for item in case.evidence})

    return {
        "case_id": case.case_id,
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK4_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "evidence_types_seen": evidence_types_seen,
        "metadata": entities,               # aligned with Notebook 4's top-level "metadata" key name
        "extracted_entities": entities,      # kept for backward-compatible readers
        "text": combined_text,
        "evidence_quality": "Unknown",
        "evidence_summary": combined_text[:280] + ("..." if len(combined_text) > 280 else ""),
    }


def _evidence_quality_score(evidence_package: Dict[str, Any]) -> float:
    '''
    Review item #6: a deterministic 0-100 proxy for evidence quality,
    used as a new input to Confidence Fusion. Built from three signals
    Notebook 8 can see directly without needing OCR/ASR internals from
    Notebook 4: how many entities were extracted, how many distinct
    evidence types were submitted, and how much text is actually there
    to reason over.
    '''
    entities = evidence_package.get("metadata") or evidence_package.get("extracted_entities") or {}
    entity_count = sum(len(v) for v in entities.values() if isinstance(v, list))
    text_len = len(evidence_package.get("text", "") or "")
    type_count = len(evidence_package.get("evidence_types_seen", []) or [])
    score = 30.0 + min(30.0, entity_count * 4.0) + min(20.0, text_len / 20.0) + min(20.0, type_count * 7.0)
    return round(min(100.0, score), 1)


# ============================================================================
# 7. Module 4 - Run Fraud Intelligence (real Notebook 2 call, stub fallback)
# ============================================================================

_FRAUD_TYPE_KEYWORDS: Dict[str, List[str]] = {
    "Digital Arrest Scam": ["digital arrest", "rbi", "cbi", "arrest warrant", "video call", "police officer", "customs"],
    "UPI / Payment Fraud": ["upi", "refund", "cashback", "wrong transaction", "collect request"],
    "Romance Scam": ["love", "relationship", "gift", "customs duty", "dating"],
    "Job Scam": ["job offer", "work from home", "registration fee", "part time job", "recruiter"],
    "Lottery Scam": ["lottery", "prize", "winner", "claim your", "lucky draw"],
    "Investment Scam": ["investment", "guaranteed return", "trading tips", "stock tips", "crypto"],
}


def _severity_from_risk(risk_score: float) -> str:
    '''Shared severity banding, kept consistent with Notebook 2's own bands.'''
    if risk_score <= CONFIG.SEVERITY_LOW_MAX:
        return "Low"
    if risk_score <= CONFIG.SEVERITY_MEDIUM_MAX:
        return "Medium"
    if risk_score <= CONFIG.SEVERITY_HIGH_MAX:
        return "High"
    return "Critical"


def _fraud_stub(case: CaseIntake, evidence_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Deterministic, keyword-weighted fallback classifier used only when Notebook 2 is unavailable or errors.'''
    text = (evidence_package.get("text") or " ".join(item.content for item in case.evidence if item.content)).lower()

    scores: Dict[str, int] = {}
    matched_keywords: Dict[str, List[str]] = {}
    for fraud_type, keywords in _FRAUD_TYPE_KEYWORDS.items():
        hits = [kw for kw in keywords if kw in text]
        if hits:
            scores[fraud_type] = len(hits)
            matched_keywords[fraud_type] = hits

    if scores:
        fraud_type = max(scores, key=scores.get)
        keyword_component = min(CONFIG.FRAUD_KEYWORD_SCORE_CAP, scores[fraud_type] * 15.0)
        reasoning = [f"Matched signal keywords for '{fraud_type}': {', '.join(matched_keywords[fraud_type])}."]
    else:
        fraud_type = "Unclassified Suspicious Activity"
        keyword_component = 0.0
        reasoning = ["No known fraud-pattern keywords were matched in the submitted evidence text."]

    entities = evidence_package.get("metadata") or evidence_package.get("extracted_entities") or {}
    entity_count = sum(len(v) for v in entities.values() if isinstance(v, list))
    entity_component = min(15.0, entity_count * 2.0)
    if entity_count:
        reasoning.append(f"{entity_count} identifiable entity(ies) extracted from evidence.")

    amount_component = min(5.0, case.amount_involved / 20000.0) if case.amount_involved else 0.0
    if case.amount_involved:
        reasoning.append(f"Amount involved: Rs {case.amount_involved:,.0f}.")

    risk_score = round(min(100.0, CONFIG.FRAUD_BASE_SCORE + keyword_component + entity_component + amount_component), 1)
    confidence = round(min(95.0, 40.0 + keyword_component + entity_component), 1)

    return {
        "case_id": case.case_id,
        "timestamp": case.timestamp,
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK2_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "fraud_type": fraud_type,
        "risk_score": risk_score,
        "confidence": confidence,
        "severity": _severity_from_risk(risk_score),
        "indicators": [],
        "entities": entities,
        "citations": {},
        "reasoning": reasoning,
        "summary": reasoning[0] if reasoning else "",
        "matched_keywords": matched_keywords,
    }


def run_fraud_intelligence(case: CaseIntake, evidence_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 4 entry point (Notebook 2 adapter, real-first with stub fallback).'''
    if _NOTEBOOK2_AVAILABLE:
        try:
            combined_text = evidence_package.get("text") or " ".join(item.content for item in case.evidence if item.content)
            result = notebook2.analyze_case(combined_text)
            result["engine_source"] = EngineSource.REAL.value
            result.setdefault("reasoning", [result.get("summary", "")])
            return result
        except Exception as exc:
            PLOG.warning("Notebook 2 raised at call time (likely missing OPENROUTER_API_KEY or no network); "
                         "falling back to stub fraud classification. error=%s", exc)

    return _fraud_stub(case, evidence_package)


# ============================================================================
# 8. Module 5 - Counterfeit Check (real Notebook 5 call, stub fallback)
# ============================================================================


def _counterfeit_stub(currency_items: List[EvidenceItem]) -> Dict[str, Any]:
    '''Deterministic pseudo-scored fallback, used only when Notebook 5 is unavailable, errors, or no file is on disk.'''
    findings = []
    max_suspicion = 0.0
    for item in currency_items:
        digest_source = item.content or item.metadata.get("file_path", item.evidence_type)
        digest = hashlib.sha256(digest_source.encode("utf-8")).hexdigest()
        pseudo_score = (int(digest[:8], 16) % 1000) / 1000.0
        if item.metadata.get("citizen_reported_suspicious"):
            pseudo_score = min(1.0, pseudo_score + 0.25)
        max_suspicion = max(max_suspicion, pseudo_score)
        findings.append({
            "evidence_content": digest_source,
            "suspicion_score": round(pseudo_score, 3),
            "citizen_flagged": bool(item.metadata.get("citizen_reported_suspicious", False)),
        })

    verdict = "Likely Counterfeit" if max_suspicion >= CONFIG.COUNTERFEIT_SUSPICION_THRESHOLD else "Inconclusive - Needs Manual Review"

    return {
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK5_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "items_checked": len(currency_items),
        "findings": findings,
        "max_suspicion_score": round(max_suspicion, 3),
        "verdict": verdict,
        "note": "Pseudo-scored stand-in; treat verdict as advisory only.",
    }


def run_counterfeit_check(case: CaseIntake) -> Optional[Dict[str, Any]]:
    '''Module 5 entry point (Notebook 5 adapter). Returns None if there is nothing to check.'''
    currency_items = [item for item in case.evidence if item.evidence_type == "currency_image"]
    if not currency_items:
        return None

    if _NOTEBOOK5_AVAILABLE:
        items_with_files = [item for item in currency_items if item.metadata.get("file_path")]
        if items_with_files:
            try:
                per_item_results = []
                max_counterfeit_prob = 0.0
                for item in items_with_files:
                    result = notebook5.analyze_currency_image(
                        item.metadata["file_path"],
                        denomination_hint=item.metadata.get("denomination_hint"),
                        known_serial_database=item.metadata.get("known_serial_database"),
                    )
                    per_item_results.append(result)
                    max_counterfeit_prob = max(max_counterfeit_prob, result["currency_analysis"]["counterfeit_probability"])

                verdict = "Likely Counterfeit" if max_counterfeit_prob >= CONFIG.COUNTERFEIT_SUSPICION_THRESHOLD else "Likely Genuine"
                return {
                    "engine_source": EngineSource.REAL.value,
                    "items_checked": len(per_item_results),
                    "per_item_results": per_item_results,
                    "max_suspicion_score": round(max_counterfeit_prob, 3),
                    "verdict": verdict,
                }
            except Exception as exc:
                PLOG.warning("Notebook 5 raised at call time; falling back to stub counterfeit check. error=%s", exc)

    return _counterfeit_stub(currency_items)


# ============================================================================
# 9. Module 6 - Run Network Intelligence (real Notebook 6 call)
# ============================================================================


def _build_network_case_record(case: CaseIntake, evidence_package: Dict[str, Any], fraud_package: Dict[str, Any]) -> Any:
    entities = evidence_package.get("metadata") or evidence_package.get("extracted_entities") or {}
    return notebook6.CaseRecord(
        case_id=case.case_id,
        victim_id=case.victim_id,
        fraud_type=fraud_package.get("fraud_type"),
        risk_score=fraud_package.get("risk_score", 0.0),
        phone_numbers=entities.get("phone_numbers", []),
        upi_ids=entities.get("upi_ids", []),
        emails=entities.get("emails", []),
        bank_accounts=entities.get("bank_accounts", []),
        urls=entities.get("urls", []),
        organizations=entities.get("organizations", []),
        amount_involved=case.amount_involved,
        city=case.city,
        state=case.state,
        timestamp=case.timestamp,
    )


def run_network_intelligence(case: CaseIntake, evidence_package: Dict[str, Any], fraud_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 6 entry point (real Notebook 6 call, with a minimal stub fallback). Thread-safe: only touches CASE_REGISTRY.network_cases.'''
    if not _NOTEBOOK6_AVAILABLE:
        return {
            "engine_source": EngineSource.STUB.value,
            "connected_cases": [], "communities": [], "fraud_campaigns": [], "money_mule_accounts": [],
            "central_actor": None,
            "risk_propagation": {
                "original_risk": fraud_package.get("risk_score", 0.0),
                "network_adjusted_risk": fraud_package.get("risk_score", 0.0),
                "boost_applied": 0.0, "driving_entity": None,
                "reasons": ["Notebook 6 module not available; network-adjusted risk defaults to the standalone score."],
            },
            "note": "fraud_network_intelligence_engine.py was not importable.",
        }

    try:
        record = _build_network_case_record(case, evidence_package, fraud_package)
        CASE_REGISTRY.register_network_case(case.case_id, record)

        package = notebook6.analyze_fraud_network(
            CASE_REGISTRY.all_network_cases(),
            focus_case_id=case.case_id,
            save_visualization=True,
            generate_report=False,   # Notebook 8 produces its own single merged report
        )
        package["engine_source"] = EngineSource.REAL.value
        return package
    except Exception as exc:
        PLOG.warning("Notebook 6 raised at call time; falling back to stub network package. error=%s", exc)
        return {
            "engine_source": EngineSource.STUB_FALLBACK.value,
            "connected_cases": [], "communities": [], "fraud_campaigns": [], "money_mule_accounts": [],
            "central_actor": None,
            "risk_propagation": {
                "original_risk": fraud_package.get("risk_score", 0.0),
                "network_adjusted_risk": fraud_package.get("risk_score", 0.0),
                "boost_applied": 0.0, "driving_entity": None,
                "reasons": [f"Notebook 6 raised an error at call time ({exc}); defaulting to the standalone risk score."],
            },
        }


# ============================================================================
# 10. Module 7 - Run Geospatial Intelligence (real Notebook 7 call)
# ============================================================================


def _find_case_campaign_id(case_id: str, network_package: Dict[str, Any]) -> Optional[str]:
    for campaign in network_package.get("fraud_campaigns", []):
        if case_id in campaign.get("linked_cases", []):
            return campaign["campaign_id"]
    return None


def _find_case_community_id(case_id: str, network_package: Dict[str, Any]) -> Optional[str]:
    for community in network_package.get("communities", []):
        if case_id in community.get("connected_cases", []):
            return community["community_id"]
    return None


def _case_touches_mule_account(case_id: str, network_package: Dict[str, Any]) -> bool:
    return any(case_id in mule.get("connected_cases", []) for mule in network_package.get("money_mule_accounts", []))


def run_geospatial_intelligence(case: CaseIntake, fraud_package: Dict[str, Any], network_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 7 entry point (real Notebook 7 call, with a minimal stub fallback). Runs after Notebook 6 - it needs network_package.'''
    if not _NOTEBOOK7_AVAILABLE:
        return {
            "engine_source": EngineSource.STUB.value,
            "hotspots": [], "district_risk": [], "predicted_hotspots": [], "campaign_spread": [],
            "resource_recommendations": [], "nearest_cyber_cells": [], "confidence": 0.0,
            "note": "geospatial_crime_pattern_intelligence_engine.py was not importable.",
        }

    try:
        network_adjusted_risk = network_package.get("risk_propagation", {}).get(
            "network_adjusted_risk", fraud_package.get("risk_score", 0.0)
        )

        record = notebook7.GeoCaseRecord(
            case_id=case.case_id,
            city=case.city,
            state=case.state,
            latitude=case.latitude,
            longitude=case.longitude,
            fraud_type=fraud_package.get("fraud_type"),
            risk_score=network_adjusted_risk,
            amount_involved=case.amount_involved,
            timestamp=case.timestamp,
            campaign_id=_find_case_campaign_id(case.case_id, network_package),
            community_id=_find_case_community_id(case.case_id, network_package),
            is_mule_location=_case_touches_mule_account(case.case_id, network_package),
        )
        CASE_REGISTRY.register_geo_case(case.case_id, record)

        package = notebook7.analyze_geospatial_intelligence(
            CASE_REGISTRY.all_geo_cases(),
            save_visualization=True,
            generate_report=False,   # Notebook 8 produces its own single merged report
        )
        package["engine_source"] = EngineSource.REAL.value
        return package
    except Exception as exc:
        PLOG.warning("Notebook 7 raised at call time; falling back to stub geospatial package. error=%s", exc)
        return {
            "engine_source": EngineSource.STUB_FALLBACK.value,
            "hotspots": [], "district_risk": [], "predicted_hotspots": [], "campaign_spread": [],
            "resource_recommendations": [], "nearest_cyber_cells": [], "confidence": 0.0,
        }


# ============================================================================
# 11. PII Masking (review item #8)
# ============================================================================
#
# Applied on citizen- and telecom-facing dashboards only. Police and the
# administrator dashboard keep unmasked values, since investigators need
# the real phone numbers / UPI IDs / account numbers to act on a case.


def _mask_generic(value: str, keep_start: int = 2, keep_end: int = 2) -> str:
    if not value:
        return value
    if len(value) <= keep_start + keep_end:
        return "*" * len(value)
    return value[:keep_start] + "*" * (len(value) - keep_start - keep_end) + value[-keep_end:]


def mask_phone(phone: str) -> str:
    digits = re.sub(r"\D", "", phone)
    if len(digits) <= 5:
        return "*" * len(digits)
    return digits[:3] + "*" * (len(digits) - 5) + digits[-2:]


def mask_upi(upi_id: str) -> str:
    return _mask_generic(upi_id, keep_start=2, keep_end=2)


def mask_bank_account(account: str) -> str:
    digits = re.sub(r"\D", "", account)
    if len(digits) <= 4:
        return "*" * len(digits)
    return "*" * (len(digits) - 4) + digits[-4:]


def mask_email(email: str) -> str:
    if "@" not in email:
        return _mask_generic(email)
    local, domain = email.split("@", 1)
    masked_local = "*" * len(local) if len(local) <= 2 else local[0] + "*" * (len(local) - 1)
    return f"{masked_local}@{domain}"


def mask_phone_list(phones: List[str]) -> List[str]:
    return [mask_phone(p) for p in phones]


# ============================================================================
# 12. Threat Fusion Engine (with dynamic weight redistribution)
# ============================================================================


def _counterfeit_signal_score(counterfeit_package: Optional[Dict[str, Any]]) -> float:
    '''Converts the counterfeit package (real or stub) into a 0-100 threat signal. No evidence -> neutral 0.'''
    if not counterfeit_package:
        return 0.0
    return round(counterfeit_package.get("max_suspicion_score", 0.0) * 100, 1)


def _geo_signal_score(case_id: str, geo_package: Dict[str, Any]) -> float:
    '''
    Converts geospatial context into a 0-100 threat signal: the priority
    of any hotspot this case falls in, otherwise the priority of this
    case's district, otherwise a neutral 0.
    '''
    for hotspot in geo_package.get("hotspots", []):
        if case_id in hotspot.get("case_ids", []):
            return CONFIG.PRIORITY_TO_SCORE.get(hotspot["priority"], 0.0)
    for district in geo_package.get("district_risk", []):
        if case_id in district.get("case_ids", []):
            return CONFIG.PRIORITY_TO_SCORE.get(district["priority"], 0.0)
    return 0.0


def _redistribute_weights(base_weights: Dict[str, float], inactive_keys: List[str]) -> Dict[str, float]:
    '''
    Review item #5. Any signal in inactive_keys has no real evidence for
    this case, so its configured weight is redistributed evenly across
    the remaining active signals instead of quietly shrinking the total.
    Returns a new weight dict; base_weights is left untouched so the
    original configured weights can still be reported for audit.
    '''
    weights = dict(base_weights)
    inactive = set(k for k in inactive_keys if k in weights)
    active = [k for k in weights if k not in inactive]
    if not inactive or not active:
        return weights
    freed_weight = sum(weights[k] for k in inactive)
    bonus_each = freed_weight / len(active)
    for k in active:
        weights[k] += bonus_each
    for k in inactive:
        weights[k] = 0.0
    return weights


def fuse_threat_score(
    case_id: str,
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    '''
    Threat Fusion Engine. Blends four independent signals into one 0-100
    overall threat score, rather than trusting any single engine's number
    in isolation. Every component is reported alongside the final score,
    along with both the configured base weights and the effective
    per-case weights (after dynamic redistribution), so the fusion is
    auditable, not a black box.
    '''
    fraud_risk = fraud_package.get(Config.FIELD_RISK_SCORE, 0.0)
    network_risk = network_package.get("risk_propagation", {}).get("network_adjusted_risk", fraud_risk)
    geo_signal = _geo_signal_score(case_id, geo_package)
    counterfeit_signal = _counterfeit_signal_score(counterfeit_package)

    base_weights = {
        "network_adjusted_risk": CONFIG.FUSION_WEIGHT_NETWORK_RISK,
        "fraud_standalone_risk": CONFIG.FUSION_WEIGHT_FRAUD_RISK,
        "geospatial_signal": CONFIG.FUSION_WEIGHT_GEO_SIGNAL,
        "counterfeit_signal": CONFIG.FUSION_WEIGHT_COUNTERFEIT_SIGNAL,
    }

    inactive_keys = []
    if counterfeit_package is None:
        inactive_keys.append("counterfeit_signal")
    if not geo_package.get("hotspots") and not geo_package.get("district_risk"):
        inactive_keys.append("geospatial_signal")

    effective_weights = _redistribute_weights(base_weights, inactive_keys)

    components = {
        "fraud_standalone_risk": fraud_risk,
        "network_adjusted_risk": network_risk,
        "geospatial_signal": geo_signal,
        "counterfeit_signal": counterfeit_signal,
    }

    fused = sum(components[k] * effective_weights[k] for k in components)
    fused = round(min(100.0, fused), 1)

    return {
        "overall_threat_score": fused,
        "severity": _severity_from_risk(fused),
        "components": components,
        "base_weights": base_weights,
        "effective_weights": effective_weights,
        "weights": effective_weights,  # backward-compatible key name used by the report builder
        "redistributed_signals": inactive_keys,
    }


def build_risk_breakdown(
    case: CaseIntake,
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    fusion_result: Dict[str, Any],
    decision_package: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Review item #7 (AI improvements - "also produce Financial Risk /
    Victim Risk / National Risk / Priority"). These are transparent
    heuristics built from numbers Notebook 8 already has, not a
    separately trained model - documented here so nobody mistakes them
    for a fifth ML engine.
    '''
    components = fusion_result["components"]

    financial_risk = round(min(100.0, (
        (case.amount_involved / 100000.0) * 40.0
        + components["counterfeit_signal"] * 0.3
        + fraud_package.get(Config.FIELD_RISK_SCORE, 0.0) * 0.3
    )), 1)

    victim_risk = round(min(100.0, (
        fraud_package.get(Config.FIELD_RISK_SCORE, 0.0) * 0.5
        + (30.0 if network_package.get("money_mule_accounts") else 0.0)
        + (20.0 if fraud_package.get(Config.FIELD_SEVERITY) == "Critical" else 0.0)
    )), 1)

    national_risk = round(min(100.0, (
        components["geospatial_signal"] * 0.5
        + (25.0 if network_package.get("fraud_campaigns") else 0.0)
        + (25.0 if network_package.get("money_mule_accounts") else 0.0)
    )), 1)

    return {
        "financial_risk": financial_risk,
        "victim_risk": victim_risk,
        "national_risk": national_risk,
        "priority": _decision_category_label(decision_package),
    }


# ============================================================================
# 13. Confidence Fusion Engine (now includes an evidence-quality signal)
# ============================================================================


def fuse_confidence(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
    evidence_package: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Weighted confidence fusion. A component that produced no signal (e.g.
    no communities detected, no counterfeit evidence submitted) is
    excluded from the blend and the remaining weights are re-normalized,
    rather than silently treating "no signal" as "zero confidence".
    Review item #6 adds evidence_quality as a fifth, always-present
    component.
    '''
    components: Dict[str, float] = {
        "fraud": fraud_package.get(Config.FIELD_CONFIDENCE, 0.0),
        "evidence_quality": _evidence_quality_score(evidence_package),
    }
    weights: Dict[str, float] = {
        "fraud": CONFIG.CONFIDENCE_WEIGHT_FRAUD,
        "evidence_quality": CONFIG.CONFIDENCE_WEIGHT_EVIDENCE_QUALITY,
    }

    communities = network_package.get("communities", [])
    if communities:
        components["network"] = communities[0].get(Config.FIELD_CONFIDENCE, 0.0)
        weights["network"] = CONFIG.CONFIDENCE_WEIGHT_NETWORK

    if geo_package.get(Config.FIELD_CONFIDENCE):
        components["geospatial"] = geo_package[Config.FIELD_CONFIDENCE]
        weights["geospatial"] = CONFIG.CONFIDENCE_WEIGHT_GEO

    if counterfeit_package:
        certainty = abs(counterfeit_package.get("max_suspicion_score", 0.5) - 0.5) * 2 * 100
        components["counterfeit"] = round(certainty, 1)
        weights["counterfeit"] = CONFIG.CONFIDENCE_WEIGHT_COUNTERFEIT

    total_weight = sum(weights.values()) or 1.0
    fused = sum(components[k] * weights[k] for k in components) / total_weight

    return {
        "overall_confidence": round(min(99.0, fused), 1),
        "components": components,
        "weights_used": weights,
    }


# ============================================================================
# 14. Cross-Notebook Validation
# ============================================================================


def validate_cross_notebook_consistency(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Compares Notebook 2's classified fraud_type against Notebook 6's
    dominant community fraud type and Notebook 7's dominant hotspot fraud
    type. A single engine disagreeing is common and not flagged (small
    samples vary); TWO OR MORE independent engines disagreeing with
    Notebook 2 is treated as a genuine conflict that should force human
    review rather than an automatic high-confidence decision.
    '''
    fraud_type = fraud_package.get(Config.FIELD_FRAUD_TYPE)
    conflicts: List[str] = []

    communities = network_package.get("communities", [])
    if communities and communities[0].get("dominant_fraud") not in (None, "Unclassified", fraud_type):
        conflicts.append(
            f"Notebook 2 classified this case as '{fraud_type}', but the network community "
            f"it belongs to is dominated by '{communities[0]['dominant_fraud']}'."
        )

    hotspots = geo_package.get("hotspots", [])
    if hotspots and hotspots[0].get("dominant_fraud") not in (None, "Unclassified", fraud_type):
        conflicts.append(
            f"Notebook 2 classified this case as '{fraud_type}', but the geographic hotspot "
            f"it falls in is dominated by '{hotspots[0]['dominant_fraud']}'."
        )

    return {
        "is_consistent": len(conflicts) < 2,
        "conflicts_found": conflicts,
        "force_human_review": len(conflicts) >= 2,
    }


# ============================================================================
# 15. Module 8 - Final Decision (real Notebook 3 call, stub fallback)
# ============================================================================


class DecisionCategory(str, Enum):
    EMERGENCY = "Emergency"
    URGENT_ACTION = "Urgent Action"
    NEEDS_HUMAN_REVIEW = "Needs Human Review"
    AWARENESS_ONLY = "Awareness Only"
    NO_ACTION = "No Action - Benign"


def _decision_stub(
    fraud_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
    fused_risk: float,
    validation: Dict[str, Any],
) -> Dict[str, Any]:
    '''Deterministic fallback decision policy, used only when Notebook 3 is unavailable or errors.'''
    reasons: List[str] = [f"Fused overall threat score is {fused_risk}."]

    if validation["force_human_review"]:
        category = DecisionCategory.NEEDS_HUMAN_REVIEW
        reasons.append("Cross-notebook validation found conflicting fraud-type signals; routed to human review.")
    elif fused_risk >= CONFIG.DECISION_EMERGENCY_RISK_MIN:
        category = DecisionCategory.EMERGENCY
    elif fused_risk >= CONFIG.DECISION_URGENT_RISK_MIN:
        category = DecisionCategory.URGENT_ACTION
    elif fused_risk >= CONFIG.DECISION_HUMAN_REVIEW_RISK_MIN:
        category = DecisionCategory.AWARENESS_ONLY
    else:
        category = DecisionCategory.NO_ACTION

    if counterfeit_package and counterfeit_package.get("verdict") == "Likely Counterfeit":
        reasons.append("Submitted currency evidence was flagged as likely counterfeit.")

    stakeholder_actions = {
        "citizen": [
            "File a formal complaint on cybercrime.gov.in if not already done.",
            "Do not make any further payments to the numbers/accounts involved.",
        ],
        "police": [], "bank": [], "telecom": [],
    }
    if category in (DecisionCategory.EMERGENCY, DecisionCategory.URGENT_ACTION):
        stakeholder_actions["police"].append("Prioritize investigation; cross-reference network and geospatial findings.")

    return {
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK3_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "case_decision": category.value,
        "priority": category.value,
        "reasons": reasons,
        "citizen_actions": stakeholder_actions["citizen"],
        "police_actions": stakeholder_actions["police"],
        "bank_actions": stakeholder_actions["bank"],
        "telecom_actions": stakeholder_actions["telecom"],
    }


def run_decision_engine(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
    fusion_result: Dict[str, Any],
    validation: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Module 8 entry point (Notebook 3 adapter). Feeds Notebook 3 the FUSED
    threat score and severity (not the raw Notebook-2 standalone numbers),
    so its policy engine reasons over the same number the rest of this
    package reports as the overall threat level.
    '''
    if _NOTEBOOK3_AVAILABLE:
        try:
            adjusted_input = dict(fraud_package)
            adjusted_input[Config.FIELD_RISK_SCORE] = fusion_result["overall_threat_score"]
            adjusted_input[Config.FIELD_SEVERITY] = fusion_result["severity"]

            similar_cases = [
                {"case_id": cid, "similarity": 0.75}
                for cid in network_package.get("connected_cases", [])
            ]

            result = notebook3.build_decision_package(adjusted_input, similar_cases=similar_cases or None)
            result["engine_source"] = EngineSource.REAL.value

            if validation["force_human_review"] and result.get("case_decision") != notebook3.CaseDecision.NEEDS_HUMAN_REVIEW.value:
                result["case_decision"] = notebook3.CaseDecision.NEEDS_HUMAN_REVIEW.value
                result.setdefault("policy_reasons", []).append(
                    "Overridden by Notebook 8: cross-notebook validation found conflicting fraud-type signals."
                )
            return result
        except Exception as exc:
            PLOG.warning("Notebook 3 raised at call time; falling back to stub decision policy. error=%s", exc)

    return _decision_stub(fraud_package, counterfeit_package, fusion_result["overall_threat_score"], validation)


def _decision_category_label(decision_package: Dict[str, Any]) -> str:
    '''Normalizes whichever key the real Notebook 3 or the stub used for the final category.'''
    return decision_package.get("case_decision") or decision_package.get("decision_category") or "Unknown"


def _decision_reasons(decision_package: Dict[str, Any]) -> List[str]:
    return decision_package.get("policy_reasons") or decision_package.get("reasons") or []


def _stakeholder_actions(decision_package: Dict[str, Any], stakeholder: str) -> List[str]:
    '''Normalizes stakeholder action lookup across the real Notebook 3 shape and the stub shape.'''
    direct_key = f"{stakeholder}_actions"
    if direct_key in decision_package:
        return decision_package[direct_key]
    return decision_package.get("stakeholder_actions", {}).get(stakeholder, [])


# ============================================================================
# 16. Explainability Chain + Flowchart
# ============================================================================


def build_explainability(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    fusion_result: Dict[str, Any],
    validation: Dict[str, Any],
    decision_package: Dict[str, Any],
) -> List[Dict[str, str]]:
    '''Module 16 entry point. A short, ordered "why" chain a reviewer can read in one glance.'''
    chain: List[Dict[str, str]] = []
    chain.append({"step": "Fraud type identified", "detail": fraud_package.get(Config.FIELD_FRAUD_TYPE, "Unclassified")})
    for reason in fraud_package.get("reasoning", []):
        chain.append({"step": "Evidence signal", "detail": reason})

    risk_info = network_package.get("risk_propagation", {})
    if risk_info.get("boost_applied", 0) > 0:
        chain.append({
            "step": "Network context raised risk",
            "detail": f"{risk_info['original_risk']} -> {risk_info['network_adjusted_risk']} (driven by {risk_info.get('driving_entity')})",
        })

    hotspots = geo_package.get("hotspots", [])
    if hotspots:
        chain.append({
            "step": "Geographic context",
            "detail": f"Case falls within hotspot {hotspots[0]['hotspot_id']} ({hotspots[0]['primary_city']}), priority {hotspots[0]['priority']}.",
        })

    chain.append({
        "step": "Threat fusion",
        "detail": f"Overall threat score {fusion_result['overall_threat_score']} ({fusion_result['severity']}), "
                  f"blended from fraud/network/geo/counterfeit signals.",
    })

    if validation["conflicts_found"]:
        for conflict in validation["conflicts_found"]:
            chain.append({"step": "Validation conflict", "detail": conflict})

    for reason in _decision_reasons(decision_package):
        chain.append({"step": "Decision factor", "detail": reason})

    chain.append({"step": "Final decision", "detail": _decision_category_label(decision_package)})
    return chain


def build_explainability_flowchart(routing_plan: Dict[str, Any]) -> str:
    '''
    Review item #16 (flowchart style). A compact arrow-chain summarizing
    which stages actually ran for this case, for a one-line "how did we
    get here" view above the detailed chain.
    '''
    steps = ["Evidence", "Entities", "Fraud Detection"]
    steps.append("Network")
    steps.append("Geo" if routing_plan.get("run_geospatial_intelligence") else "Geo [skipped]")
    if routing_plan.get("run_counterfeit_check"):
        steps.append("Counterfeit")
    steps.append("Threat Fusion")
    steps.append("Decision")
    return " -> ".join(steps)


# ============================================================================
# 17. Incident Timeline
# ============================================================================


def build_incident_timeline(
    case: CaseIntake,
    evidence_package: Dict[str, Any],
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    decision_package: Dict[str, Any],
) -> List[Dict[str, str]]:
    '''
    Reconstructs a plain-language, chronological view of how this case
    moved through the platform - useful for an investigator or auditor to
    see when each stage fired.
    '''
    timeline: List[Dict[str, str]] = []
    base_time = case.timestamp or datetime.now(timezone.utc).isoformat()

    timeline.append({"event": f"Case {case.case_id} received via {case.source}.", "timestamp": base_time})
    timeline.append({"event": f"Evidence normalized ({len(case.evidence)} item(s)).", "timestamp": base_time})
    timeline.append({
        "event": f"Fraud classified as '{fraud_package.get(Config.FIELD_FRAUD_TYPE)}' "
                 f"(standalone risk {fraud_package.get(Config.FIELD_RISK_SCORE)}).",
        "timestamp": base_time,
    })

    if network_package.get("connected_cases"):
        timeline.append({
            "event": f"Matched to {len(network_package['connected_cases'])} prior case(s) in the fraud network.",
            "timestamp": base_time,
        })
    if network_package.get("money_mule_accounts"):
        timeline.append({"event": "Linked account(s) flagged as likely money mule.", "timestamp": base_time})

    if geo_package.get("hotspots"):
        for hotspot in geo_package["hotspots"]:
            if case.case_id in hotspot.get("case_ids", []):
                timeline.append({"event": f"Case falls within geographic hotspot {hotspot['hotspot_id']}.", "timestamp": base_time})
                break

    timeline.append({
        "event": f"Final decision: {_decision_category_label(decision_package)}.",
        "timestamp": base_time,
    })
    return timeline


# ============================================================================
# 18. Engine Health Dashboard
# ============================================================================


def build_engine_health(audit_trail: List[Dict[str, Any]]) -> Dict[str, Any]:
    '''
    One place to see, per pipeline stage, whether the real engine ran, a
    stub ran by design, or a stub ran because the real engine failed at
    call time - the last case is the one worth watching in a live
    deployment.
    '''
    health: Dict[str, Any] = {}
    for entry in audit_trail:
        stage = entry["stage"]
        source = entry.get("engine_source", EngineSource.SKIPPED.value)
        if source == EngineSource.REAL.value:
            status = "Healthy (real engine)"
        elif source == EngineSource.STUB.value:
            status = "Stub by design (module not installed / not applicable)"
        elif source == EngineSource.STUB_FALLBACK.value:
            status = "DEGRADED - real engine failed at runtime, stub fallback used"
        else:
            status = "Skipped"
        health[stage] = {
            "status": status,
            "engine_source": source,
            "duration_ms": entry.get("duration_ms"),
            "failure_reason": entry.get("failure_reason"),
            "recovered": entry.get("recovered", False),
        }
    return health


def print_engine_registry_table() -> None:
    '''Review item #1/#2. Plain-text table, no emoji, for the demo terminal.'''
    print("=" * 74)
    print("ENGINE REGISTRY")
    print("=" * 74)
    header = f"{'Engine':38s} {'Version':10s} {'Loaded':7s} {'Health':10s}"
    print(header)
    print("-" * 74)
    for meta in ENGINE_REGISTRY.values():
        print(f"{meta.name:38s} {meta.version:10s} {str(meta.loaded):7s} {meta.health:10s}")
    print("=" * 74)


# ============================================================================
# 19. Modules 10-14 - Audience-Specific Dashboards
# ============================================================================


def _risk_band(score: float) -> str:
    if score >= CONFIG.DECISION_EMERGENCY_RISK_MIN:
        return "Critical"
    if score >= CONFIG.DECISION_URGENT_RISK_MIN:
        return "High"
    if score >= CONFIG.DECISION_HUMAN_REVIEW_RISK_MIN:
        return "Medium"
    return "Low"


def build_citizen_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 11 entry point. Citizens see outcomes and guidance, not investigative internals.'''
    fraud = master["fraud_intelligence"]
    decision = master["decision_intelligence"]
    risk_breakdown = master["risk_breakdown"]
    return {
        "case_id": master["case"]["case_id"],
        "fraud_type": fraud.get(Config.FIELD_FRAUD_TYPE),
        "risk_level": _risk_band(master["overall_threat_level"]),
        "what_this_means": _decision_category_label(decision),
        "recommended_actions": _stakeholder_actions(decision, "citizen"),
        "national_cyber_crime_helpline": "1930",
        "national_cyber_crime_portal": "cybercrime.gov.in",
        "estimated_recovery_chance": "Higher if reported within 24 hours (the 'Golden Hour' window for freezing transactions)",
        "complaint_status": "Not yet filed - please file on cybercrime.gov.in or via the 1930 helpline",
        "expected_investigation_time": "Varies by case complexity; typically 7-30 days for an initial response",
        "priority": risk_breakdown["priority"],
        "safety_advice": [
            "Do not share OTPs, passwords, or remote-access app codes with anyone.",
            "Government agencies do not conduct arrests or investigations over video call.",
            "When in doubt, hang up and call the organization back using an officially published number.",
        ],
    }


def build_police_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 12 entry point. Police get the full investigative picture, unmasked.'''
    return {
        "case_id": master["case"]["case_id"],
        "summary": master["executive_summary"],
        "fraud_intelligence": master["fraud_intelligence"],
        "fraud_network_intelligence": master["fraud_network_intelligence"],
        "geospatial_intelligence": master["geospatial_intelligence"],
        "counterfeit_intelligence": master["counterfeit_intelligence"],
        "decision_intelligence": master["decision_intelligence"],
        "risk_breakdown": master["risk_breakdown"],
        "explainability": master["explainability"],
        "explainability_flowchart": master["explainability_flowchart"],
        "incident_timeline": master["incident_timeline"],
        "validation": master["cross_notebook_validation"],
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "police"),
    }


def build_bank_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 13 entry point. Banks see financial-entity risk only.'''
    network = master["fraud_network_intelligence"]
    return {
        "case_id": master["case"]["case_id"],
        "money_mule_accounts": network.get("money_mule_accounts", []),
        "network_adjusted_risk": network.get("risk_propagation", {}).get("network_adjusted_risk"),
        "financial_risk": master["risk_breakdown"]["financial_risk"],
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "bank"),
    }


def build_telecom_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 14 entry point. Telecom providers see phone/campaign risk only, with phone numbers masked.'''
    entities = master["evidence"].get("metadata") or master["evidence"].get("extracted_entities") or {}
    network = master["fraud_network_intelligence"]
    raw_phones = entities.get("phone_numbers", [])
    phones = mask_phone_list(raw_phones) if CONFIG.ENABLE_PII_MASKING else raw_phones
    return {
        "case_id": master["case"]["case_id"],
        "phone_numbers": phones,
        "linked_campaigns": [c["campaign_id"] for c in network.get("fraud_campaigns", [])],
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "telecom"),
    }


def build_administrator_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''
    Fifth audience view. Administrators care about platform health and
    pipeline performance, not case-specific investigative detail.

    IMPORTANT: this function requires master["audit_trail"],
    master["execution_statistics"], and master["engine_health"] to
    already be populated. The caller (process_case) MUST set those three
    keys on `master` BEFORE calling this function - that ordering is the
    fix that shipped in Revision 4 and remains unchanged here.
    '''
    audit_trail = master["audit_trail"]
    stage_seconds = master["execution_statistics"]["stage_seconds"]
    failures = [e for e in audit_trail if e.get("status") == "Failed"]
    fallbacks = [e for e in audit_trail if e.get("engine_source") == EngineSource.STUB_FALLBACK.value]

    if stage_seconds:
        slowest_stage, slowest_seconds = max(stage_seconds.items(), key=lambda kv: kv[1])
        fastest_stage, fastest_seconds = min(stage_seconds.items(), key=lambda kv: kv[1])
    else:
        slowest_stage, slowest_seconds = None, None
        fastest_stage, fastest_seconds = None, None

    return {
        "case_id": master["case"]["case_id"],
        "engine_availability": master["engine_availability"],
        "engine_registry": [dataclasses.asdict(meta) for meta in ENGINE_REGISTRY.values()],
        "engine_health": master["engine_health"],
        "audit_trail": master["audit_trail"],
        "execution_statistics": master["execution_statistics"],
        "total_cases_in_registry": CASE_REGISTRY.total_cases(),
        "cross_notebook_validation": master["cross_notebook_validation"],
        "cpu_time_seconds": round(time.process_time(), 4),
        "memory_peak_mb": _current_memory_mb(),
        "slowest_stage": {"stage": slowest_stage, "seconds": slowest_seconds},
        "fastest_stage": {"stage": fastest_stage, "seconds": fastest_seconds},
        "failure_count": len(failures),
        "fallback_count": len(fallbacks),
    }


def _current_memory_mb() -> Optional[float]:
    '''Review item #17 (admin dashboard: CPU time, memory). Returns None if the resource module is unavailable (e.g. Windows).'''
    if not _RESOURCE_AVAILABLE:
        return None
    try:
        usage_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        return round(usage_kb / 1024.0, 2)
    except Exception:
        return None


# ============================================================================
# 20. Executive Summary
# ============================================================================


def build_executive_summary(master: Dict[str, Any]) -> Dict[str, Any]:
    '''
    One-screen summary an officer should be able to read in under 30
    seconds: what happened, how bad it is, and what is already known
    about it from the network and geography.
    '''
    network = master["fraud_network_intelligence"]
    geo = master["geospatial_intelligence"]
    fusion = master["threat_fusion"]
    risk_breakdown = master["risk_breakdown"]

    top_community = network.get("communities", [{}])[0] if network.get("communities") else None
    top_hotspot = geo.get("hotspots", [{}])[0] if geo.get("hotspots") else None

    return {
        "case_id": master["case"]["case_id"],
        "fraud_type": master["fraud_intelligence"].get(Config.FIELD_FRAUD_TYPE),
        "threat_level": fusion["severity"],
        "overall_threat_score": fusion["overall_threat_score"],
        "overall_confidence": master["overall_confidence"],
        "financial_risk": risk_breakdown["financial_risk"],
        "victim_risk": risk_breakdown["victim_risk"],
        "national_risk": risk_breakdown["national_risk"],
        "amount_involved": master["case"]["amount_involved"],
        "connected_cases_count": len(network.get("connected_cases", [])),
        "campaign_id": network.get("fraud_campaigns", [{}])[0].get("campaign_id") if network.get("fraud_campaigns") else None,
        "money_mule_flagged": bool(network.get("money_mule_accounts")),
        "hotspot": top_hotspot["hotspot_id"] if top_hotspot else None,
        "hotspot_city": top_hotspot["primary_city"] if top_hotspot else None,
        "community_dominant_fraud": top_community.get("dominant_fraud") if top_community else None,
        "decision": _decision_category_label(master["decision_intelligence"]),
        "validation_conflicts": master["cross_notebook_validation"]["conflicts_found"],
    }


# ============================================================================
# 21. Platform Dashboard (compact, presentation-style summary)
# ============================================================================


def build_platform_dashboard_text(master: Dict[str, Any]) -> str:
    '''Renders a compact, box-drawn summary block suitable for a terminal demo or a report cover page.'''
    exec_summary = master["executive_summary"]
    bar = "=" * 56
    lines = [
        bar,
        "DIGITAL PUBLIC SAFETY PLATFORM",
        bar,
        f"Case ID           : {exec_summary['case_id']}",
        f"Threat Level      : {exec_summary['threat_level'].upper()}",
        f"Fraud Type        : {exec_summary['fraud_type']}",
        f"Overall Threat    : {exec_summary['overall_threat_score']}/100",
        f"Confidence        : {exec_summary['overall_confidence']}%",
        f"Financial Risk    : {exec_summary['financial_risk']}/100",
        f"Victim Risk       : {exec_summary['victim_risk']}/100",
        f"National Risk     : {exec_summary['national_risk']}/100",
        f"Money Mule Flag   : {'YES' if exec_summary['money_mule_flagged'] else 'No'}",
        f"Campaign          : {exec_summary['campaign_id'] or 'None identified'}",
        f"Connected Cases   : {exec_summary['connected_cases_count']}",
        f"Hotspot           : {exec_summary['hotspot'] or 'None'} ({exec_summary['hotspot_city'] or 'n/a'})",
        f"Decision          : {exec_summary['decision']}",
        bar,
    ]
    return "\n".join(lines)


def _ascii_threat_gauge(score: float) -> str:
    '''Review item #18 (report improvement: threat meter / risk gauge). 20-segment ASCII bar.'''
    filled = int(round(score / 5.0))
    filled = max(0, min(20, filled))
    bar = "#" * filled + "-" * (20 - filled)
    return f"[{bar}] {score}/100"


# ============================================================================
# 22. Final Intelligence Report (PDF, with plain-text fallback)
# ============================================================================


def _build_final_report_text_lines(master: Dict[str, Any]) -> List[str]:
    lines: List[str] = []
    lines.append(master["platform_dashboard_text"])
    lines.append("")

    lines.append("THREAT GAUGE")
    lines.append(f"  {_ascii_threat_gauge(master['threat_fusion']['overall_threat_score'])}")
    lines.append("")

    lines.append("EXECUTIVE SUMMARY")
    for key, value in master["executive_summary"].items():
        lines.append(f"  {key}: {value}")
    lines.append("")

    lines.append("EXPLAINABILITY FLOWCHART")
    lines.append(f"  {master['explainability_flowchart']}")
    lines.append("")

    lines.append("EXPLAINABILITY (DETAIL)")
    for step in master["explainability"]:
        lines.append(f"  [{step['step']}] {step['detail']}")
    lines.append("")

    lines.append("INCIDENT TIMELINE")
    for event in master["incident_timeline"]:
        lines.append(f"  {event['timestamp']}  -  {event['event']}")
    lines.append("")

    if master["cross_notebook_validation"]["conflicts_found"]:
        lines.append("VALIDATION CONFLICTS")
        for conflict in master["cross_notebook_validation"]["conflicts_found"]:
            lines.append(f"  - {conflict}")
        lines.append("")

    lines.append("FRAUD INTELLIGENCE")
    lines.append(f"  Type: {master['fraud_intelligence'].get(Config.FIELD_FRAUD_TYPE)}  |  Standalone risk: {master['fraud_intelligence'].get(Config.FIELD_RISK_SCORE)}")
    lines.append("")

    if master["counterfeit_intelligence"]:
        lines.append("COUNTERFEIT INTELLIGENCE")
        lines.append(f"  Verdict: {master['counterfeit_intelligence'].get('verdict')}")
        lines.append("")

    network = master["fraud_network_intelligence"]
    lines.append("FRAUD NETWORK INTELLIGENCE")
    lines.append(f"  Connected cases: {len(network.get('connected_cases', []))}")
    lines.append(f"  Communities: {len(network.get('communities', []))}  |  Money mule accounts: {len(network.get('money_mule_accounts', []))}")
    lines.append("")

    geo = master["geospatial_intelligence"]
    lines.append("GEOSPATIAL INTELLIGENCE")
    lines.append(f"  Hotspots: {len(geo.get('hotspots', []))}  |  Districts monitored: {len(geo.get('district_risk', []))}")
    lines.append("")

    lines.append("THREAT FUSION")
    fusion = master["threat_fusion"]
    lines.append(f"  Overall threat score: {fusion['overall_threat_score']} ({fusion['severity']})")
    for component, value in fusion["components"].items():
        base_w = fusion["base_weights"][component]
        eff_w = fusion["effective_weights"][component]
        redistributed = " (redistributed)" if component in fusion["redistributed_signals"] else ""
        lines.append(f"    {component}: {value}  (base weight {base_w}, effective weight {eff_w}{redistributed})")
    lines.append("")

    lines.append("RISK BREAKDOWN")
    for key, value in master["risk_breakdown"].items():
        lines.append(f"  {key}: {value}")
    lines.append("")

    decision = master["decision_intelligence"]
    lines.append("DECISION")
    lines.append(f"  Category: {_decision_category_label(decision)}")
    for reason in _decision_reasons(decision):
        lines.append(f"  Reason: {reason}")
    lines.append("")

    lines.append("ENGINE REGISTRY")
    for meta in ENGINE_REGISTRY.values():
        lines.append(f"  {meta.name:38s} | v{meta.version:10s} | loaded={meta.loaded} | {meta.health}")
    lines.append("")

    lines.append("ENGINE HEALTH")
    for stage, health in master["engine_health"].items():
        lines.append(f"  {stage:36s} | {health['status']}")
    lines.append("")

    lines.append("AUDIT TRAIL")
    for entry in master["audit_trail"]:
        reason = f" | reason: {entry.get('failure_reason')}" if entry.get("failure_reason") else ""
        lines.append(f"  {entry['stage']:36s} | {entry['engine_source']:36s} | {entry['duration_ms']} ms{reason}")
    lines.append("")

    lines.append("EXECUTION STATISTICS")
    for stage, seconds in master["execution_statistics"]["stage_seconds"].items():
        lines.append(f"  {stage:36s} {seconds:.3f} sec")
    lines.append(f"  {'TOTAL':36s} {master['execution_statistics']['total_seconds']:.3f} sec")
    lines.append("")

    admin = master["administrator_response"]
    lines.append("ADMINISTRATOR DIAGNOSTICS")
    lines.append(f"  CPU time (process): {admin['cpu_time_seconds']} sec")
    lines.append(f"  Peak memory: {admin['memory_peak_mb']} MB" if admin["memory_peak_mb"] is not None else "  Peak memory: unavailable on this platform")
    lines.append(f"  Slowest stage: {admin['slowest_stage']['stage']} ({admin['slowest_stage']['seconds']} sec)")
    lines.append(f"  Fastest stage: {admin['fastest_stage']['stage']} ({admin['fastest_stage']['seconds']} sec)")
    lines.append(f"  Failures: {admin['failure_count']}  |  Fallbacks used: {admin['fallback_count']}")

    return lines


def _build_threat_gauge_drawing(score: float, severity: str) -> Any:
    '''Review item #18. A simple horizontal bar gauge for the PDF report, built from reportlab primitives only (no external image files or matplotlib dependency).'''
    width, height = 400, 26
    color_map = {"Low": colors.green, "Medium": colors.gold, "High": colors.orange, "Critical": colors.red}
    fill_color = color_map.get(severity, colors.grey)
    filled_width = max(0.0, min(width, width * (score / 100.0)))

    drawing = Drawing(width, height + 20)
    drawing.add(Rect(0, 15, width, height, fillColor=colors.lightgrey, strokeColor=colors.grey))
    drawing.add(Rect(0, 15, filled_width, height, fillColor=fill_color, strokeColor=None))
    drawing.add(String(2, height + 18, f"Threat Score: {score}/100 ({severity})", fontSize=9))
    return drawing


def generate_final_intelligence_report(master: Dict[str, Any], output_path: str) -> str:
    '''
    Module 15 entry point. One PDF (or plain-text fallback) covering:
    platform dashboard, threat gauge, executive summary, explainability
    flowchart + detail, incident timeline, validation conflicts,
    per-engine intelligence, threat fusion (with redistributed weights),
    risk breakdown, decision, engine registry, engine health, audit
    trail, execution stats, and administrator diagnostics.

    NOTE: this function reads master["audit_trail"],
    master["execution_statistics"], master["engine_health"],
    master["executive_summary"], master["administrator_response"], and
    master["platform_dashboard_text"], so the caller MUST populate all
    of those keys on `master` before calling this function.
    '''
    lines = _build_final_report_text_lines(master)

    if not _REPORTLAB_AVAILABLE:
        fallback_path = os.path.splitext(output_path)[0] + ".txt"
        with open(fallback_path, "w", encoding="utf-8") as fh:
            fh.write("\n".join(lines))
        PLOG.info("reportlab not available; wrote plain-text final report to %s", fallback_path)
        return fallback_path

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("ReportTitle", parent=styles["Title"], fontSize=16)
    heading_style = ParagraphStyle("ReportHeading", parent=styles["Heading2"], spaceBefore=10, spaceAfter=4)
    body_style = ParagraphStyle("ReportBody", parent=styles["BodyText"], spaceAfter=2)

    doc = SimpleDocTemplate(output_path, pagesize=A4, topMargin=1.5 * cm, bottomMargin=1.5 * cm)
    story: List[Any] = []

    story.append(Paragraph(f"Digital Public Safety Report - {master['package_id']}", title_style))
    story.append(Paragraph(f"Case: {master['case']['case_id']}", body_style))
    story.append(Spacer(1, 8))

    fusion = master["threat_fusion"]
    if _REPORTLAB_GRAPHICS_AVAILABLE:
        story.append(_build_threat_gauge_drawing(fusion["overall_threat_score"], fusion["severity"]))
        story.append(Spacer(1, 6))

    story.append(Paragraph("Executive Summary", heading_style))
    for key, value in master["executive_summary"].items():
        story.append(Paragraph(f"<b>{key}:</b> {value}", body_style))

    story.append(Paragraph("Explainability Flowchart", heading_style))
    story.append(Paragraph(master["explainability_flowchart"], body_style))

    story.append(Paragraph("Explainability (Detail)", heading_style))
    for step in master["explainability"]:
        story.append(Paragraph(f"<b>{step['step']}:</b> {step['detail']}", body_style))

    story.append(Paragraph("Incident Timeline", heading_style))
    for event in master["incident_timeline"]:
        story.append(Paragraph(f"{event['timestamp']} - {event['event']}", body_style))

    if master["cross_notebook_validation"]["conflicts_found"]:
        story.append(Paragraph("Validation Conflicts", heading_style))
        for conflict in master["cross_notebook_validation"]["conflicts_found"]:
            story.append(Paragraph(conflict, body_style))

    story.append(Paragraph("Fraud Intelligence", heading_style))
    story.append(Paragraph(
        f"Type: {master['fraud_intelligence'].get(Config.FIELD_FRAUD_TYPE)} | Standalone risk: {master['fraud_intelligence'].get(Config.FIELD_RISK_SCORE)}",
        body_style,
    ))

    if master["counterfeit_intelligence"]:
        story.append(Paragraph("Counterfeit Intelligence", heading_style))
        story.append(Paragraph(f"Verdict: {master['counterfeit_intelligence'].get('verdict')}", body_style))

    network = master["fraud_network_intelligence"]
    story.append(Paragraph("Fraud Network Intelligence", heading_style))
    story.append(Paragraph(
        f"Connected cases: {len(network.get('connected_cases', []))} | "
        f"Communities: {len(network.get('communities', []))} | "
        f"Money mule accounts: {len(network.get('money_mule_accounts', []))}",
        body_style,
    ))

    geo = master["geospatial_intelligence"]
    story.append(Paragraph("Geospatial Intelligence", heading_style))
    story.append(Paragraph(
        f"Hotspots: {len(geo.get('hotspots', []))} | Districts monitored: {len(geo.get('district_risk', []))}",
        body_style,
    ))

    story.append(Paragraph("Threat Fusion", heading_style))
    story.append(Paragraph(f"Overall threat score: {fusion['overall_threat_score']} ({fusion['severity']})", body_style))
    fusion_rows = [["Component", "Value", "Base Weight", "Effective Weight"]]
    for component, value in fusion["components"].items():
        fusion_rows.append([
            component, str(value),
            str(fusion["base_weights"][component]),
            str(fusion["effective_weights"][component]),
        ])
    fusion_table = Table(fusion_rows, colWidths=[5 * cm, 3 * cm, 3 * cm, 3.5 * cm])
    fusion_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
    ]))
    story.append(fusion_table)

    story.append(Paragraph("Risk Breakdown", heading_style))
    for key, value in master["risk_breakdown"].items():
        story.append(Paragraph(f"<b>{key}:</b> {value}", body_style))

    decision = master["decision_intelligence"]
    story.append(Paragraph("Decision", heading_style))
    story.append(Paragraph(f"Category: {_decision_category_label(decision)}", body_style))
    for reason in _decision_reasons(decision):
        story.append(Paragraph(f"Reason: {reason}", body_style))

    story.append(PageBreak())

    story.append(Paragraph("Engine Registry", heading_style))
    registry_rows = [["Engine", "Version", "Loaded", "Health"]]
    for meta in ENGINE_REGISTRY.values():
        registry_rows.append([meta.name, meta.version, str(meta.loaded), meta.health])
    registry_table = Table(registry_rows, colWidths=[7 * cm, 3 * cm, 2.5 * cm, 3 * cm])
    registry_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ]))
    story.append(registry_table)

    story.append(Paragraph("Engine Health", heading_style))
    health_rows = [["Stage", "Status"]]
    for stage, health in master["engine_health"].items():
        health_rows.append([stage, health["status"]])
    health_table = Table(health_rows, colWidths=[6 * cm, 9 * cm])
    health_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ]))
    story.append(health_table)

    story.append(Paragraph("Audit Trail", heading_style))
    audit_rows = [["Stage", "Engine", "Duration (ms)"]]
    for entry in master["audit_trail"]:
        audit_rows.append([entry["stage"], entry["engine_source"], str(entry["duration_ms"])])
    audit_table = Table(audit_rows, colWidths=[6 * cm, 6 * cm, 3 * cm])
    audit_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ]))
    story.append(audit_table)

    story.append(Paragraph("Execution Statistics", heading_style))
    story.append(Paragraph(f"Total processing time: {master['execution_statistics']['total_seconds']:.3f} sec", body_style))

    admin = master["administrator_response"]
    story.append(Paragraph("Administrator Diagnostics", heading_style))
    story.append(Paragraph(f"CPU time (process): {admin['cpu_time_seconds']} sec", body_style))
    mem_text = f"{admin['memory_peak_mb']} MB" if admin["memory_peak_mb"] is not None else "unavailable on this platform"
    story.append(Paragraph(f"Peak memory: {mem_text}", body_style))
    story.append(Paragraph(f"Slowest stage: {admin['slowest_stage']['stage']} ({admin['slowest_stage']['seconds']} sec)", body_style))
    story.append(Paragraph(f"Fastest stage: {admin['fastest_stage']['stage']} ({admin['fastest_stage']['seconds']} sec)", body_style))
    story.append(Paragraph(f"Failures: {admin['failure_count']} | Fallbacks used: {admin['fallback_count']}", body_style))

    doc.build(story)
    PLOG.info("Final intelligence report saved to %s", output_path)
    return output_path


# ============================================================================
# 23. Timing / audit wrapper (with richer diagnostics + thread safety)
# ============================================================================

_AUDIT_LOCK = threading.Lock()


def _run_stage(
    stage_name: str,
    engine_source: str,
    fn: Callable[[], Any],
    audit_trail: List[Dict[str, Any]],
    stage_seconds: Dict[str, float],
    required: bool = True,
    engine_key: Optional[str] = None,
) -> Any:
    '''
    Shared timing/audit wrapper for every pipeline stage. Review item #2:
    on failure, records the failure reason and whether a fallback
    recovered the stage, instead of just "Failed". Safe to call from
    multiple threads concurrently - fn() itself runs unlocked (so real
    parallel work happens), only the bookkeeping at the end is locked.
    Also updates the Engine Registry for stages tied to Notebooks 2-7.
    '''
    start = time.perf_counter()
    cpu_start = time.process_time()
    try:
        result = fn()
    except Exception as exc:
        duration_ms = round((time.perf_counter() - start) * 1000, 2)
        cpu_ms = round((time.process_time() - cpu_start) * 1000, 2)
        with _AUDIT_LOCK:
            audit_trail.append({
                "stage": stage_name, "status": "Failed", "engine_source": engine_source,
                "duration_ms": duration_ms, "cpu_ms": cpu_ms,
                "failure_reason": str(exc), "recovered": False,
            })
            stage_seconds[stage_name] = round((time.perf_counter() - start), 4)
            if engine_key and engine_key in ENGINE_REGISTRY:
                meta = ENGINE_REGISTRY[engine_key]
                meta.status, meta.health = "Failed", "FAILED"
                meta.last_inference_seconds = duration_ms / 1000.0
                meta.last_updated = datetime.now(timezone.utc).isoformat()
        if required:
            raise OrchestrationError(f"Stage '{stage_name}' failed: {exc}") from exc
        PLOG.warning("Optional stage '%s' failed and was skipped: %s", stage_name, exc)
        return None

    duration_ms = round((time.perf_counter() - start) * 1000, 2)
    cpu_ms = round((time.process_time() - cpu_start) * 1000, 2)
    actual_source = result.get("engine_source", engine_source) if isinstance(result, dict) else engine_source
    recovered = actual_source == EngineSource.STUB_FALLBACK.value

    with _AUDIT_LOCK:
        audit_trail.append({
            "stage": stage_name,
            "status": "Completed" if result is not None else "Skipped",
            "engine_source": actual_source if result is not None else EngineSource.SKIPPED.value,
            "duration_ms": duration_ms,
            "cpu_ms": cpu_ms,
            "failure_reason": None,
            "recovered": recovered,
        })
        stage_seconds[stage_name] = round((time.perf_counter() - start), 4)

        if engine_key and engine_key in ENGINE_REGISTRY and result is not None:
            meta = ENGINE_REGISTRY[engine_key]
            meta.last_inference_seconds = duration_ms / 1000.0
            meta.last_updated = datetime.now(timezone.utc).isoformat()
            if actual_source == EngineSource.REAL.value:
                meta.status, meta.health = "Loaded", "Healthy"
            elif actual_source == EngineSource.STUB.value:
                meta.status, meta.health = "Stub (module not installed)", "Degraded (stub by design)"
            elif actual_source == EngineSource.STUB_FALLBACK.value:
                meta.status, meta.health = "Stub fallback", "DEGRADED (real engine failed, recovered via fallback)"

    return result


# ============================================================================
# 24. Dict-subclass wrapper for dot-access (review item #4)
# ============================================================================


class _EngineView:
    '''Read-only convenience accessor: master.engines.fraud instead of master["fraud_intelligence"].'''

    def __init__(self, master: Dict[str, Any]) -> None:
        self._m = master

    @property
    def fraud(self) -> Dict[str, Any]:
        return self._m["fraud_intelligence"]

    @property
    def network(self) -> Dict[str, Any]:
        return self._m["fraud_network_intelligence"]

    @property
    def geo(self) -> Dict[str, Any]:
        return self._m["geospatial_intelligence"]

    @property
    def counterfeit(self) -> Optional[Dict[str, Any]]:
        return self._m["counterfeit_intelligence"]

    @property
    def decision(self) -> Dict[str, Any]:
        return self._m["decision_intelligence"]

    @property
    def evidence(self) -> Dict[str, Any]:
        return self._m["evidence"]


class DigitalPublicSafetyPackage(dict):
    '''
    A dict subclass, so every existing master["key"] access in this file
    (and in any downstream code you already wrote against Revision 4)
    keeps working unchanged, while new code can use the dot-access form
    package.engines.fraud, package.engines.network, etc.
    '''

    @property
    def engines(self) -> _EngineView:
        return _EngineView(self)


# ============================================================================
# 25. Module 9 - Merge Everything / Orchestration Entry Point
# ============================================================================

_STARTUP_BANNER_PRINTED = False


def _print_startup_banner() -> None:
    global _STARTUP_BANNER_PRINTED
    if _STARTUP_BANNER_PRINTED:
        return
    print("=" * 58)
    print("DIGITAL PUBLIC SAFETY PLATFORM")
    print("Powered by Multi-Agent AI Intelligence")
    print("ET AI Hackathon 2026")
    print("=" * 58)
    print_engine_registry_table()
    _STARTUP_BANNER_PRINTED = True


def process_case(raw_case: CaseIntake, report_dir: str = "/tmp/notebook8_reports") -> DigitalPublicSafetyPackage:
    '''
    Notebook 8 orchestration (Revision 5).

    Ordering: every key that any dashboard-builder reads from `master`
    is populated BEFORE any dashboard is built - the Revision 4 fix -
    plus the new risk_breakdown / explainability_flowchart keys added in
    this revision, which are also populated before the dashboards read
    them.

      1. Run the analysis pipeline:
         intake -> evidence routing -> Notebook 4 (evidence) ->
         Notebook 2 (fraud) -> [Notebook 5 (counterfeit) and Notebook 6
         (network) concurrently] -> Notebook 7 (geo, depends on
         Notebook 6) -> threat fusion (dynamic weights) -> confidence
         fusion (with evidence-quality signal) -> cross-notebook
         validation -> Notebook 3 (decision) -> risk breakdown ->
         explainability (chain + flowchart).
      2. Build the base `master` dict (case, packages, fusion,
         validation, decision, risk breakdown, explainability, engine
         availability).
      3. Build incident_timeline.
      4. Populate audit_trail and execution_statistics (interim total).
      5. Build engine_health (needs audit_trail).
      6. Build executive_summary and platform_dashboard_text.
      7. Build the administrator dashboard's diagnostic fields need to
         come from a first pass at engine_health/execution_statistics,
         so steps 4-6 happen before ANY dashboard is built.
      8. Build the five audience dashboards - citizen, police, bank,
         telecom, administrator.
      9. Generate the final PDF/text report (reads the same keys, plus
         the administrator dashboard for the diagnostics section).
     10. Refresh audit-trail-derived fields once more, since report
         generation itself added one more audit-trail entry.
    '''
    _print_startup_banner()

    pipeline_start = time.perf_counter()
    audit_trail: List[Dict[str, Any]] = []
    stage_seconds: Dict[str, float] = {}

    # --- Step 1: analysis pipeline ---
    case = _run_stage("Case Intake", EngineSource.REAL.value, lambda: intake_case(raw_case), audit_trail, stage_seconds)
    print(f"[DONE] Case intake complete ({case.case_id})")

    routing_plan = _run_stage("Evidence Routing", EngineSource.REAL.value, lambda: route_evidence(case), audit_trail, stage_seconds)

    evidence_package = _run_stage(
        "Notebook 4 - Evidence Intelligence", EngineSource.REAL.value,
        lambda: run_evidence_engine(case), audit_trail, stage_seconds, engine_key="notebook4",
    )
    print("[DONE] Evidence processed")

    fraud_package = _run_stage(
        "Notebook 2 - Fraud Intelligence", EngineSource.REAL.value,
        lambda: run_fraud_intelligence(case, evidence_package), audit_trail, stage_seconds, engine_key="notebook2",
    )
    print("[DONE] Fraud detected")

    # --- Notebook 5 (counterfeit) and Notebook 6 (network) run
    # concurrently: neither depends on the other's output, both depend
    # only on evidence_package/fraud_package which are already final. ---
    counterfeit_package: Optional[Dict[str, Any]] = None
    network_package: Dict[str, Any]

    if CONFIG.ENABLE_CONCURRENT_STAGES:
        with ThreadPoolExecutor(max_workers=2) as executor:
            future_map: Dict[Any, str] = {}
            if routing_plan["run_counterfeit_check"]:
                future_map[executor.submit(
                    _run_stage, "Notebook 5 - Counterfeit Intelligence", EngineSource.REAL.value,
                    lambda: run_counterfeit_check(case), audit_trail, stage_seconds, False, "notebook5",
                )] = "counterfeit"
            future_map[executor.submit(
                _run_stage, "Notebook 6 - Fraud Network Intelligence", EngineSource.REAL.value,
                lambda: run_network_intelligence(case, evidence_package, fraud_package), audit_trail, stage_seconds, True, "notebook6",
            )] = "network"

            concurrent_results: Dict[str, Any] = {}
            for fut in as_completed(future_map):
                concurrent_results[future_map[fut]] = fut.result()

        counterfeit_package = concurrent_results.get("counterfeit")
        network_package = concurrent_results["network"]
    else:
        if routing_plan["run_counterfeit_check"]:
            counterfeit_package = _run_stage(
                "Notebook 5 - Counterfeit Intelligence", EngineSource.REAL.value,
                lambda: run_counterfeit_check(case), audit_trail, stage_seconds, required=False, engine_key="notebook5",
            )
        network_package = _run_stage(
            "Notebook 6 - Fraud Network Intelligence", EngineSource.REAL.value,
            lambda: run_network_intelligence(case, evidence_package, fraud_package), audit_trail, stage_seconds, engine_key="notebook6",
        )
    print("[DONE] Network analysed" + (" (counterfeit check included)" if routing_plan["run_counterfeit_check"] else ""))

    geo_package: Dict[str, Any] = {}
    if routing_plan["run_geospatial_intelligence"]:
        geo_package = _run_stage(
            "Notebook 7 - Geospatial Intelligence", EngineSource.REAL.value,
            lambda: run_geospatial_intelligence(case, fraud_package, network_package), audit_trail, stage_seconds, engine_key="notebook7",
        ) or {}
    print("[DONE] Geo intelligence" + ("" if routing_plan["run_geospatial_intelligence"] else " (skipped - no location)"))

    threat_fusion = _run_stage(
        "Threat Fusion Engine", EngineSource.REAL.value,
        lambda: fuse_threat_score(case.case_id, fraud_package, network_package, geo_package, counterfeit_package),
        audit_trail, stage_seconds,
    )
    print("[DONE] Threat fusion complete")

    confidence_fusion = _run_stage(
        "Confidence Fusion Engine", EngineSource.REAL.value,
        lambda: fuse_confidence(fraud_package, network_package, geo_package, counterfeit_package, evidence_package),
        audit_trail, stage_seconds,
    )

    validation = _run_stage(
        "Cross-Notebook Validation", EngineSource.REAL.value,
        lambda: validate_cross_notebook_consistency(fraud_package, network_package, geo_package),
        audit_trail, stage_seconds,
    )

    decision_package = _run_stage(
        "Notebook 3 - Decision Intelligence", EngineSource.REAL.value,
        lambda: run_decision_engine(fraud_package, network_package, counterfeit_package, threat_fusion, validation),
        audit_trail, stage_seconds, engine_key="notebook3",
    )
    print("[DONE] Decision generated")

    risk_breakdown = _run_stage(
        "Risk Breakdown", EngineSource.REAL.value,
        lambda: build_risk_breakdown(case, fraud_package, network_package, threat_fusion, decision_package),
        audit_trail, stage_seconds,
    )

    explainability = _run_stage(
        "Explainability", EngineSource.REAL.value,
        lambda: build_explainability(fraud_package, network_package, geo_package, threat_fusion, validation, decision_package),
        audit_trail, stage_seconds,
    )
    explainability_flowchart = build_explainability_flowchart(routing_plan)

    # --- Step 2: base master dict ---
    case_dict = {
        "case_id": case.case_id, "citizen_name": case.citizen_name, "victim_id": case.victim_id,
        "timestamp": case.timestamp, "city": case.city, "state": case.state,
        "priority": case.priority, "source": case.source, "amount_involved": case.amount_involved,
    }

    master: DigitalPublicSafetyPackage = DigitalPublicSafetyPackage({
        "package_id": f"DPSP-{datetime.now(timezone.utc).year}-{uuid.uuid4().hex[:6].upper()}",
        "case": case_dict,
        "evidence_routing_plan": routing_plan,
        "evidence": evidence_package,
        "fraud_intelligence": fraud_package,
        "counterfeit_intelligence": counterfeit_package,
        "fraud_network_intelligence": network_package,
        "geospatial_intelligence": geo_package,
        "threat_fusion": threat_fusion,
        "confidence_fusion": confidence_fusion,
        "risk_breakdown": risk_breakdown,
        "cross_notebook_validation": validation,
        "decision_intelligence": decision_package,
        "explainability": explainability,
        "explainability_flowchart": explainability_flowchart,
        "overall_threat_level": threat_fusion["overall_threat_score"],
        "overall_confidence": confidence_fusion["overall_confidence"],
        "engine_availability": {
            "notebook2_fraud_intelligence": _NOTEBOOK2_AVAILABLE,
            "notebook3_decision_intelligence": _NOTEBOOK3_AVAILABLE,
            "notebook4_evidence_intelligence": _NOTEBOOK4_AVAILABLE,
            "notebook5_counterfeit_intelligence": _NOTEBOOK5_AVAILABLE,
            "notebook6_network_intelligence": _NOTEBOOK6_AVAILABLE,
            "notebook7_geospatial_intelligence": _NOTEBOOK7_AVAILABLE,
        },
    })

    # --- Step 3: incident timeline ---
    master["incident_timeline"] = build_incident_timeline(
        case, evidence_package, fraud_package, network_package, geo_package, decision_package
    )

    # --- Step 4: audit_trail / execution_statistics (interim total) ---
    # THIS MUST HAPPEN BEFORE ANY DASHBOARD IS BUILT.
    interim_total_seconds = round(time.perf_counter() - pipeline_start, 4)
    master["audit_trail"] = audit_trail
    master["execution_statistics"] = {
        "stage_seconds": stage_seconds,
        "total_seconds": interim_total_seconds,
    }

    # --- Step 5: engine_health (needs audit_trail, which now exists) ---
    master["engine_health"] = build_engine_health(audit_trail)

    # --- Step 6: executive summary and platform dashboard text ---
    master["executive_summary"] = build_executive_summary(master)
    master["platform_dashboard_text"] = build_platform_dashboard_text(master)

    # --- Step 7/8: build the administrator dashboard first (it derives
    # cpu/memory/slowest/fastest from audit_trail + execution_statistics,
    # both of which now exist), then the remaining four dashboards. ---
    master["administrator_response"] = build_administrator_dashboard(master)
    master["citizen_response"] = build_citizen_dashboard(master)
    master["police_response"] = build_police_dashboard(master)
    master["bank_response"] = build_bank_dashboard(master)
    master["telecom_response"] = build_telecom_dashboard(master)

    # --- Step 9: final report ---
    os.makedirs(report_dir, exist_ok=True)
    report_path = _run_stage(
        "Final Intelligence Report", EngineSource.REAL.value,
        lambda: generate_final_intelligence_report(master, os.path.join(report_dir, f"digital_public_safety_report_{uuid.uuid4().hex[:8]}.pdf")),
        audit_trail, stage_seconds,
    )
    master["final_report"] = report_path
    print("[DONE] Report generated")

    # --- Step 10: refresh audit-trail-derived fields after the report stage ---
    master["engine_health"] = build_engine_health(audit_trail)
    total_seconds = round(time.perf_counter() - pipeline_start, 4)
    master["execution_statistics"]["total_seconds"] = total_seconds
    master["administrator_response"] = build_administrator_dashboard(master)

    case_digest = hashlib.sha256(case.case_id.encode("utf-8")).hexdigest()
    master["audit"] = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "case_id_hash": case_digest,
        "notebook_version": CONFIG.NOTEBOOK_VERSION,
    }

    CASE_REGISTRY.register_master_package(case.case_id, master)

    PLOG.success(
        "Case processing complete. case_id=%s decision=%s threat_level=%s confidence=%s total_seconds=%.3f",
        case.case_id, _decision_category_label(decision_package),
        master["overall_threat_level"], master["overall_confidence"], total_seconds,
    )
    return master


# ============================================================================
# 26. Synthetic Case Generation and Deterministic Test Suite
# ============================================================================


def _build_synthetic_cases() -> List[CaseIntake]:
    base_time = datetime(2026, 7, 1, tzinfo=timezone.utc)
    cases: List[CaseIntake] = []

    # --- Three linked "Digital Arrest" cases, sharing a phone number and
    # UPI id, reported from Kolhapur then Pune. ---
    shared_phone = "9876543210"
    shared_upi = "rahul@okaxis"
    cities = ["Kolhapur", "Kolhapur", "Pune"]
    for i in range(1, 4):
        cases.append(CaseIntake(
            case_id=f"DPSP-CASE-{i:03d}",
            citizen_name=f"Citizen {i}",
            city=cities[i - 1],
            state="Maharashtra",
            timestamp=base_time.replace(day=min(28, base_time.day + i)).isoformat(),
            amount_involved=45000.0 + i * 5000,
            evidence=[
                EvidenceItem(
                    evidence_type="call_recording",
                    content=(
                        f"Caller claimed to be from CBI and RBI, said there is an arrest warrant against me, "
                        f"demanded a video call and payment via UPI to {shared_upi} to avoid digital arrest. "
                        f"Caller phone number was {shared_phone}."
                    ),
                ),
            ],
        ))

    # --- One case with a currency image (no real file on disk in this
    # test, so the counterfeit stub path is exercised). ---
    cases.append(CaseIntake(
        case_id="DPSP-CASE-CURRENCY-001",
        citizen_name="Citizen 4",
        city="Mumbai",
        state="Maharashtra",
        timestamp=base_time.replace(day=15).isoformat(),
        amount_involved=500.0,
        evidence=[
            EvidenceItem(evidence_type="text", content="I received this 500 rupee note as change and it feels off."),
            EvidenceItem(
                evidence_type="currency_image",
                content="note_scan_001.jpg",
                metadata={"citizen_reported_suspicious": True},
            ),
        ],
    ))

    # --- One low-signal case with no fraud keywords, no location, and a
    # tiny amount, to exercise the No Action decision path. ---
    cases.append(CaseIntake(
        case_id="DPSP-CASE-LOWSIGNAL-001",
        citizen_name="Citizen 5",
        timestamp=base_time.replace(day=20).isoformat(),
        amount_involved=200.0,
        evidence=[EvidenceItem(evidence_type="text", content="Just wanted to ask a general question about safe UPI usage.")],
    ))

    return cases


def run_notebook8_test_suite() -> Dict[str, Any]:
    print("=== Notebook 8 Test Suite: end-to-end Digital Public Safety Platform (Revision 5) ===\n")

    cases = _build_synthetic_cases()
    checks: List[bool] = []
    results: List[DigitalPublicSafetyPackage] = []

    def _check(label: str, actual: Any, expected: Any) -> None:
        ok = actual == expected
        checks.append(ok)
        print(f"    [{'PASS' if ok else 'FAIL'}] {label}: expected={expected!r} actual={actual!r}")

    def _check_true(label: str, condition: bool) -> None:
        checks.append(condition)
        print(f"    [{'PASS' if condition else 'FAIL'}] {label}")

    print("--- Processing cases sequentially through the orchestrator ---")
    for case in cases:
        print(f"\nProcessing {case.case_id} ...")
        master = process_case(case)
        results.append(master)
        print(master["platform_dashboard_text"])

    ring_result = results[2]
    currency_result = results[3]
    low_signal_result = results[4]

    # --- Case intake / routing ---
    _check("first case id preserved", results[0]["case"]["case_id"], "DPSP-CASE-001")
    _check_true("currency case routed counterfeit check on", currency_result["evidence_routing_plan"]["run_counterfeit_check"])
    _check_true("linked-ring case routed counterfeit check off", ring_result["evidence_routing_plan"]["run_counterfeit_check"] is False)

    # --- Evidence engine ---
    entities = ring_result["evidence"].get("metadata") or ring_result["evidence"].get("extracted_entities") or {}
    _check_true("phone number extracted from ring case evidence", "9876543210" in entities.get("phone_numbers", []))

    # --- Fraud intelligence ---
    _check_true("low-signal case classified without a hard crash",
                low_signal_result["fraud_intelligence"].get(Config.FIELD_FRAUD_TYPE) is not None)

    # --- Threat fusion (dynamic weights) ---
    fusion = ring_result["threat_fusion"]
    _check_true("threat fusion score is within 0-100", 0.0 <= fusion["overall_threat_score"] <= 100.0)
    _check_true("threat fusion reports all four components",
                set(fusion["components"].keys()) == {"fraud_standalone_risk", "network_adjusted_risk", "geospatial_signal", "counterfeit_signal"})
    _check_true("threat fusion effective weights still sum to 1.0",
                abs(sum(fusion["effective_weights"].values()) - 1.0) < 1e-6)
    _check_true("geo signal was redistributed for the location-bearing ring case only if geo produced no hotspot/district",
                ("geospatial_signal" in fusion["redistributed_signals"]) == (not ring_result["geospatial_intelligence"].get("hotspots") and not ring_result["geospatial_intelligence"].get("district_risk")))
    low_fusion = low_signal_result["threat_fusion"]
    _check_true("low-signal case (no location) redistributes the geo weight",
                "geospatial_signal" in low_fusion["redistributed_signals"])
    _check_true("low-signal case (no counterfeit evidence) redistributes the counterfeit weight",
                "counterfeit_signal" in low_fusion["redistributed_signals"])

    # --- Confidence fusion ---
    conf = ring_result["confidence_fusion"]
    _check_true("confidence fusion score is within 0-100", 0.0 <= conf["overall_confidence"] <= 100.0)
    _check_true("confidence fusion includes the evidence_quality component", "evidence_quality" in conf["components"])

    # --- Risk breakdown ---
    risk_breakdown = ring_result["risk_breakdown"]
    _check_true("risk breakdown carries all four new fields",
                {"financial_risk", "victim_risk", "national_risk", "priority"}.issubset(set(risk_breakdown.keys())))

    # --- Cross-notebook validation ---
    _check_true("validation result carries a boolean consistency flag", isinstance(ring_result["cross_notebook_validation"]["is_consistent"], bool))

    # --- Counterfeit intelligence ---
    _check_true("counterfeit check ran for currency case", currency_result["counterfeit_intelligence"] is not None)
    _check_true("counterfeit check did not run for ring case", ring_result["counterfeit_intelligence"] is None)

    # --- Decision intelligence ---
    _check_true("decision category was produced for every case",
                all(_decision_category_label(r["decision_intelligence"]) != "Unknown" for r in results))

    # --- Explainability / flowchart / timeline ---
    _check_true("explainability chain ends with the final decision",
                ring_result["explainability"][-1]["detail"] == _decision_category_label(ring_result["decision_intelligence"]))
    _check_true("explainability flowchart is a non-empty arrow chain", "->" in ring_result["explainability_flowchart"])
    _check_true("incident timeline is non-empty", len(ring_result["incident_timeline"]) > 0)

    # --- Executive summary / platform dashboard ---
    _check_true("executive summary carries a threat_level", "threat_level" in ring_result["executive_summary"])
    _check_true("executive summary carries the new risk fields",
                {"financial_risk", "victim_risk", "national_risk"}.issubset(set(ring_result["executive_summary"].keys())))
    _check_true("platform dashboard text is a non-empty string", isinstance(ring_result["platform_dashboard_text"], str) and len(ring_result["platform_dashboard_text"]) > 0)

    # --- Dashboards ---
    citizen_dash = ring_result["citizen_response"]
    police_dash = ring_result["police_response"]
    bank_dash = ring_result["bank_response"]
    telecom_dash = ring_result["telecom_response"]
    admin_dash = ring_result["administrator_response"]

    _check_true("citizen dashboard excludes raw network graph internals", "fraud_network_intelligence" not in citizen_dash)
    _check_true("citizen dashboard carries the national helpline", citizen_dash["national_cyber_crime_helpline"] == "1930")
    _check_true("citizen dashboard carries the new recovery/complaint fields",
                {"estimated_recovery_chance", "complaint_status", "expected_investigation_time"}.issubset(set(citizen_dash.keys())))
    _check_true("police dashboard includes full network intelligence", "fraud_network_intelligence" in police_dash)
    _check_true("police dashboard includes the explainability flowchart", "explainability_flowchart" in police_dash)

    # telecom dashboard: phone numbers must be masked, not raw
    telecom_phones = telecom_dash["phone_numbers"]
    _check_true("telecom dashboard masks phone numbers",
                all("*" in p for p in telecom_phones) if telecom_phones else True)

    _check_true("administrator dashboard was built without a KeyError (Revision 4 regression test, still holds)",
                admin_dash is not None and "engine_health" in admin_dash and "audit_trail" in admin_dash
                and "execution_statistics" in admin_dash and "total_cases_in_registry" in admin_dash
                and "cross_notebook_validation" in admin_dash)
    _check_true("administrator dashboard's execution_statistics total accounts for report generation time",
                admin_dash["execution_statistics"]["total_seconds"] >= admin_dash["execution_statistics"]["stage_seconds"].get("Final Intelligence Report", 0.0))
    _check_true("administrator dashboard carries the new engine registry",
                "engine_registry" in admin_dash and len(admin_dash["engine_registry"]) == 6)
    _check_true("administrator dashboard carries CPU/memory/slowest/fastest diagnostics",
                {"cpu_time_seconds", "memory_peak_mb", "slowest_stage", "fastest_stage", "failure_count", "fallback_count"}.issubset(set(admin_dash.keys())))

    # --- Dot-access wrapper ---
    _check_true("master package supports dict-style access", ring_result["fraud_intelligence"] is ring_result.engines.fraud)
    _check_true("master package supports dot-access via .engines", ring_result.engines.network is ring_result["fraud_network_intelligence"])

    # --- Case registry (SQLite-backed) ---
    _check_true("case registry has processed all synthetic cases", CASE_REGISTRY.total_cases() >= len(cases))

    # --- Master package structure ---
    expected_keys = {
        "package_id", "case", "evidence_routing_plan", "evidence", "fraud_intelligence",
        "counterfeit_intelligence", "fraud_network_intelligence", "geospatial_intelligence",
        "threat_fusion", "confidence_fusion", "risk_breakdown", "cross_notebook_validation", "decision_intelligence",
        "explainability", "explainability_flowchart", "incident_timeline", "engine_health", "executive_summary",
        "platform_dashboard_text", "overall_threat_level", "overall_confidence", "engine_availability",
        "citizen_response", "police_response", "bank_response", "telecom_response", "administrator_response",
        "final_report", "audit_trail", "execution_statistics", "audit",
    }
    _check_true("master package contains all expected top-level keys", expected_keys.issubset(set(ring_result.keys())))

    # --- Audit trail / performance statistics ---
    _check_true("audit trail has an entry for every pipeline stage", len(ring_result["audit_trail"]) >= 10)
    _check_true("execution statistics report a positive total time", ring_result["execution_statistics"]["total_seconds"] > 0)
    _check_true("audit trail entries carry the new failure_reason/recovered fields",
                all("failure_reason" in e and "recovered" in e for e in ring_result["audit_trail"]))

    # --- Final report ---
    _check_true("final intelligence report file was generated", ring_result["final_report"] is not None)
    _check_true("final intelligence report file exists on disk", os.path.exists(ring_result["final_report"]))

    print(f"\nSUMMARY: {sum(checks)}/{len(checks)} checks passed\n")

    print("Overall results by case:")
    for m in results:
        print(
            f"  {m['case']['case_id']:26s} threat={m['overall_threat_level']:6.1f} "
            f"decision={_decision_category_label(m['decision_intelligence']):20s} "
            f"confidence={m['overall_confidence']}%"
        )

    print(f"\nEngine availability: {json.dumps(ring_result['engine_availability'], indent=2)}")
    print_engine_registry_table()
    print(f"\nEngine health for {ring_result['case']['case_id']}:")
    for stage, health in ring_result["engine_health"].items():
        print(f"  {stage:36s} | {health['status']}")

    admin = ring_result["administrator_response"]
    print(f"\nAdministrator diagnostics for {ring_result['case']['case_id']}:")
    print(f"  CPU time: {admin['cpu_time_seconds']} sec | Memory: {admin['memory_peak_mb']} MB")
    print(f"  Slowest stage: {admin['slowest_stage']}")
    print(f"  Fastest stage: {admin['fastest_stage']}")
    print(f"  Failures: {admin['failure_count']} | Fallbacks used: {admin['fallback_count']}")

    print(f"\nFinal report file: {ring_result['final_report']}")

    return {"results": results, "checks_passed": sum(checks), "checks_total": len(checks)}


if __name__ == "__main__":
    run_notebook8_test_suite()

2026-07-22 17:22:27,279 | INFO | digital_public_safety_platform | Notebook 8 configuration loaded. version=v5.0 notebook2=True notebook3=True notebook4=True notebook5=True notebook6=True notebook7=True
2026-07-22 17:22:27,302 | INFO | digital_public_safety_platform | Case intake complete. case_id=DPSP-CASE-001 evidence_items=1
2026-07-22 17:22:27,304 | INFO | digital_public_safety_platform | Evidence routing complete. case_id=DPSP-CASE-001 plan={'run_evidence_engine': True, 'run_fraud_intelligence': True, 'run_counterfeit_check': False, 'run_network_intelligence': True, 'run_geospatial_intelligence': True, 'run_decision_engine': True}
2026-07-22 17:22:27,304 | INFO | digital_evidence_intelligence_engine | Ingested evidence input. source_channel=call_recording file_path=None
2026-07-22 17:22:27,306 | INFO | digital_evidence_intelligence_engine | Processed evidence item. type=Text evidence_type=Call Transcript status=Success quality=High engine=direct_text
2026-07-22 17:22:27,306 | INFO 

=== Notebook 8 Test Suite: end-to-end Digital Public Safety Platform (Revision 5) ===

--- Processing cases sequentially through the orchestrator ---

Processing DPSP-CASE-001 ...
DIGITAL PUBLIC SAFETY PLATFORM
Powered by Multi-Agent AI Intelligence
ET AI Hackathon 2026
ENGINE REGISTRY
Engine                                 Version    Loaded  Health    
--------------------------------------------------------------------------
Fraud Intelligence Engine              2.1        True    Unknown   
Decision Intelligence Engine           unknown    True    Unknown   
Digital Evidence Intelligence Engine   unknown    True    Unknown   
Counterfeit Currency Intelligence Engine unknown    True    Unknown   
Fraud Network Intelligence Engine      unknown    True    Unknown   
Geospatial Crime Pattern Intelligence Engine unknown    True    Unknown   
[DONE] Case intake complete (DPSP-CASE-001)
[DONE] Evidence processed


Batches: 100%|██████████| 1/1 [00:00<00:00, 29.08it/s]
2026-07-22 17:22:27,352 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-22 17:22:27,356 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (google/gemini-2.0-flash-exp:free), attempt 1, max_tokens=4096.


=== DEBUG BEFORE LLM CALL ===
User input length: 237
Signals entities: 3
Retrieved chunks: 5
✅ Prompt built successfully.


2026-07-22 17:22:28,452 | ERROR | fraud_intelligence_engine | Failed to analyse case 93838222-c012-4988-a7b4-1f13ccb365a8
Traceback (most recent call last):
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 1194, in analyze_case
    llm_output = run_llm_reasoning(user_input, signals, retrieved_chunks)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 742, in run_llm_reasoning
    response_json = _call_openrouter(messages, token_budget)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 697, in _call_openrouter
    raise RuntimeError(
RuntimeError: OpenRouter API request failed with status 404: {"error":{"message":"No endpoints found for google/gemini-2.0-flash-exp:free.","code":404},"user_id":"user_2zrK3eMXUFTiTU

In [7]:
# ============================================================================
# TEST CASE 1 - Digital Arrest Scam (High Risk)
# Run this in a FRESH kernel/session (or right after clearing the registry
# below) so network/geo numbers reflect ONLY this case, not leftover state
# from earlier synthetic test runs.
# ============================================================================

from digital_public_safety_platform import (
    CaseIntake, EvidenceItem, process_case, CASE_REGISTRY, Config
)

# --- Clear any leftover state from previous test runs in this session ---
# (This is the fix for the registry-contamination issue found earlier.
# If you're running this in a brand-new kernel, this is a no-op.)
CASE_REGISTRY.network_cases.clear()
CASE_REGISTRY.geo_cases.clear()
print(f"[INFO] Registry cleared. network_cases={len(CASE_REGISTRY.network_cases)} "
      f"geo_cases={len(CASE_REGISTRY.geo_cases)}")

# --- Build the test case exactly as specified ---
EVIDENCE_TEXT = (
    "Hello, this is Inspector Rajesh Sharma from Mumbai Crime Branch. "
    "Your Aadhaar card has been linked to illegal financial activities. "
    "A warrant has been issued in your name. Do not disconnect this call. "
    "Transfer ₹2,50,000 immediately to the RBI verification account. "
    "UPI ID: rbi-verification@okaxis "
    "Phone: +91 9876543210"
)

test_case_1 = CaseIntake(
    case_id="TEST-CASE-001-DIGITAL-ARREST",
    citizen_name="Rahul Sharma",
    city="Mumbai",
    state="Maharashtra",
    amount_involved=250000.0,
    priority="High",   # advisory only per design - does not override the real decision engine
    evidence=[
        EvidenceItem(
            evidence_type="call_recording",
            content=EVIDENCE_TEXT,
            metadata={"source_channel": "Call Transcript"},
        ),
    ],
)

print("\n=== Running Test Case 1 through process_case() ===\n")
master = process_case(test_case_1)

# ============================================================================
# RESULT INSPECTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST CASE 1 RESULT")
print("=" * 60)
print(master["platform_dashboard_text"])

print("\n--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---")
print(f"Fraud type        : {master['fraud_intelligence'].get('fraud_type')}")
print(f"Standalone risk    : {master['fraud_intelligence'].get('risk_score')}")
print(f"Severity           : {master['fraud_intelligence'].get('severity')}")
print(f"Confidence         : {master['fraud_intelligence'].get('confidence')}")
print(f"Indicators         : {master['fraud_intelligence'].get('indicators')}")

print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master["threat_fusion"]
print(f"Overall threat     : {fusion['overall_threat_score']} ({fusion['severity']})")
for comp, val in fusion["components"].items():
    base_w = fusion["base_weights"][comp]
    eff_w = fusion["effective_weights"][comp]
    tag = " [REDISTRIBUTED]" if comp in fusion["redistributed_signals"] else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")

print("\n--- Final Decision (Notebook 3) ---")
print(f"Decision           : {master['decision_intelligence'].get('case_decision')}")

print("\n--- Network Intelligence sanity check (should be minimal for a solo new case) ---")
network = master["fraud_network_intelligence"]
print(f"Connected cases    : {network.get('connected_cases')}")
print(f"Communities        : {len(network.get('communities', []))}")
print(f"Money mule accounts: {len(network.get('money_mule_accounts', []))}")
print(f"Fraud campaigns    : {len(network.get('fraud_campaigns', []))}")

print("\n--- KNOWN ISSUE CHECK: Critical-dilution ---")
fraud_severity = master["fraud_intelligence"].get("severity")
fraud_conf = master["fraud_intelligence"].get("confidence", 0)
final_decision = master["decision_intelligence"].get("case_decision")
if fraud_severity == "Critical" and fraud_conf >= 70 and final_decision not in ("Emergency",):
    print(f"[FLAG] Notebook 2 said Critical/{fraud_conf}% confidence, "
          f"but final decision is '{final_decision}', not Emergency. "
          f"This reproduces the dilution issue discussed earlier.")
else:
    print(f"[OK] No dilution flag triggered (fraud_severity={fraud_severity}, "
          f"conf={fraud_conf}, decision={final_decision}).")

print(f"\nFinal report: {master['final_report']}")

2026-07-22 16:32:32,591 | INFO | digital_public_safety_platform | Case intake complete. case_id=TEST-CASE-001-DIGITAL-ARREST evidence_items=1
2026-07-22 16:32:32,593 | INFO | digital_public_safety_platform | Evidence routing complete. case_id=TEST-CASE-001-DIGITAL-ARREST plan={'run_evidence_engine': True, 'run_fraud_intelligence': True, 'run_counterfeit_check': False, 'run_network_intelligence': True, 'run_geospatial_intelligence': True, 'run_decision_engine': True}
2026-07-22 16:32:32,593 | INFO | digital_evidence_intelligence_engine | Ingested evidence input. source_channel=Call Transcript file_path=None
2026-07-22 16:32:32,593 | INFO | digital_evidence_intelligence_engine | Processed evidence item. type=Text evidence_type=Call Transcript status=Success quality=High engine=direct_text
2026-07-22 16:32:32,595 | INFO | digital_evidence_intelligence_engine | Case TEST-CASE-001-DIGITAL-ARREST packaged. items=1 quality=High language=English duplicates=0 relationships=0
2026-07-22 16:32:32

[INFO] Registry cleared. network_cases=0 geo_cases=0

=== Running Test Case 1 through process_case() ===



Batches: 100%|██████████| 1/1 [00:00<00:00, 17.98it/s]
2026-07-22 16:32:32,657 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-22 16:32:32,659 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-22 16:32:32,659 | ERROR | fraud_intelligence_engine | Failed to analyse case fddef4ba-7696-4d1a-87ab-5d5e4ec90d49
Traceback (most recent call last):
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 1193, in analyze_case
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 719, in run_llm_reasoning
    user_prompt = _build_reasoning_prompt(user_input, signals, retrieved_chunks)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 643, in _call_openrouter
    "content (%d c


TEST CASE 1 RESULT
DIGITAL PUBLIC SAFETY PLATFORM
Case ID           : TEST-CASE-001-DIGITAL-ARREST
Threat Level      : MEDIUM
Fraud Type        : Digital Arrest Scam
Overall Threat    : 50.2/100
Confidence        : 50.8%
Money Mule Flag   : No
Campaign          : None identified
Connected Cases   : 0
Hotspot           : None (n/a)
Decision          : Urgent Action

--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---
Fraud type        : Digital Arrest Scam
Standalone risk    : 48.0
Severity           : Medium
Confidence         : 63.0
Indicators         : []

--- Threat Fusion (Notebook 8) ---
Overall threat     : 50.2 (Medium)


KeyError: 'base_weights'

In [10]:
print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master["threat_fusion"]
print(f"Overall threat     : {fusion.get('overall_threat_score', 'N/A')} ({fusion.get('severity', 'N/A')})")

components = fusion.get("components", {})
base_weights = fusion.get("base_weights", {})
eff_weights = fusion.get("effective_weights", {})
redist = fusion.get("redistributed_signals", [])

for comp, val in components.items():
    base_w = base_weights.get(comp, "N/A")
    eff_w = eff_weights.get(comp, "N/A")
    tag = " [REDISTRIBUTED]" if comp in redist else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")


--- Threat Fusion (Notebook 8) ---
Overall threat     : 50.2 (Medium)
  fraud_standalone_risk   :   48.0  base_w=N/A  eff_w=N/A
  network_adjusted_risk   :   48.0  base_w=N/A  eff_w=N/A
  geospatial_signal       :   95.0  base_w=N/A  eff_w=N/A
  counterfeit_signal      :    0.0  base_w=N/A  eff_w=N/A


In [16]:
# ============================================================================
# FIXED TEST CASE 1 - Digital Arrest Scam (High Risk)
# ============================================================================

from digital_public_safety_platform import (
    CaseIntake, EvidenceItem, process_case, CASE_REGISTRY, Config
)

# Clear registry
CASE_REGISTRY.network_cases.clear()
CASE_REGISTRY.geo_cases.clear()
print(f"[INFO] Registry cleared. network_cases={len(CASE_REGISTRY.network_cases)} "
      f"geo_cases={len(CASE_REGISTRY.geo_cases)}")

EVIDENCE_TEXT = (
    "Hello, this is Inspector Rajesh Sharma from Mumbai Crime Branch. "
    "Your Aadhaar card has been linked to illegal financial activities. "
    "A warrant has been issued in your name. Do not disconnect this call. "
    "Transfer ₹2,50,000 immediately to the RBI verification account. "
    "UPI ID: rbi-verification@okaxis "
    "Phone: +91 9876543210"
)

test_case_1 = CaseIntake(
    case_id="TEST-CASE-001-DIGITAL-ARREST",
    citizen_name="Rahul Sharma",
    city="Mumbai",
    state="Maharashtra",
    amount_involved=250000.0,
    priority="High",
    evidence=[
        EvidenceItem(
            evidence_type="call_recording",
            content=EVIDENCE_TEXT,
            metadata={"source_channel": "Call Transcript"},
        ),
    ],
)

print("\n=== Running Test Case 1 ===\n")
master = process_case(test_case_1)

# ============================================================================
# ROBUST RESULT INSPECTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST CASE 1 RESULT")
print("=" * 60)
print(master.get("platform_dashboard_text", "No dashboard text"))

print("\n--- Notebook 2 (Fraud Intelligence) ---")
fi = master.get("fraud_intelligence", {})
print(f"Fraud type        : {fi.get('fraud_type')}")
print(f"Standalone risk    : {fi.get('risk_score')}")
print(f"Severity           : {fi.get('severity')}")
print(f"Confidence         : {fi.get('confidence')}")
print(f"Indicators         : {fi.get('indicators')}")

print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master.get("threat_fusion", {})
print(f"Overall threat     : {fusion.get('overall_threat_score', 'N/A')} ({fusion.get('severity', 'N/A')})")

components = fusion.get("components", {})
for comp, val in components.items():
    base_w = fusion.get("base_weights", {}).get(comp, "N/A")
    eff_w = fusion.get("effective_weights", {}).get(comp, "N/A")
    tag = " [REDISTRIBUTED]" if comp in fusion.get("redistributed_signals", []) else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")

print("\n--- Final Decision (Notebook 3) ---")
di = master.get("decision_intelligence", {})
print(f"Decision           : {di.get('case_decision')}")

print("\n--- Network Intelligence ---")
net = master.get("fraud_network_intelligence", {})
print(f"Connected cases    : {net.get('connected_cases')}")
print(f"Communities        : {len(net.get('communities', []))}")
print(f"Money mule accounts: {len(net.get('money_mule_accounts', []))}")
print(f"Fraud campaigns    : {len(net.get('fraud_campaigns', []))}")

# Dilution check
fraud_sev = fi.get("severity")
fraud_conf = fi.get("confidence", 0)
final_dec = di.get("case_decision")
if fraud_sev == "Critical" and fraud_conf >= 70 and final_dec != "Emergency":
    print(f"\n[FLAG] Critical-dilution issue: Notebook 2 = Critical/{fraud_conf}%, Decision = {final_dec}")
else:
    print(f"\n[OK] No dilution (severity={fraud_sev}, conf={fraud_conf}, decision={final_dec})")

print(f"\nFinal report path: {master.get('final_report')}")

2026-07-22 16:46:57,604 | INFO | digital_public_safety_platform | Case intake complete. case_id=TEST-CASE-001-DIGITAL-ARREST evidence_items=1
2026-07-22 16:46:57,606 | INFO | digital_public_safety_platform | Evidence routing complete. case_id=TEST-CASE-001-DIGITAL-ARREST plan={'run_evidence_engine': True, 'run_fraud_intelligence': True, 'run_counterfeit_check': False, 'run_network_intelligence': True, 'run_geospatial_intelligence': True, 'run_decision_engine': True}
2026-07-22 16:46:57,606 | INFO | digital_evidence_intelligence_engine | Ingested evidence input. source_channel=Call Transcript file_path=None
2026-07-22 16:46:57,608 | INFO | digital_evidence_intelligence_engine | Processed evidence item. type=Text evidence_type=Call Transcript status=Success quality=High engine=direct_text
2026-07-22 16:46:57,608 | INFO | digital_evidence_intelligence_engine | Case TEST-CASE-001-DIGITAL-ARREST packaged. items=1 quality=High language=English duplicates=0 relationships=0
2026-07-22 16:46:57

[INFO] Registry cleared. network_cases=0 geo_cases=0

=== Running Test Case 1 ===



Batches: 100%|██████████| 1/1 [00:00<00:00, 15.79it/s]
2026-07-22 16:46:57,679 | INFO | fraud_intelligence_engine | Retrieved 5 relevant chunks (of 5 candidates) for query.
2026-07-22 16:46:57,679 | INFO | fraud_intelligence_engine | Sending request to OpenRouter (tencent/hy3:free), attempt 1, max_tokens=4096.
2026-07-22 16:47:00,608 | ERROR | fraud_intelligence_engine | Failed to analyse case affd4703-d91e-42a6-8df4-9c3f1663cece
Traceback (most recent call last):
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 1193, in analyze_case
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 719, in run_llm_reasoning
    user_prompt = _build_reasoning_prompt(user_input, signals, retrieved_chunks)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Shrivardhan\Downloads\ettttt2\ettttt2\server\fraud_intelligence_engine.py", line 674, in _call_openrouter
    "reasoning": {


TEST CASE 1 RESULT
DIGITAL PUBLIC SAFETY PLATFORM
Case ID           : TEST-CASE-001-DIGITAL-ARREST
Threat Level      : MEDIUM
Fraud Type        : Digital Arrest Scam
Overall Threat    : 50.2/100
Confidence        : 50.8%
Money Mule Flag   : No
Campaign          : None identified
Connected Cases   : 0
Hotspot           : None (n/a)
Decision          : Urgent Action

--- Notebook 2 (Fraud Intelligence) ---
Fraud type        : Digital Arrest Scam
Standalone risk    : 48.0
Severity           : Medium
Confidence         : 63.0
Indicators         : []

--- Threat Fusion (Notebook 8) ---
Overall threat     : 50.2 (Medium)
  fraud_standalone_risk   :   48.0  base_w=N/A  eff_w=N/A
  network_adjusted_risk   :   48.0  base_w=N/A  eff_w=N/A
  geospatial_signal       :   95.0  base_w=N/A  eff_w=N/A
  counterfeit_signal      :    0.0  base_w=N/A  eff_w=N/A

--- Final Decision (Notebook 3) ---
Decision           : Urgent Action

--- Network Intelligence ---
Connected cases    : []
Communities       

In [ ]:
import os
os.environ["OPENROUTER_API_KEY"] = "sk-or-....."   # ← your real key

# Digital Public Safety Platform

**ET AI Hackathon 2026 - Digital Public Safety Platform (PS6)**

**Notebook 8 - Master Orchestrator (Revision 4 - fixed dashboard ordering bug)**

This notebook is a direct, cell-by-cell conversion of `digital_public_safety_platform.py`. No code or comments have been changed; the original file has only been split into notebook cells along its existing numbered section headers, with a short markdown heading added before each section for readability.

In [ ]:
# digital_public_safety_platform.py
# ET AI Hackathon 2026 - Digital Public Safety Platform (PS6)
# Notebook 8 - Master Orchestrator (Revision 4 - fixed dashboard ordering bug)
#
# Mission (one sentence):
# Take a single citizen-reported case from intake to a merged, multi-
# audience intelligence package by calling every specialized engine
# (Notebooks 2-7) in the right order, fusing their outputs into one
# threat score, validating them against each other, and assembling one
# Digital Public Safety Intelligence Package.
#
# REVISION 4 FIX (this revision):
# build_administrator_dashboard(master) reads master["audit_trail"],
# master["execution_statistics"], and master["engine_health"]. Revision 3
# called it together with the other dashboards BEFORE those three keys
# were populated on `master`, which raised:
#     KeyError: 'audit_trail'
# (and would have raised KeyError: 'execution_statistics' /
# KeyError: 'engine_health' right after, for the same reason).
#
# The fix: populate, in this exact order, right after the decision
# pipeline finishes and BEFORE any dashboard is built:
#   1. master["incident_timeline"]
#   2. master["audit_trail"]              (interim - stages so far)
#   3. master["execution_statistics"]     (interim total_seconds)
#   4. master["engine_health"]            (derived from audit_trail)
#   5. master["executive_summary"]
#   6. master["platform_dashboard_text"]
# ONLY THEN are the five audience dashboards built (citizen, police, bank,
# telecom, administrator) - all five can now safely read any of the keys
# above. After the final report is generated (which adds one more audit
# entry for its own stage), audit_trail/execution_statistics/engine_health
# are refreshed once more so the totals/health stored on the returned
# master package are fully accurate, exactly as Revision 2 already did
# for audit_trail/execution_statistics alone.
#
# What this notebook is NOT:
#   - It is not an AI reasoning engine. It performs no fraud
#     classification, no computer vision, no graph analytics, and no
#     geospatial clustering itself - it calls the notebooks that do.
#   - It is not a replacement for Notebooks 2-7. It depends on them.
#   - It does not store data long-term. The in-memory Case Registry here
#     exists only to let the network (Notebook 6) and geospatial
#     (Notebook 7) engines see prior cases within a single running
#     process; a production deployment would back this with a database.
#
# Design approach - engine adapters:
# Each of Notebooks 2-7 is treated as an independent module. This file
# tries to import the real module first; if that import fails, or if the
# real module raises at call time, it automatically falls back to a
# small, deterministic, clearly-labeled stand-in so the pipeline still
# runs end-to-end and every stage is honestly labeled "real_engine" or
# "stub_adapter" (or "stub_fallback_after_real_engine_error") in the
# audit trail. Nothing here re-implements fraud classification, network
# analytics, or geospatial clustering as a permanent design choice - the
# stand-ins exist only for graceful degradation.

import hashlib
import json
import logging
import os
import re
import time
import uuid
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Dict, List, Optional, Tuple

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("digital_public_safety_platform")

## 1. Engine Availability - import real Notebook 2-7 modules

In [ ]:
# ============================================================================
# 1. Engine Availability - import real Notebook 2-7 modules
# ============================================================================
#
# Every notebook is imported the same way: try the real module first, and
# only fall back to a stand-in if the import itself fails. A second,
# independent fallback also exists at CALL time (see _run_stage and each
# adapter below) in case the real module imports fine but raises during
# execution (e.g. Notebook 2 needs OPENROUTER_API_KEY, Notebook 5 needs an
# actual image file on disk). Both fallback paths are logged so the audit
# trail never silently pretends a stub result came from a real engine.
#
# NOTE: the warnings you saw in your run ("fraud_intelligence_engine.py
# not found", etc.) are expected and harmless if those .py files are not
# sitting next to this file in the same working directory / on sys.path.
# The orchestrator is DESIGNED to keep working in that case (stub mode).
# To get "real_engine" instead of "stub_adapter" in the audit trail,
# place the actual notebook .py files (matching the import names below)
# in the same directory as this file, or on your PYTHONPATH.

try:
    import fraud_intelligence_engine as notebook2
    _NOTEBOOK2_AVAILABLE = True
except ImportError:
    notebook2 = None
    _NOTEBOOK2_AVAILABLE = False
    logger.warning("fraud_intelligence_engine.py not found; Notebook 2 will run in stub mode.")

try:
    import decision_intelligence_engine as notebook3
    _NOTEBOOK3_AVAILABLE = True
except ImportError:
    notebook3 = None
    _NOTEBOOK3_AVAILABLE = False
    logger.warning("decision_intelligence_engine.py not found; Notebook 3 will run in stub mode.")

try:
    import digital_evidence_intelligence_engine as notebook4
    _NOTEBOOK4_AVAILABLE = True
except ImportError:
    notebook4 = None
    _NOTEBOOK4_AVAILABLE = False
    logger.warning("digital_evidence_intelligence_engine.py not found; Notebook 4 will run in stub mode.")

try:
    import counterfeit_currency_intelligence_engine as notebook5
    _NOTEBOOK5_AVAILABLE = True
except ImportError:
    notebook5 = None
    _NOTEBOOK5_AVAILABLE = False
    logger.warning("counterfeit_currency_intelligence_engine.py not found; Notebook 5 will run in stub mode.")

try:
    import fraud_network_intelligence_engine as notebook6
    _NOTEBOOK6_AVAILABLE = True
except ImportError:
    notebook6 = None
    _NOTEBOOK6_AVAILABLE = False
    logger.warning("fraud_network_intelligence_engine.py not found; Notebook 6 will run in stub mode.")

try:
    import geospatial_crime_pattern_intelligence_engine as notebook7
    _NOTEBOOK7_AVAILABLE = True
except ImportError:
    notebook7 = None
    _NOTEBOOK7_AVAILABLE = False
    logger.warning("geospatial_crime_pattern_intelligence_engine.py not found; Notebook 7 will run in stub mode.")

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    from reportlab.lib import colors
    _REPORTLAB_AVAILABLE = True
except ImportError:
    _REPORTLAB_AVAILABLE = False


class EngineSource(str, Enum):
    '''Marks where a stage's output actually came from.'''
    REAL = "real_engine"
    STUB = "stub_adapter"
    STUB_FALLBACK = "stub_fallback_after_real_engine_error"
    SKIPPED = "skipped"

## 2. Configuration

In [ ]:
# ============================================================================
# 2. Configuration
# ============================================================================


class Config:
    NOTEBOOK_VERSION = "v4.0"

    # --- Decision thresholds (stub Notebook 3 fallback only) ---
    DECISION_EMERGENCY_RISK_MIN = 90.0
    DECISION_URGENT_RISK_MIN = 70.0
    DECISION_HUMAN_REVIEW_RISK_MIN = 40.0

    # --- Fraud stub (fallback Notebook 2) ---
    FRAUD_KEYWORD_SCORE_CAP = 60.0
    FRAUD_BASE_SCORE = 20.0

    # --- Severity bands, shared by the fraud stub and the fusion engine
    # so severity labels stay consistent whether Notebook 2 or the stub
    # produced the underlying risk score. ---
    SEVERITY_LOW_MAX = 30
    SEVERITY_MEDIUM_MAX = 60
    SEVERITY_HIGH_MAX = 85

    # --- Counterfeit stub (fallback Notebook 5) ---
    COUNTERFEIT_SUSPICION_THRESHOLD = 0.55

    # --- Threat Fusion Engine weights (must sum to 1.0). Network-adjusted
    # risk gets the largest weight because it already folds in the
    # standalone fraud score plus corroborating network evidence. ---
    FUSION_WEIGHT_NETWORK_RISK = 0.45
    FUSION_WEIGHT_FRAUD_RISK = 0.20
    FUSION_WEIGHT_GEO_SIGNAL = 0.20
    FUSION_WEIGHT_COUNTERFEIT_SIGNAL = 0.15

    # --- Confidence Fusion weights (must sum to 1.0) ---
    CONFIDENCE_WEIGHT_FRAUD = 0.40
    CONFIDENCE_WEIGHT_NETWORK = 0.30
    CONFIDENCE_WEIGHT_GEO = 0.20
    CONFIDENCE_WEIGHT_COUNTERFEIT = 0.10

    # --- Hotspot / district priority -> numeric signal (0-100) used only
    # inside the Threat Fusion Engine, never shown to the user directly. ---
    PRIORITY_TO_SCORE = {"Critical": 95.0, "High": 75.0, "Medium": 50.0, "Low": 20.0}


CONFIG = Config()
assert abs(
    CONFIG.FUSION_WEIGHT_NETWORK_RISK + CONFIG.FUSION_WEIGHT_FRAUD_RISK
    + CONFIG.FUSION_WEIGHT_GEO_SIGNAL + CONFIG.FUSION_WEIGHT_COUNTERFEIT_SIGNAL - 1.0
) < 1e-6, "Threat fusion weights must sum to 1.0."
assert abs(
    CONFIG.CONFIDENCE_WEIGHT_FRAUD + CONFIG.CONFIDENCE_WEIGHT_NETWORK
    + CONFIG.CONFIDENCE_WEIGHT_GEO + CONFIG.CONFIDENCE_WEIGHT_COUNTERFEIT - 1.0
) < 1e-6, "Confidence fusion weights must sum to 1.0."

logger.info(
    "Notebook 8 configuration loaded. version=%s notebook2=%s notebook3=%s notebook4=%s notebook5=%s notebook6=%s notebook7=%s",
    CONFIG.NOTEBOOK_VERSION, _NOTEBOOK2_AVAILABLE, _NOTEBOOK3_AVAILABLE,
    _NOTEBOOK4_AVAILABLE, _NOTEBOOK5_AVAILABLE, _NOTEBOOK6_AVAILABLE, _NOTEBOOK7_AVAILABLE,
)


class OrchestrationError(Exception):
    '''Raised when Notebook 8 cannot produce a valid Digital Public Safety Intelligence Package.'''

## 3. Module 1 - Case Intake (input contract)

In [ ]:
# ============================================================================
# TEST CASE 1 - Digital Arrest Scam (High Risk)
# Run this in a FRESH kernel/session (or right after clearing the registry
# below) so network/geo numbers reflect ONLY this case, not leftover state
# from earlier synthetic test runs.
# ============================================================================

from digital_public_safety_platform import (
    CaseIntake, EvidenceItem, process_case, CASE_REGISTRY, Config
)

# --- Clear any leftover state from previous test runs in this session ---
# (This is the fix for the registry-contamination issue found earlier.
# If you're running this in a brand-new kernel, this is a no-op.)
CASE_REGISTRY.network_cases.clear()
CASE_REGISTRY.geo_cases.clear()
print(f"[INFO] Registry cleared. network_cases={len(CASE_REGISTRY.network_cases)} "
      f"geo_cases={len(CASE_REGISTRY.geo_cases)}")

# --- Build the test case exactly as specified ---
EVIDENCE_TEXT = (
    "Hello, this is Inspector Rajesh Sharma from Mumbai Crime Branch. "
    "Your Aadhaar card has been linked to illegal financial activities. "
    "A warrant has been issued in your name. Do not disconnect this call. "
    "Transfer ₹2,50,000 immediately to the RBI verification account. "
    "UPI ID: rbi-verification@okaxis "
    "Phone: +91 9876543210"
)

test_case_1 = CaseIntake(
    case_id="TEST-CASE-001-DIGITAL-ARREST",
    citizen_name="Rahul Sharma",
    city="Mumbai",
    state="Maharashtra",
    amount_involved=250000.0,
    priority="High",   # advisory only per design - does not override the real decision engine
    evidence=[
        EvidenceItem(
            evidence_type="call_recording",
            content=EVIDENCE_TEXT,
            metadata={"source_channel": "Call Transcript"},
        ),
    ],
)

print("\n=== Running Test Case 1 through process_case() ===\n")
master = process_case(test_case_1)

# ============================================================================
# RESULT INSPECTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST CASE 1 RESULT")
print("=" * 60)
print(master["platform_dashboard_text"])

print("\n--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---")
print(f"Fraud type        : {master['fraud_intelligence'].get('fraud_type')}")
print(f"Standalone risk    : {master['fraud_intelligence'].get('risk_score')}")
print(f"Severity           : {master['fraud_intelligence'].get('severity')}")
print(f"Confidence         : {master['fraud_intelligence'].get('confidence')}")
print(f"Indicators         : {master['fraud_intelligence'].get('indicators')}")

print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master["threat_fusion"]
print(f"Overall threat     : {fusion['overall_threat_score']} ({fusion['severity']})")
for comp, val in fusion["components"].items():
    base_w = fusion["base_weights"][comp]
    eff_w = fusion["effective_weights"][comp]
    tag = " [REDISTRIBUTED]" if comp in fusion["redistributed_signals"] else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")

print("\n--- Final Decision (Notebook 3) ---")
print(f"Decision           : {master['decision_intelligence'].get('case_decision')}")

print("\n--- Network Intelligence sanity check (should be minimal for a solo new case) ---")
network = master["fraud_network_intelligence"]
print(f"Connected cases    : {network.get('connected_cases')}")
print(f"Communities        : {len(network.get('communities', []))}")
print(f"Money mule accounts: {len(network.get('money_mule_accounts', []))}")
print(f"Fraud campaigns    : {len(network.get('fraud_campaigns', []))}")

print("\n--- KNOWN ISSUE CHECK: Critical-dilution ---")
fraud_severity = master["fraud_intelligence"].get("severity")
fraud_conf = master["fraud_intelligence"].get("confidence", 0)
final_decision = master["decision_intelligence"].get("case_decision")
if fraud_severity == "Critical" and fraud_conf >= 70 and final_decision not in ("Emergency",):
    print(f"[FLAG] Notebook 2 said Critical/{fraud_conf}% confidence, "
          f"but final decision is '{final_decision}', not Emergency. "
          f"This reproduces the dilution issue discussed earlier.")
else:
    print(f"[OK] No dilution flag triggered (fraud_severity={fraud_severity}, "
          f"conf={fraud_conf}, decision={final_decision}).")

print(f"\nFinal report: {master['final_report']}")

## 4. Module 9 - Case Registry (shared, in-memory, across process_case calls)

In [ ]:
# ============================================================================
# TEST CASE 1 - Digital Arrest Scam (High Risk)
# Run this in a FRESH kernel/session (or right after clearing the registry
# below) so network/geo numbers reflect ONLY this case, not leftover state
# from earlier synthetic test runs.
# ============================================================================

from digital_public_safety_platform import (
    CaseIntake, EvidenceItem, process_case, CASE_REGISTRY, Config
)

# --- Clear any leftover state from previous test runs in this session ---
# (This is the fix for the registry-contamination issue found earlier.
# If you're running this in a brand-new kernel, this is a no-op.)
CASE_REGISTRY.network_cases.clear()
CASE_REGISTRY.geo_cases.clear()
print(f"[INFO] Registry cleared. network_cases={len(CASE_REGISTRY.network_cases)} "
      f"geo_cases={len(CASE_REGISTRY.geo_cases)}")

# --- Build the test case exactly as specified ---
EVIDENCE_TEXT = (
    "Hello, this is Inspector Rajesh Sharma from Mumbai Crime Branch. "
    "Your Aadhaar card has been linked to illegal financial activities. "
    "A warrant has been issued in your name. Do not disconnect this call. "
    "Transfer ₹2,50,000 immediately to the RBI verification account. "
    "UPI ID: rbi-verification@okaxis "
    "Phone: +91 9876543210"
)

test_case_1 = CaseIntake(
    case_id="TEST-CASE-001-DIGITAL-ARREST",
    citizen_name="Rahul Sharma",
    city="Mumbai",
    state="Maharashtra",
    amount_involved=250000.0,
    priority="High",   # advisory only per design - does not override the real decision engine
    evidence=[
        EvidenceItem(
            evidence_type="call_recording",
            content=EVIDENCE_TEXT,
            metadata={"source_channel": "Call Transcript"},
        ),
    ],
)

print("\n=== Running Test Case 1 through process_case() ===\n")
master = process_case(test_case_1)

# ============================================================================
# RESULT INSPECTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST CASE 1 RESULT")
print("=" * 60)
print(master["platform_dashboard_text"])

print("\n--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---")
print(f"Fraud type        : {master['fraud_intelligence'].get('fraud_type')}")
print(f"Standalone risk    : {master['fraud_intelligence'].get('risk_score')}")
print(f"Severity           : {master['fraud_intelligence'].get('severity')}")
print(f"Confidence         : {master['fraud_intelligence'].get('confidence')}")
print(f"Indicators         : {master['fraud_intelligence'].get('indicators')}")

print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master["threat_fusion"]
print(f"Overall threat     : {fusion['overall_threat_score']} ({fusion['severity']})")
for comp, val in fusion["components"].items():
    base_w = fusion["base_weights"][comp]
    eff_w = fusion["effective_weights"][comp]
    tag = " [REDISTRIBUTED]" if comp in fusion["redistributed_signals"] else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")

print("\n--- Final Decision (Notebook 3) ---")
print(f"Decision           : {master['decision_intelligence'].get('case_decision')}")

print("\n--- Network Intelligence sanity check (should be minimal for a solo new case) ---")
network = master["fraud_network_intelligence"]
print(f"Connected cases    : {network.get('connected_cases')}")
print(f"Communities        : {len(network.get('communities', []))}")
print(f"Money mule accounts: {len(network.get('money_mule_accounts', []))}")
print(f"Fraud campaigns    : {len(network.get('fraud_campaigns', []))}")

print("\n--- KNOWN ISSUE CHECK: Critical-dilution ---")
fraud_severity = master["fraud_intelligence"].get("severity")
fraud_conf = master["fraud_intelligence"].get("confidence", 0)
final_decision = master["decision_intelligence"].get("case_decision")
if fraud_severity == "Critical" and fraud_conf >= 70 and final_decision not in ("Emergency",):
    print(f"[FLAG] Notebook 2 said Critical/{fraud_conf}% confidence, "
          f"but final decision is '{final_decision}', not Emergency. "
          f"This reproduces the dilution issue discussed earlier.")
else:
    print(f"[OK] No dilution flag triggered (fraud_severity={fraud_severity}, "
          f"conf={fraud_conf}, decision={final_decision}).")

print(f"\nFinal report: {master['final_report']}")

## 5. Module 2 - Evidence Router (smarter, per-entity routing)

In [21]:
# ============================================================================
# TEST CASE 1 - Digital Arrest Scam (High Risk)
# Run this in a FRESH kernel/session (or right after clearing the registry
# below) so network/geo numbers reflect ONLY this case, not leftover state
# from earlier synthetic test runs.
# ============================================================================

from digital_public_safety_platform import (
    CaseIntake, EvidenceItem, process_case, CASE_REGISTRY, Config
)

# --- Clear any leftover state from previous test runs in this session ---
# (This is the fix for the registry-contamination issue found earlier.
# If you're running this in a brand-new kernel, this is a no-op.)
CASE_REGISTRY.network_cases.clear()
CASE_REGISTRY.geo_cases.clear()
print(f"[INFO] Registry cleared. network_cases={len(CASE_REGISTRY.network_cases)} "
      f"geo_cases={len(CASE_REGISTRY.geo_cases)}")

# --- Build the test case exactly as specified ---
EVIDENCE_TEXT = (
    "Hello, this is Inspector Rajesh Sharma from Mumbai Crime Branch. "
    "Your Aadhaar card has been linked to illegal financial activities. "
    "A warrant has been issued in your name. Do not disconnect this call. "
    "Transfer ₹2,50,000 immediately to the RBI verification account. "
    "UPI ID: rbi-verification@okaxis "
    "Phone: +91 9876543210"
)

test_case_1 = CaseIntake(
    case_id="TEST-CASE-001-DIGITAL-ARREST",
    citizen_name="Rahul Sharma",
    city="Mumbai",
    state="Maharashtra",
    amount_involved=250000.0,
    priority="High",   # advisory only per design - does not override the real decision engine
    evidence=[
        EvidenceItem(
            evidence_type="call_recording",
            content=EVIDENCE_TEXT,
            metadata={"source_channel": "Call Transcript"},
        ),
    ],
)

print("\n=== Running Test Case 1 through process_case() ===\n")
master = process_case(test_case_1)

# ============================================================================
# RESULT INSPECTION
# ============================================================================

print("\n" + "=" * 60)
print("TEST CASE 1 RESULT")
print("=" * 60)
print(master["platform_dashboard_text"])

print("\n--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---")
print(f"Fraud type        : {master['fraud_intelligence'].get('fraud_type')}")
print(f"Standalone risk    : {master['fraud_intelligence'].get('risk_score')}")
print(f"Severity           : {master['fraud_intelligence'].get('severity')}")
print(f"Confidence         : {master['fraud_intelligence'].get('confidence')}")
print(f"Indicators         : {master['fraud_intelligence'].get('indicators')}")

print("\n--- Threat Fusion (Notebook 8) ---")
fusion = master["threat_fusion"]
print(f"Overall threat     : {fusion['overall_threat_score']} ({fusion['severity']})")
for comp, val in fusion["components"].items():
    base_w = fusion["base_weights"][comp]
    eff_w = fusion["effective_weights"][comp]
    tag = " [REDISTRIBUTED]" if comp in fusion["redistributed_signals"] else ""
    print(f"  {comp:24s}: {val:6.1f}  base_w={base_w}  eff_w={eff_w}{tag}")

print("\n--- Final Decision (Notebook 3) ---")
print(f"Decision           : {master['decision_intelligence'].get('case_decision')}")

print("\n--- Network Intelligence sanity check (should be minimal for a solo new case) ---")
network = master["fraud_network_intelligence"]
print(f"Connected cases    : {network.get('connected_cases')}")
print(f"Communities        : {len(network.get('communities', []))}")
print(f"Money mule accounts: {len(network.get('money_mule_accounts', []))}")
print(f"Fraud campaigns    : {len(network.get('fraud_campaigns', []))}")

print("\n--- KNOWN ISSUE CHECK: Critical-dilution ---")
fraud_severity = master["fraud_intelligence"].get("severity")
fraud_conf = master["fraud_intelligence"].get("confidence", 0)
final_decision = master["decision_intelligence"].get("case_decision")
if fraud_severity == "Critical" and fraud_conf >= 70 and final_decision not in ("Emergency",):
    print(f"[FLAG] Notebook 2 said Critical/{fraud_conf}% confidence, "
          f"but final decision is '{final_decision}', not Emergency. "
          f"This reproduces the dilution issue discussed earlier.")
else:
    print(f"[OK] No dilution flag triggered (fraud_severity={fraud_severity}, "
          f"conf={fraud_conf}, decision={final_decision}).")

print(f"\nFinal report: {master['final_report']}")

2026-07-22 10:55:34,617 | INFO | digital_public_safety_platform | Case intake complete. case_id=TEST-CASE-001-DIGITAL-ARREST evidence_items=1
2026-07-22 10:55:34,617 | INFO | digital_public_safety_platform | Evidence routing complete. case_id=TEST-CASE-001-DIGITAL-ARREST plan={'run_evidence_engine': True, 'run_fraud_intelligence': True, 'run_counterfeit_check': False, 'run_network_intelligence': True, 'run_geospatial_intelligence': True, 'run_decision_engine': True}
2026-07-22 10:55:34,619 | INFO | digital_evidence_intelligence_engine | Ingested evidence input. source_channel=Call Transcript file_path=None
2026-07-22 10:55:34,619 | INFO | digital_evidence_intelligence_engine | Processed evidence item. type=Text evidence_type=Call Transcript status=Success quality=High engine=direct_text
2026-07-22 10:55:34,621 | INFO | digital_evidence_intelligence_engine | Case TEST-CASE-001-DIGITAL-ARREST packaged. items=1 quality=High language=English duplicates=0 relationships=0
2026-07-22 10:55:34

[INFO] Registry cleared. network_cases=0 geo_cases=0

=== Running Test Case 1 through process_case() ===



2026-07-22 10:55:34,805 | INFO | fraud_network_intelligence_engine | Network visualization saved to /tmp/notebook6_graphs\fraud_network_graph_74e2b137.png
2026-07-22 10:55:34,805 | INFO | fraud_network_intelligence_engine | Fraud network analysis complete. focus_case=TEST-CASE-001-DIGITAL-ARREST connected_cases=0 communities=1 mules=0
2026-07-22 10:55:34,805 | INFO | geospatial_crime_pattern_intelligence_engine | Location collection complete. cases=1
2026-07-22 10:55:34,805 | INFO | geospatial_crime_pattern_intelligence_engine | Geocoding complete. {'already_had_coordinates': 0, 'geocoded_from_city': 1, 'unresolved': 0}
2026-07-22 10:55:34,805 | INFO | geospatial_crime_pattern_intelligence_engine | Campaign spread analysis complete. campaigns_analyzed=0
2026-07-22 10:55:34,805 | INFO | geospatial_crime_pattern_intelligence_engine | District risk ranking complete. districts_ranked=1
2026-07-22 10:55:34,817 | INFO | geospatial_crime_pattern_intelligence_engine | Predictive hotspot analys


TEST CASE 1 RESULT
DIGITAL PUBLIC SAFETY PLATFORM
Case ID           : TEST-CASE-001-DIGITAL-ARREST
Threat Level      : MEDIUM
Fraud Type        : Digital Arrest Scam
Overall Threat    : 50.2/100
Confidence        : 50.8%
Money Mule Flag   : No
Campaign          : None identified
Connected Cases   : 0
Hotspot           : None (n/a)
Decision          : Urgent Action

--- Notebook 2 (Fraud Intelligence) - ground truth from the LLM ---
Fraud type        : Digital Arrest Scam
Standalone risk    : 48.0
Severity           : Medium
Confidence         : 63.0
Indicators         : []

--- Threat Fusion (Notebook 8) ---
Overall threat     : 50.2 (Medium)


KeyError: 'base_weights'

## 6. Module 3 - Run Evidence Engine (real Notebook 4 call, stub fallback)

In [ ]:
# ============================================================================
# 6. Module 3 - Run Evidence Engine (real Notebook 4 call, stub fallback)
# ============================================================================

_ENTITY_PATTERNS: Dict[str, re.Pattern] = {
    "phone_numbers": re.compile(r"(?:\+91[\-\s]?)?[6-9]\d{9}\b"),
    "upi_ids": re.compile(r"\b[\w.\-]{2,}@[a-zA-Z]{2,}\b"),
    "emails": re.compile(r"\b[\w.\-]+@[\w\-]+\.[a-zA-Z]{2,}\b"),
    "bank_accounts": re.compile(r"\b\d{9,18}\b"),
    "urls": re.compile(r"https?://[^\s]+"),
}


def _extract_entities_from_text_stub(text: str) -> Dict[str, List[str]]:
    '''Fallback-only regex entity extraction, used when Notebook 4 is unavailable or errors.'''
    entities: Dict[str, List[str]] = defaultdict(list)
    for entity_type, pattern in _ENTITY_PATTERNS.items():
        for match in pattern.findall(text):
            if match not in entities[entity_type]:
                entities[entity_type].append(match)
    entities["emails"] = [e for e in entities["emails"] if "." in e.split("@", 1)[-1]]
    entities["upi_ids"] = [u for u in entities["upi_ids"] if u not in entities["emails"]]
    return dict(entities)


def _to_notebook4_inputs(case: CaseIntake) -> List[Any]:
    '''Converts our EvidenceItem list into Notebook 4's EvidenceInput contract.'''
    inputs = []
    for item in case.evidence:
        file_path = item.metadata.get("file_path")
        inputs.append(notebook4.EvidenceInput(
            file_path=file_path,
            raw_text=item.content if not file_path else None,
            source_channel=item.metadata.get("source_channel", item.evidence_type),
            submitted_at=item.metadata.get("submitted_at", case.timestamp),
            original_filename=item.metadata.get("original_filename"),
            override_extracted_text=item.metadata.get("override_extracted_text"),
        ))
    return inputs


def run_evidence_engine(case: CaseIntake) -> Dict[str, Any]:
    '''Module 3 entry point (Notebook 4 adapter, real-first with stub fallback).'''
    if _NOTEBOOK4_AVAILABLE:
        try:
            inputs = _to_notebook4_inputs(case)
            package = notebook4.package_case_evidence(inputs, case_id=case.case_id)
            package["engine_source"] = EngineSource.REAL.value
            return package
        except Exception as exc:
            logger.warning("Notebook 4 raised at call time; falling back to stub evidence extraction. error=%s", exc)

    combined_text = " ".join(item.content for item in case.evidence if item.content)
    entities = _extract_entities_from_text_stub(combined_text)
    evidence_types_seen = sorted({item.evidence_type for item in case.evidence})

    return {
        "case_id": case.case_id,
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK4_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "evidence_types_seen": evidence_types_seen,
        "metadata": entities,               # aligned with Notebook 4's top-level "metadata" key name
        "extracted_entities": entities,      # kept for backward-compatible readers
        "text": combined_text,
        "evidence_quality": "Unknown",
        "evidence_summary": combined_text[:280] + ("..." if len(combined_text) > 280 else ""),
    }

## 7. Module 4 - Run Fraud Intelligence (real Notebook 2 call, stub fallback)

In [ ]:
# ============================================================================
# 7. Module 4 - Run Fraud Intelligence (real Notebook 2 call, stub fallback)
# ============================================================================

_FRAUD_TYPE_KEYWORDS: Dict[str, List[str]] = {
    "Digital Arrest Scam": ["digital arrest", "rbi", "cbi", "arrest warrant", "video call", "police officer", "customs"],
    "UPI / Payment Fraud": ["upi", "refund", "cashback", "wrong transaction", "collect request"],
    "Romance Scam": ["love", "relationship", "gift", "customs duty", "dating"],
    "Job Scam": ["job offer", "work from home", "registration fee", "part time job", "recruiter"],
    "Lottery Scam": ["lottery", "prize", "winner", "claim your", "lucky draw"],
    "Investment Scam": ["investment", "guaranteed return", "trading tips", "stock tips", "crypto"],
}


def _severity_from_risk(risk_score: float) -> str:
    '''Shared severity banding, kept consistent with Notebook 2's own bands.'''
    if risk_score <= CONFIG.SEVERITY_LOW_MAX:
        return "Low"
    if risk_score <= CONFIG.SEVERITY_MEDIUM_MAX:
        return "Medium"
    if risk_score <= CONFIG.SEVERITY_HIGH_MAX:
        return "High"
    return "Critical"


def _fraud_stub(case: CaseIntake, evidence_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Deterministic, keyword-weighted fallback classifier used only when Notebook 2 is unavailable or errors.'''
    text = (evidence_package.get("text") or " ".join(item.content for item in case.evidence if item.content)).lower()

    scores: Dict[str, int] = {}
    matched_keywords: Dict[str, List[str]] = {}
    for fraud_type, keywords in _FRAUD_TYPE_KEYWORDS.items():
        hits = [kw for kw in keywords if kw in text]
        if hits:
            scores[fraud_type] = len(hits)
            matched_keywords[fraud_type] = hits

    if scores:
        fraud_type = max(scores, key=scores.get)
        keyword_component = min(CONFIG.FRAUD_KEYWORD_SCORE_CAP, scores[fraud_type] * 15.0)
        reasoning = [f"Matched signal keywords for '{fraud_type}': {', '.join(matched_keywords[fraud_type])}."]
    else:
        fraud_type = "Unclassified Suspicious Activity"
        keyword_component = 0.0
        reasoning = ["No known fraud-pattern keywords were matched in the submitted evidence text."]

    entities = evidence_package.get("metadata") or evidence_package.get("extracted_entities") or {}
    entity_count = sum(len(v) for v in entities.values() if isinstance(v, list))
    entity_component = min(15.0, entity_count * 2.0)
    if entity_count:
        reasoning.append(f"{entity_count} identifiable entity(ies) extracted from evidence.")

    amount_component = min(5.0, case.amount_involved / 20000.0) if case.amount_involved else 0.0
    if case.amount_involved:
        reasoning.append(f"Amount involved: Rs {case.amount_involved:,.0f}.")

    risk_score = round(min(100.0, CONFIG.FRAUD_BASE_SCORE + keyword_component + entity_component + amount_component), 1)
    confidence = round(min(95.0, 40.0 + keyword_component + entity_component), 1)

    return {
        "case_id": case.case_id,
        "timestamp": case.timestamp,
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK2_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "fraud_type": fraud_type,
        "risk_score": risk_score,
        "confidence": confidence,
        "severity": _severity_from_risk(risk_score),
        "indicators": [],
        "entities": entities,
        "citations": {},
        "reasoning": reasoning,
        "summary": reasoning[0] if reasoning else "",
        "matched_keywords": matched_keywords,
    }


def run_fraud_intelligence(case: CaseIntake, evidence_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 4 entry point (Notebook 2 adapter, real-first with stub fallback).'''
    if _NOTEBOOK2_AVAILABLE:
        try:
            combined_text = evidence_package.get("text") or " ".join(item.content for item in case.evidence if item.content)
            result = notebook2.analyze_case(combined_text)
            result["engine_source"] = EngineSource.REAL.value
            result.setdefault("reasoning", [result.get("summary", "")])
            return result
        except Exception as exc:
            logger.warning("Notebook 2 raised at call time (likely missing OPENROUTER_API_KEY or no network); "
                            "falling back to stub fraud classification. error=%s", exc)

    return _fraud_stub(case, evidence_package)

## 8. Module 5 - Counterfeit Check (real Notebook 5 call, stub fallback)

In [ ]:
# ============================================================================
# 8. Module 5 - Counterfeit Check (real Notebook 5 call, stub fallback)
# ============================================================================


def _counterfeit_stub(currency_items: List[EvidenceItem]) -> Dict[str, Any]:
    '''Deterministic pseudo-scored fallback, used only when Notebook 5 is unavailable, errors, or no file is on disk.'''
    findings = []
    max_suspicion = 0.0
    for item in currency_items:
        digest_source = item.content or item.metadata.get("file_path", item.evidence_type)
        digest = hashlib.sha256(digest_source.encode("utf-8")).hexdigest()
        pseudo_score = (int(digest[:8], 16) % 1000) / 1000.0
        if item.metadata.get("citizen_reported_suspicious"):
            pseudo_score = min(1.0, pseudo_score + 0.25)
        max_suspicion = max(max_suspicion, pseudo_score)
        findings.append({
            "evidence_content": digest_source,
            "suspicion_score": round(pseudo_score, 3),
            "citizen_flagged": bool(item.metadata.get("citizen_reported_suspicious", False)),
        })

    verdict = "Likely Counterfeit" if max_suspicion >= CONFIG.COUNTERFEIT_SUSPICION_THRESHOLD else "Inconclusive - Needs Manual Review"

    return {
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK5_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "items_checked": len(currency_items),
        "findings": findings,
        "max_suspicion_score": round(max_suspicion, 3),
        "verdict": verdict,
        "note": "Pseudo-scored stand-in; treat verdict as advisory only.",
    }


def run_counterfeit_check(case: CaseIntake) -> Optional[Dict[str, Any]]:
    '''Module 5 entry point (Notebook 5 adapter). Returns None if there is nothing to check.'''
    currency_items = [item for item in case.evidence if item.evidence_type == "currency_image"]
    if not currency_items:
        return None

    if _NOTEBOOK5_AVAILABLE:
        items_with_files = [item for item in currency_items if item.metadata.get("file_path")]
        if items_with_files:
            try:
                per_item_results = []
                max_counterfeit_prob = 0.0
                for item in items_with_files:
                    result = notebook5.analyze_currency_image(
                        item.metadata["file_path"],
                        denomination_hint=item.metadata.get("denomination_hint"),
                        known_serial_database=item.metadata.get("known_serial_database"),
                    )
                    per_item_results.append(result)
                    max_counterfeit_prob = max(max_counterfeit_prob, result["currency_analysis"]["counterfeit_probability"])

                verdict = "Likely Counterfeit" if max_counterfeit_prob >= CONFIG.COUNTERFEIT_SUSPICION_THRESHOLD else "Likely Genuine"
                return {
                    "engine_source": EngineSource.REAL.value,
                    "items_checked": len(per_item_results),
                    "per_item_results": per_item_results,
                    "max_suspicion_score": round(max_counterfeit_prob, 3),
                    "verdict": verdict,
                }
            except Exception as exc:
                logger.warning("Notebook 5 raised at call time; falling back to stub counterfeit check. error=%s", exc)

    return _counterfeit_stub(currency_items)

## 9. Module 6 - Run Network Intelligence (real Notebook 6 call)

In [ ]:
# ============================================================================
# 9. Module 6 - Run Network Intelligence (real Notebook 6 call)
# ============================================================================


def _build_network_case_record(case: CaseIntake, evidence_package: Dict[str, Any], fraud_package: Dict[str, Any]) -> Any:
    entities = evidence_package.get("metadata") or evidence_package.get("extracted_entities") or {}
    return notebook6.CaseRecord(
        case_id=case.case_id,
        victim_id=case.victim_id,
        fraud_type=fraud_package.get("fraud_type"),
        risk_score=fraud_package.get("risk_score", 0.0),
        phone_numbers=entities.get("phone_numbers", []),
        upi_ids=entities.get("upi_ids", []),
        emails=entities.get("emails", []),
        bank_accounts=entities.get("bank_accounts", []),
        urls=entities.get("urls", []),
        organizations=entities.get("organizations", []),
        amount_involved=case.amount_involved,
        city=case.city,
        state=case.state,
        timestamp=case.timestamp,
    )


def run_network_intelligence(case: CaseIntake, evidence_package: Dict[str, Any], fraud_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 6 entry point (real Notebook 6 call, with a minimal stub fallback).'''
    if not _NOTEBOOK6_AVAILABLE:
        return {
            "engine_source": EngineSource.STUB.value,
            "connected_cases": [], "communities": [], "fraud_campaigns": [], "money_mule_accounts": [],
            "central_actor": None,
            "risk_propagation": {
                "original_risk": fraud_package.get("risk_score", 0.0),
                "network_adjusted_risk": fraud_package.get("risk_score", 0.0),
                "boost_applied": 0.0, "driving_entity": None,
                "reasons": ["Notebook 6 module not available; network-adjusted risk defaults to the standalone score."],
            },
            "note": "fraud_network_intelligence_engine.py was not importable.",
        }

    try:
        record = _build_network_case_record(case, evidence_package, fraud_package)
        CASE_REGISTRY.register_network_case(case.case_id, record)

        package = notebook6.analyze_fraud_network(
            CASE_REGISTRY.all_network_cases(),
            focus_case_id=case.case_id,
            save_visualization=True,
            generate_report=False,   # Notebook 8 produces its own single merged report
        )
        package["engine_source"] = EngineSource.REAL.value
        return package
    except Exception as exc:
        logger.warning("Notebook 6 raised at call time; falling back to stub network package. error=%s", exc)
        return {
            "engine_source": EngineSource.STUB_FALLBACK.value,
            "connected_cases": [], "communities": [], "fraud_campaigns": [], "money_mule_accounts": [],
            "central_actor": None,
            "risk_propagation": {
                "original_risk": fraud_package.get("risk_score", 0.0),
                "network_adjusted_risk": fraud_package.get("risk_score", 0.0),
                "boost_applied": 0.0, "driving_entity": None,
                "reasons": [f"Notebook 6 raised an error at call time ({exc}); defaulting to the standalone risk score."],
            },
        }

## 10. Module 7 - Run Geospatial Intelligence (real Notebook 7 call)

In [ ]:
# ============================================================================
# 10. Module 7 - Run Geospatial Intelligence (real Notebook 7 call)
# ============================================================================


def _find_case_campaign_id(case_id: str, network_package: Dict[str, Any]) -> Optional[str]:
    for campaign in network_package.get("fraud_campaigns", []):
        if case_id in campaign.get("linked_cases", []):
            return campaign["campaign_id"]
    return None


def _find_case_community_id(case_id: str, network_package: Dict[str, Any]) -> Optional[str]:
    for community in network_package.get("communities", []):
        if case_id in community.get("connected_cases", []):
            return community["community_id"]
    return None


def _case_touches_mule_account(case_id: str, network_package: Dict[str, Any]) -> bool:
    return any(case_id in mule.get("connected_cases", []) for mule in network_package.get("money_mule_accounts", []))


def run_geospatial_intelligence(case: CaseIntake, fraud_package: Dict[str, Any], network_package: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 7 entry point (real Notebook 7 call, with a minimal stub fallback).'''
    if not _NOTEBOOK7_AVAILABLE:
        return {
            "engine_source": EngineSource.STUB.value,
            "hotspots": [], "district_risk": [], "predicted_hotspots": [], "campaign_spread": [],
            "resource_recommendations": [], "nearest_cyber_cells": [], "confidence": 0.0,
            "note": "geospatial_crime_pattern_intelligence_engine.py was not importable.",
        }

    try:
        network_adjusted_risk = network_package.get("risk_propagation", {}).get(
            "network_adjusted_risk", fraud_package.get("risk_score", 0.0)
        )

        record = notebook7.GeoCaseRecord(
            case_id=case.case_id,
            city=case.city,
            state=case.state,
            latitude=case.latitude,
            longitude=case.longitude,
            fraud_type=fraud_package.get("fraud_type"),
            risk_score=network_adjusted_risk,
            amount_involved=case.amount_involved,
            timestamp=case.timestamp,
            campaign_id=_find_case_campaign_id(case.case_id, network_package),
            community_id=_find_case_community_id(case.case_id, network_package),
            is_mule_location=_case_touches_mule_account(case.case_id, network_package),
        )
        CASE_REGISTRY.register_geo_case(case.case_id, record)

        package = notebook7.analyze_geospatial_intelligence(
            CASE_REGISTRY.all_geo_cases(),
            save_visualization=True,
            generate_report=False,   # Notebook 8 produces its own single merged report
        )
        package["engine_source"] = EngineSource.REAL.value
        return package
    except Exception as exc:
        logger.warning("Notebook 7 raised at call time; falling back to stub geospatial package. error=%s", exc)
        return {
            "engine_source": EngineSource.STUB_FALLBACK.value,
            "hotspots": [], "district_risk": [], "predicted_hotspots": [], "campaign_spread": [],
            "resource_recommendations": [], "nearest_cyber_cells": [], "confidence": 0.0,
        }

## 11. Threat Fusion Engine

In [ ]:
# ============================================================================
# 11. Threat Fusion Engine
# ============================================================================


def _counterfeit_signal_score(counterfeit_package: Optional[Dict[str, Any]]) -> float:
    '''Converts the counterfeit package (real or stub) into a 0-100 threat signal. No evidence -> neutral 0.'''
    if not counterfeit_package:
        return 0.0
    return round(counterfeit_package.get("max_suspicion_score", 0.0) * 100, 1)


def _geo_signal_score(case_id: str, geo_package: Dict[str, Any]) -> float:
    '''
    Converts geospatial context into a 0-100 threat signal: the priority
    of any hotspot this case falls in, otherwise the priority of this
    case's district, otherwise a neutral 0.
    '''
    for hotspot in geo_package.get("hotspots", []):
        if case_id in hotspot.get("case_ids", []):
            return CONFIG.PRIORITY_TO_SCORE.get(hotspot["priority"], 0.0)
    for district in geo_package.get("district_risk", []):
        if case_id in district.get("case_ids", []):
            return CONFIG.PRIORITY_TO_SCORE.get(district["priority"], 0.0)
    return 0.0


def fuse_threat_score(
    case_id: str,
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    '''
    Threat Fusion Engine. Blends four independent signals into one 0-100
    overall threat score, rather than trusting any single engine's number
    in isolation. Every component is reported alongside the final score
    so the fusion is auditable, not a black box.
    '''
    fraud_risk = fraud_package.get("risk_score", 0.0)
    network_risk = network_package.get("risk_propagation", {}).get("network_adjusted_risk", fraud_risk)
    geo_signal = _geo_signal_score(case_id, geo_package)
    counterfeit_signal = _counterfeit_signal_score(counterfeit_package)

    fused = (
        CONFIG.FUSION_WEIGHT_NETWORK_RISK * network_risk
        + CONFIG.FUSION_WEIGHT_FRAUD_RISK * fraud_risk
        + CONFIG.FUSION_WEIGHT_GEO_SIGNAL * geo_signal
        + CONFIG.FUSION_WEIGHT_COUNTERFEIT_SIGNAL * counterfeit_signal
    )
    fused = round(min(100.0, fused), 1)

    return {
        "overall_threat_score": fused,
        "severity": _severity_from_risk(fused),
        "components": {
            "fraud_standalone_risk": fraud_risk,
            "network_adjusted_risk": network_risk,
            "geospatial_signal": geo_signal,
            "counterfeit_signal": counterfeit_signal,
        },
        "weights": {
            "network_adjusted_risk": CONFIG.FUSION_WEIGHT_NETWORK_RISK,
            "fraud_standalone_risk": CONFIG.FUSION_WEIGHT_FRAUD_RISK,
            "geospatial_signal": CONFIG.FUSION_WEIGHT_GEO_SIGNAL,
            "counterfeit_signal": CONFIG.FUSION_WEIGHT_COUNTERFEIT_SIGNAL,
        },
    }

## 12. Confidence Fusion Engine

In [ ]:
# ============================================================================
# 12. Confidence Fusion Engine
# ============================================================================


def fuse_confidence(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    '''
    Weighted confidence fusion. A component that produced no signal (e.g.
    no communities detected, no counterfeit evidence submitted) is
    excluded from the blend and the remaining weights are re-normalized,
    rather than silently treating "no signal" as "zero confidence".
    '''
    components: Dict[str, float] = {"fraud": fraud_package.get("confidence", 0.0)}
    weights: Dict[str, float] = {"fraud": CONFIG.CONFIDENCE_WEIGHT_FRAUD}

    communities = network_package.get("communities", [])
    if communities:
        components["network"] = communities[0].get("confidence", 0.0)
        weights["network"] = CONFIG.CONFIDENCE_WEIGHT_NETWORK

    if geo_package.get("confidence"):
        components["geospatial"] = geo_package["confidence"]
        weights["geospatial"] = CONFIG.CONFIDENCE_WEIGHT_GEO

    if counterfeit_package:
        certainty = abs(counterfeit_package.get("max_suspicion_score", 0.5) - 0.5) * 2 * 100
        components["counterfeit"] = round(certainty, 1)
        weights["counterfeit"] = CONFIG.CONFIDENCE_WEIGHT_COUNTERFEIT

    total_weight = sum(weights.values()) or 1.0
    fused = sum(components[k] * weights[k] for k in components) / total_weight

    return {
        "overall_confidence": round(min(99.0, fused), 1),
        "components": components,
        "weights_used": weights,
    }

## 13. Cross-Notebook Validation

In [ ]:
# ============================================================================
# 13. Cross-Notebook Validation
# ============================================================================


def validate_cross_notebook_consistency(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Compares Notebook 2's classified fraud_type against Notebook 6's
    dominant community fraud type and Notebook 7's dominant hotspot fraud
    type. A single engine disagreeing is common and not flagged (small
    samples vary); TWO OR MORE independent engines disagreeing with
    Notebook 2 is treated as a genuine conflict that should force human
    review rather than an automatic high-confidence decision.
    '''
    fraud_type = fraud_package.get("fraud_type")
    conflicts: List[str] = []

    communities = network_package.get("communities", [])
    if communities and communities[0].get("dominant_fraud") not in (None, "Unclassified", fraud_type):
        conflicts.append(
            f"Notebook 2 classified this case as '{fraud_type}', but the network community "
            f"it belongs to is dominated by '{communities[0]['dominant_fraud']}'."
        )

    hotspots = geo_package.get("hotspots", [])
    if hotspots and hotspots[0].get("dominant_fraud") not in (None, "Unclassified", fraud_type):
        conflicts.append(
            f"Notebook 2 classified this case as '{fraud_type}', but the geographic hotspot "
            f"it falls in is dominated by '{hotspots[0]['dominant_fraud']}'."
        )

    return {
        "is_consistent": len(conflicts) < 2,
        "conflicts_found": conflicts,
        "force_human_review": len(conflicts) >= 2,
    }

## 14. Module 8 - Final Decision (real Notebook 3 call, stub fallback)

In [ ]:
# ============================================================================
# 14. Module 8 - Final Decision (real Notebook 3 call, stub fallback)
# ============================================================================


class DecisionCategory(str, Enum):
    EMERGENCY = "Emergency"
    URGENT_ACTION = "Urgent Action"
    NEEDS_HUMAN_REVIEW = "Needs Human Review"
    AWARENESS_ONLY = "Awareness Only"
    NO_ACTION = "No Action - Benign"


def _decision_stub(
    fraud_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
    fused_risk: float,
    validation: Dict[str, Any],
) -> Dict[str, Any]:
    '''Deterministic fallback decision policy, used only when Notebook 3 is unavailable or errors.'''
    reasons: List[str] = [f"Fused overall threat score is {fused_risk}."]

    if validation["force_human_review"]:
        category = DecisionCategory.NEEDS_HUMAN_REVIEW
        reasons.append("Cross-notebook validation found conflicting fraud-type signals; routed to human review.")
    elif fused_risk >= CONFIG.DECISION_EMERGENCY_RISK_MIN:
        category = DecisionCategory.EMERGENCY
    elif fused_risk >= CONFIG.DECISION_URGENT_RISK_MIN:
        category = DecisionCategory.URGENT_ACTION
    elif fused_risk >= CONFIG.DECISION_HUMAN_REVIEW_RISK_MIN:
        category = DecisionCategory.AWARENESS_ONLY
    else:
        category = DecisionCategory.NO_ACTION

    if counterfeit_package and counterfeit_package.get("verdict") == "Likely Counterfeit":
        reasons.append("Submitted currency evidence was flagged as likely counterfeit.")

    stakeholder_actions = {
        "citizen": [
            "File a formal complaint on cybercrime.gov.in if not already done.",
            "Do not make any further payments to the numbers/accounts involved.",
        ],
        "police": [], "bank": [], "telecom": [],
    }
    if category in (DecisionCategory.EMERGENCY, DecisionCategory.URGENT_ACTION):
        stakeholder_actions["police"].append("Prioritize investigation; cross-reference network and geospatial findings.")

    return {
        "engine_source": EngineSource.STUB.value if not _NOTEBOOK3_AVAILABLE else EngineSource.STUB_FALLBACK.value,
        "case_decision": category.value,
        "priority": category.value,
        "reasons": reasons,
        "citizen_actions": stakeholder_actions["citizen"],
        "police_actions": stakeholder_actions["police"],
        "bank_actions": stakeholder_actions["bank"],
        "telecom_actions": stakeholder_actions["telecom"],
    }


def run_decision_engine(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    counterfeit_package: Optional[Dict[str, Any]],
    fusion_result: Dict[str, Any],
    validation: Dict[str, Any],
) -> Dict[str, Any]:
    '''
    Module 8 entry point (Notebook 3 adapter). Feeds Notebook 3 the FUSED
    threat score and severity (not the raw Notebook-2 standalone numbers),
    so its policy engine reasons over the same number the rest of this
    package reports as the overall threat level.
    '''
    if _NOTEBOOK3_AVAILABLE:
        try:
            adjusted_input = dict(fraud_package)
            adjusted_input["risk_score"] = fusion_result["overall_threat_score"]
            adjusted_input["severity"] = fusion_result["severity"]

            similar_cases = [
                {"case_id": cid, "similarity": 0.75}
                for cid in network_package.get("connected_cases", [])
            ]

            result = notebook3.build_decision_package(adjusted_input, similar_cases=similar_cases or None)
            result["engine_source"] = EngineSource.REAL.value

            if validation["force_human_review"] and result.get("case_decision") != notebook3.CaseDecision.NEEDS_HUMAN_REVIEW.value:
                result["case_decision"] = notebook3.CaseDecision.NEEDS_HUMAN_REVIEW.value
                result.setdefault("policy_reasons", []).append(
                    "Overridden by Notebook 8: cross-notebook validation found conflicting fraud-type signals."
                )
            return result
        except Exception as exc:
            logger.warning("Notebook 3 raised at call time; falling back to stub decision policy. error=%s", exc)

    return _decision_stub(fraud_package, counterfeit_package, fusion_result["overall_threat_score"], validation)


def _decision_category_label(decision_package: Dict[str, Any]) -> str:
    '''Normalizes whichever key the real Notebook 3 or the stub used for the final category.'''
    return decision_package.get("case_decision") or decision_package.get("decision_category") or "Unknown"


def _decision_reasons(decision_package: Dict[str, Any]) -> List[str]:
    return decision_package.get("policy_reasons") or decision_package.get("reasons") or []


def _stakeholder_actions(decision_package: Dict[str, Any], stakeholder: str) -> List[str]:
    '''Normalizes stakeholder action lookup across the real Notebook 3 shape and the stub shape.'''
    direct_key = f"{stakeholder}_actions"
    if direct_key in decision_package:
        return decision_package[direct_key]
    return decision_package.get("stakeholder_actions", {}).get(stakeholder, [])

## 15. Explainability Chain

In [ ]:
# ============================================================================
# 15. Explainability Chain
# ============================================================================


def build_explainability(
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    fusion_result: Dict[str, Any],
    validation: Dict[str, Any],
    decision_package: Dict[str, Any],
) -> List[Dict[str, str]]:
    '''Module 16 entry point. A short, ordered "why" chain a reviewer can read in one glance.'''
    chain: List[Dict[str, str]] = []
    chain.append({"step": "Fraud type identified", "detail": fraud_package.get("fraud_type", "Unclassified")})
    for reason in fraud_package.get("reasoning", []):
        chain.append({"step": "Evidence signal", "detail": reason})

    risk_info = network_package.get("risk_propagation", {})
    if risk_info.get("boost_applied", 0) > 0:
        chain.append({
            "step": "Network context raised risk",
            "detail": f"{risk_info['original_risk']} -> {risk_info['network_adjusted_risk']} (driven by {risk_info.get('driving_entity')})",
        })

    hotspots = geo_package.get("hotspots", [])
    if hotspots:
        chain.append({
            "step": "Geographic context",
            "detail": f"Case falls within hotspot {hotspots[0]['hotspot_id']} ({hotspots[0]['primary_city']}), priority {hotspots[0]['priority']}.",
        })

    chain.append({
        "step": "Threat fusion",
        "detail": f"Overall threat score {fusion_result['overall_threat_score']} ({fusion_result['severity']}), "
                  f"blended from fraud/network/geo/counterfeit signals.",
    })

    if validation["conflicts_found"]:
        for conflict in validation["conflicts_found"]:
            chain.append({"step": "Validation conflict", "detail": conflict})

    for reason in _decision_reasons(decision_package):
        chain.append({"step": "Decision factor", "detail": reason})

    chain.append({"step": "Final decision", "detail": _decision_category_label(decision_package)})
    return chain

## 16. Incident Timeline

In [ ]:
# ============================================================================
# 16. Incident Timeline
# ============================================================================


def build_incident_timeline(
    case: CaseIntake,
    evidence_package: Dict[str, Any],
    fraud_package: Dict[str, Any],
    network_package: Dict[str, Any],
    geo_package: Dict[str, Any],
    decision_package: Dict[str, Any],
) -> List[Dict[str, str]]:
    '''
    Reconstructs a plain-language, chronological view of how this case
    moved through the platform - useful for an investigator or auditor to
    see when each stage fired.
    '''
    timeline: List[Dict[str, str]] = []
    base_time = case.timestamp or datetime.now(timezone.utc).isoformat()

    timeline.append({"event": f"Case {case.case_id} received via {case.source}.", "timestamp": base_time})
    timeline.append({"event": f"Evidence normalized ({len(case.evidence)} item(s)).", "timestamp": base_time})
    timeline.append({
        "event": f"Fraud classified as '{fraud_package.get('fraud_type')}' "
                 f"(standalone risk {fraud_package.get('risk_score')}).",
        "timestamp": base_time,
    })

    if network_package.get("connected_cases"):
        timeline.append({
            "event": f"Matched to {len(network_package['connected_cases'])} prior case(s) in the fraud network.",
            "timestamp": base_time,
        })
    if network_package.get("money_mule_accounts"):
        timeline.append({"event": "Linked account(s) flagged as likely money mule.", "timestamp": base_time})

    if geo_package.get("hotspots"):
        for hotspot in geo_package["hotspots"]:
            if case.case_id in hotspot.get("case_ids", []):
                timeline.append({"event": f"Case falls within geographic hotspot {hotspot['hotspot_id']}.", "timestamp": base_time})
                break

    timeline.append({
        "event": f"Final decision: {_decision_category_label(decision_package)}.",
        "timestamp": base_time,
    })
    return timeline

## 17. Engine Health Dashboard

In [ ]:
# ============================================================================
# 17. Engine Health Dashboard
# ============================================================================


def build_engine_health(audit_trail: List[Dict[str, Any]]) -> Dict[str, Any]:
    '''
    One place to see, per pipeline stage, whether the real engine ran, a
    stub ran by design, or a stub ran because the real engine failed at
    call time - the last case is the one worth watching in a live
    deployment.
    '''
    health: Dict[str, Any] = {}
    for entry in audit_trail:
        stage = entry["stage"]
        source = entry.get("engine_source", EngineSource.SKIPPED.value)
        if source == EngineSource.REAL.value:
            status = "Healthy (real engine)"
        elif source == EngineSource.STUB.value:
            status = "Stub by design (module not installed / not applicable)"
        elif source == EngineSource.STUB_FALLBACK.value:
            status = "DEGRADED - real engine failed at runtime, stub fallback used"
        else:
            status = "Skipped"
        health[stage] = {"status": status, "engine_source": source, "duration_ms": entry.get("duration_ms")}
    return health

## 18. Modules 10-14 - Audience-Specific Dashboards

In [ ]:
# ============================================================================
# 18. Modules 10-14 - Audience-Specific Dashboards
# ============================================================================


def _risk_band(score: float) -> str:
    if score >= CONFIG.DECISION_EMERGENCY_RISK_MIN:
        return "Critical"
    if score >= CONFIG.DECISION_URGENT_RISK_MIN:
        return "High"
    if score >= CONFIG.DECISION_HUMAN_REVIEW_RISK_MIN:
        return "Medium"
    return "Low"


def build_citizen_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 11 entry point. Citizens see outcomes and guidance, not investigative internals.'''
    fraud = master["fraud_intelligence"]
    decision = master["decision_intelligence"]
    return {
        "case_id": master["case"]["case_id"],
        "fraud_type": fraud.get("fraud_type"),
        "risk_level": _risk_band(master["overall_threat_level"]),
        "what_this_means": _decision_category_label(decision),
        "recommended_actions": _stakeholder_actions(decision, "citizen"),
        "national_cyber_crime_helpline": "1930",
        "national_cyber_crime_portal": "cybercrime.gov.in",
        "safety_advice": [
            "Do not share OTPs, passwords, or remote-access app codes with anyone.",
            "Government agencies do not conduct arrests or investigations over video call.",
            "When in doubt, hang up and call the organization back using an officially published number.",
        ],
    }


def build_police_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 12 entry point. Police get the full investigative picture.'''
    return {
        "case_id": master["case"]["case_id"],
        "fraud_intelligence": master["fraud_intelligence"],
        "fraud_network_intelligence": master["fraud_network_intelligence"],
        "geospatial_intelligence": master["geospatial_intelligence"],
        "counterfeit_intelligence": master["counterfeit_intelligence"],
        "decision_intelligence": master["decision_intelligence"],
        "explainability": master["explainability"],
        "incident_timeline": master["incident_timeline"],
        "validation": master["cross_notebook_validation"],
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "police"),
    }


def build_bank_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 13 entry point. Banks see financial-entity risk only.'''
    network = master["fraud_network_intelligence"]
    return {
        "case_id": master["case"]["case_id"],
        "money_mule_accounts": network.get("money_mule_accounts", []),
        "network_adjusted_risk": network.get("risk_propagation", {}).get("network_adjusted_risk"),
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "bank"),
    }


def build_telecom_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''Module 14 entry point. Telecom providers see phone/campaign risk only.'''
    entities = master["evidence"].get("metadata") or master["evidence"].get("extracted_entities") or {}
    network = master["fraud_network_intelligence"]
    return {
        "case_id": master["case"]["case_id"],
        "phone_numbers": entities.get("phone_numbers", []),
        "linked_campaigns": [c["campaign_id"] for c in network.get("fraud_campaigns", [])],
        "recommended_actions": _stakeholder_actions(master["decision_intelligence"], "telecom"),
    }


def build_administrator_dashboard(master: Dict[str, Any]) -> Dict[str, Any]:
    '''
    Fifth audience view. Administrators care about platform health and
    pipeline performance, not case-specific investigative detail.

    IMPORTANT: this function requires master["audit_trail"],
    master["execution_statistics"], and master["engine_health"] to
    already be populated. The caller (process_case) MUST set those three
    keys on `master` BEFORE calling this function - that ordering is the
    fix for the Revision 3 KeyError bug.
    '''
    return {
        "case_id": master["case"]["case_id"],
        "engine_availability": master["engine_availability"],
        "engine_health": master["engine_health"],
        "audit_trail": master["audit_trail"],
        "execution_statistics": master["execution_statistics"],
        "total_cases_in_registry": CASE_REGISTRY.total_cases(),
        "cross_notebook_validation": master["cross_notebook_validation"],
    }

## 19. Executive Summary

In [ ]:
# ============================================================================
# 19. Executive Summary
# ============================================================================


def build_executive_summary(master: Dict[str, Any]) -> Dict[str, Any]:
    '''
    One-screen summary an officer should be able to read in under 30
    seconds: what happened, how bad it is, and what is already known
    about it from the network and geography.
    '''
    network = master["fraud_network_intelligence"]
    geo = master["geospatial_intelligence"]
    fusion = master["threat_fusion"]

    top_community = network.get("communities", [{}])[0] if network.get("communities") else None
    top_hotspot = geo.get("hotspots", [{}])[0] if geo.get("hotspots") else None

    return {
        "case_id": master["case"]["case_id"],
        "fraud_type": master["fraud_intelligence"].get("fraud_type"),
        "threat_level": fusion["severity"],
        "overall_threat_score": fusion["overall_threat_score"],
        "overall_confidence": master["overall_confidence"],
        "amount_involved": master["case"]["amount_involved"],
        "connected_cases_count": len(network.get("connected_cases", [])),
        "campaign_id": network.get("fraud_campaigns", [{}])[0].get("campaign_id") if network.get("fraud_campaigns") else None,
        "money_mule_flagged": bool(network.get("money_mule_accounts")),
        "hotspot": top_hotspot["hotspot_id"] if top_hotspot else None,
        "hotspot_city": top_hotspot["primary_city"] if top_hotspot else None,
        "community_dominant_fraud": top_community.get("dominant_fraud") if top_community else None,
        "decision": _decision_category_label(master["decision_intelligence"]),
        "validation_conflicts": master["cross_notebook_validation"]["conflicts_found"],
    }

## 20. Platform Dashboard (compact, presentation-style summary)

In [ ]:
# ============================================================================
# 20. Platform Dashboard (compact, presentation-style summary)
# ============================================================================


def build_platform_dashboard_text(master: Dict[str, Any]) -> str:
    '''Renders a compact, box-drawn summary block suitable for a terminal demo or a report cover page.'''
    exec_summary = master["executive_summary"]
    bar = "=" * 56
    lines = [
        bar,
        "DIGITAL PUBLIC SAFETY PLATFORM",
        bar,
        f"Case ID           : {exec_summary['case_id']}",
        f"Threat Level      : {exec_summary['threat_level'].upper()}",
        f"Fraud Type        : {exec_summary['fraud_type']}",
        f"Overall Threat    : {exec_summary['overall_threat_score']}/100",
        f"Confidence        : {exec_summary['overall_confidence']}%",
        f"Money Mule Flag   : {'YES' if exec_summary['money_mule_flagged'] else 'No'}",
        f"Campaign          : {exec_summary['campaign_id'] or 'None identified'}",
        f"Connected Cases   : {exec_summary['connected_cases_count']}",
        f"Hotspot           : {exec_summary['hotspot'] or 'None'} ({exec_summary['hotspot_city'] or 'n/a'})",
        f"Decision          : {exec_summary['decision']}",
        bar,
    ]
    return "\n".join(lines)

## 21. Final Intelligence Report (PDF, with plain-text fallback)

In [ ]:
# ============================================================================
# 21. Final Intelligence Report (PDF, with plain-text fallback)
# ============================================================================


def _build_final_report_text_lines(master: Dict[str, Any]) -> List[str]:
    lines: List[str] = []
    lines.append(master["platform_dashboard_text"])
    lines.append("")

    lines.append("EXECUTIVE SUMMARY")
    for key, value in master["executive_summary"].items():
        lines.append(f"  {key}: {value}")
    lines.append("")

    lines.append("EXPLAINABILITY")
    for step in master["explainability"]:
        lines.append(f"  [{step['step']}] {step['detail']}")
    lines.append("")

    lines.append("INCIDENT TIMELINE")
    for event in master["incident_timeline"]:
        lines.append(f"  {event['timestamp']}  -  {event['event']}")
    lines.append("")

    if master["cross_notebook_validation"]["conflicts_found"]:
        lines.append("VALIDATION CONFLICTS")
        for conflict in master["cross_notebook_validation"]["conflicts_found"]:
            lines.append(f"  - {conflict}")
        lines.append("")

    lines.append("FRAUD INTELLIGENCE")
    lines.append(f"  Type: {master['fraud_intelligence'].get('fraud_type')}  |  Standalone risk: {master['fraud_intelligence'].get('risk_score')}")
    lines.append("")

    if master["counterfeit_intelligence"]:
        lines.append("COUNTERFEIT INTELLIGENCE")
        lines.append(f"  Verdict: {master['counterfeit_intelligence'].get('verdict')}")
        lines.append("")

    network = master["fraud_network_intelligence"]
    lines.append("FRAUD NETWORK INTELLIGENCE")
    lines.append(f"  Connected cases: {len(network.get('connected_cases', []))}")
    lines.append(f"  Communities: {len(network.get('communities', []))}  |  Money mule accounts: {len(network.get('money_mule_accounts', []))}")
    lines.append("")

    geo = master["geospatial_intelligence"]
    lines.append("GEOSPATIAL INTELLIGENCE")
    lines.append(f"  Hotspots: {len(geo.get('hotspots', []))}  |  Districts monitored: {len(geo.get('district_risk', []))}")
    lines.append("")

    lines.append("THREAT FUSION")
    fusion = master["threat_fusion"]
    lines.append(f"  Overall threat score: {fusion['overall_threat_score']} ({fusion['severity']})")
    for component, value in fusion["components"].items():
        lines.append(f"    {component}: {value}  (weight {fusion['weights'][component]})")
    lines.append("")

    decision = master["decision_intelligence"]
    lines.append("DECISION")
    lines.append(f"  Category: {_decision_category_label(decision)}")
    for reason in _decision_reasons(decision):
        lines.append(f"  Reason: {reason}")
    lines.append("")

    lines.append("ENGINE HEALTH")
    for stage, health in master["engine_health"].items():
        lines.append(f"  {stage:36s} | {health['status']}")
    lines.append("")

    lines.append("AUDIT TRAIL")
    for entry in master["audit_trail"]:
        lines.append(f"  {entry['stage']:36s} | {entry['engine_source']:36s} | {entry['duration_ms']} ms")
    lines.append("")

    lines.append("EXECUTION STATISTICS")
    for stage, seconds in master["execution_statistics"]["stage_seconds"].items():
        lines.append(f"  {stage:36s} {seconds:.3f} sec")
    lines.append(f"  {'TOTAL':36s} {master['execution_statistics']['total_seconds']:.3f} sec")

    return lines


def generate_final_intelligence_report(master: Dict[str, Any], output_path: str) -> str:
    '''
    Module 15 entry point. One PDF (or plain-text fallback) covering:
    platform dashboard, executive summary, explainability, incident
    timeline, validation conflicts, per-engine intelligence, threat
    fusion, decision, engine health, audit trail, and execution stats.

    NOTE: this function reads master["audit_trail"],
    master["execution_statistics"], master["engine_health"],
    master["executive_summary"], and master["platform_dashboard_text"],
    so the caller MUST populate all of those keys on `master` before
    calling this function.
    '''
    lines = _build_final_report_text_lines(master)

    if not _REPORTLAB_AVAILABLE:
        fallback_path = os.path.splitext(output_path)[0] + ".txt"
        with open(fallback_path, "w", encoding="utf-8") as fh:
            fh.write("\n".join(lines))
        logger.info("reportlab not available; wrote plain-text final report to %s", fallback_path)
        return fallback_path

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle("ReportTitle", parent=styles["Title"], fontSize=16)
    heading_style = ParagraphStyle("ReportHeading", parent=styles["Heading2"], spaceBefore=10, spaceAfter=4)
    body_style = ParagraphStyle("ReportBody", parent=styles["BodyText"], spaceAfter=2)

    doc = SimpleDocTemplate(output_path, pagesize=A4, topMargin=1.5 * cm, bottomMargin=1.5 * cm)
    story: List[Any] = []

    story.append(Paragraph(f"Digital Public Safety Report - {master['package_id']}", title_style))
    story.append(Paragraph(f"Case: {master['case']['case_id']}", body_style))
    story.append(Spacer(1, 8))

    story.append(Paragraph("Executive Summary", heading_style))
    for key, value in master["executive_summary"].items():
        story.append(Paragraph(f"<b>{key}:</b> {value}", body_style))

    story.append(Paragraph("Explainability", heading_style))
    for step in master["explainability"]:
        story.append(Paragraph(f"<b>{step['step']}:</b> {step['detail']}", body_style))

    story.append(Paragraph("Incident Timeline", heading_style))
    for event in master["incident_timeline"]:
        story.append(Paragraph(f"{event['timestamp']} - {event['event']}", body_style))

    if master["cross_notebook_validation"]["conflicts_found"]:
        story.append(Paragraph("Validation Conflicts", heading_style))
        for conflict in master["cross_notebook_validation"]["conflicts_found"]:
            story.append(Paragraph(conflict, body_style))

    story.append(Paragraph("Fraud Intelligence", heading_style))
    story.append(Paragraph(
        f"Type: {master['fraud_intelligence'].get('fraud_type')} | Standalone risk: {master['fraud_intelligence'].get('risk_score')}",
        body_style,
    ))

    if master["counterfeit_intelligence"]:
        story.append(Paragraph("Counterfeit Intelligence", heading_style))
        story.append(Paragraph(f"Verdict: {master['counterfeit_intelligence'].get('verdict')}", body_style))

    network = master["fraud_network_intelligence"]
    story.append(Paragraph("Fraud Network Intelligence", heading_style))
    story.append(Paragraph(
        f"Connected cases: {len(network.get('connected_cases', []))} | "
        f"Communities: {len(network.get('communities', []))} | "
        f"Money mule accounts: {len(network.get('money_mule_accounts', []))}",
        body_style,
    ))

    geo = master["geospatial_intelligence"]
    story.append(Paragraph("Geospatial Intelligence", heading_style))
    story.append(Paragraph(
        f"Hotspots: {len(geo.get('hotspots', []))} | Districts monitored: {len(geo.get('district_risk', []))}",
        body_style,
    ))

    fusion = master["threat_fusion"]
    story.append(Paragraph("Threat Fusion", heading_style))
    story.append(Paragraph(f"Overall threat score: {fusion['overall_threat_score']} ({fusion['severity']})", body_style))
    fusion_rows = [["Component", "Value", "Weight"]]
    for component, value in fusion["components"].items():
        fusion_rows.append([component, str(value), str(fusion["weights"][component])])
    fusion_table = Table(fusion_rows, colWidths=[6 * cm, 4 * cm, 3 * cm])
    fusion_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
    ]))
    story.append(fusion_table)

    decision = master["decision_intelligence"]
    story.append(Paragraph("Decision", heading_style))
    story.append(Paragraph(f"Category: {_decision_category_label(decision)}", body_style))
    for reason in _decision_reasons(decision):
        story.append(Paragraph(f"Reason: {reason}", body_style))

    story.append(PageBreak())
    story.append(Paragraph("Engine Health", heading_style))
    health_rows = [["Stage", "Status"]]
    for stage, health in master["engine_health"].items():
        health_rows.append([stage, health["status"]])
    health_table = Table(health_rows, colWidths=[6 * cm, 9 * cm])
    health_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ]))
    story.append(health_table)

    story.append(Paragraph("Audit Trail", heading_style))
    audit_rows = [["Stage", "Engine", "Duration (ms)"]]
    for entry in master["audit_trail"]:
        audit_rows.append([entry["stage"], entry["engine_source"], str(entry["duration_ms"])])
    audit_table = Table(audit_rows, colWidths=[6 * cm, 6 * cm, 3 * cm])
    audit_table.setStyle(TableStyle([
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("BACKGROUND", (0, 0), (-1, 0), colors.whitesmoke),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
    ]))
    story.append(audit_table)

    story.append(Paragraph("Execution Statistics", heading_style))
    story.append(Paragraph(f"Total processing time: {master['execution_statistics']['total_seconds']:.3f} sec", body_style))

    doc.build(story)
    logger.info("Final intelligence report saved to %s", output_path)
    return output_path

## 22. Timing / audit wrapper

In [ ]:
# ============================================================================
# 22. Timing / audit wrapper
# ============================================================================


def _run_stage(
    stage_name: str,
    engine_source: str,
    fn: Callable[[], Any],
    audit_trail: List[Dict[str, Any]],
    stage_seconds: Dict[str, float],
    required: bool = True,
) -> Any:
    '''Shared timing/audit wrapper for every pipeline stage.'''
    start = time.perf_counter()
    try:
        result = fn()
    except Exception as exc:
        duration_ms = round((time.perf_counter() - start) * 1000, 2)
        audit_trail.append({"stage": stage_name, "status": "Failed", "engine_source": engine_source, "duration_ms": duration_ms})
        stage_seconds[stage_name] = round((time.perf_counter() - start), 4)
        if required:
            raise OrchestrationError(f"Stage '{stage_name}' failed: {exc}") from exc
        logger.warning("Optional stage '%s' failed and was skipped: %s", stage_name, exc)
        return None

    duration_ms = round((time.perf_counter() - start) * 1000, 2)
    actual_source = result.get("engine_source", engine_source) if isinstance(result, dict) else engine_source
    audit_trail.append({
        "stage": stage_name,
        "status": "Completed" if result is not None else "Skipped",
        "engine_source": actual_source if result is not None else EngineSource.SKIPPED.value,
        "duration_ms": duration_ms,
    })
    stage_seconds[stage_name] = round((time.perf_counter() - start), 4)
    return result

## 23. Module 9 - Merge Everything / Orchestration Entry Point

In [ ]:
# ============================================================================
# 23. Module 9 - Merge Everything / Orchestration Entry Point
# ============================================================================


def process_case(raw_case: CaseIntake, report_dir: str = "/tmp/notebook8_reports") -> Dict[str, Any]:
    '''
    Notebook 8 orchestration.

    Ordering (this is the fix for the Revision 3 bug): every key that
    any dashboard-builder reads from `master` is populated BEFORE any
    dashboard is built. Specifically:

      1. Run the analysis pipeline (intake -> evidence -> fraud ->
         counterfeit -> network -> geo -> fusion -> validation ->
         decision -> explainability).
      2. Build the base `master` dict (case, packages, fusion,
         validation, decision, explainability, engine_availability).
      3. Build incident_timeline.
      4. Populate audit_trail and execution_statistics (interim total).
      5. Build engine_health (needs audit_trail).
      6. Build executive_summary and platform_dashboard_text.
      7. ONLY NOW build the five audience dashboards - citizen, police,
         bank, telecom, administrator - since all five, and especially
         the administrator dashboard, may read audit_trail,
         execution_statistics, and engine_health.
      8. Generate the final PDF/text report (reads the same keys).
      9. Refresh audit_trail-derived fields (engine_health,
         execution_statistics total) once more, since report generation
         itself added one more audit-trail entry.
    '''
    pipeline_start = time.perf_counter()
    audit_trail: List[Dict[str, Any]] = []
    stage_seconds: Dict[str, float] = {}

    # --- Step 1: analysis pipeline ---
    case = _run_stage("Case Intake", EngineSource.REAL.value, lambda: intake_case(raw_case), audit_trail, stage_seconds)
    routing_plan = _run_stage("Evidence Routing", EngineSource.REAL.value, lambda: route_evidence(case), audit_trail, stage_seconds)

    evidence_package = _run_stage(
        "Notebook 4 - Evidence Intelligence", EngineSource.REAL.value,
        lambda: run_evidence_engine(case), audit_trail, stage_seconds,
    )

    fraud_package = _run_stage(
        "Notebook 2 - Fraud Intelligence", EngineSource.REAL.value,
        lambda: run_fraud_intelligence(case, evidence_package), audit_trail, stage_seconds,
    )

    counterfeit_package = None
    if routing_plan["run_counterfeit_check"]:
        counterfeit_package = _run_stage(
            "Notebook 5 - Counterfeit Intelligence", EngineSource.REAL.value,
            lambda: run_counterfeit_check(case), audit_trail, stage_seconds, required=False,
        )

    network_package = _run_stage(
        "Notebook 6 - Fraud Network Intelligence", EngineSource.REAL.value,
        lambda: run_network_intelligence(case, evidence_package, fraud_package), audit_trail, stage_seconds,
    )

    geo_package: Dict[str, Any] = {}
    if routing_plan["run_geospatial_intelligence"]:
        geo_package = _run_stage(
            "Notebook 7 - Geospatial Intelligence", EngineSource.REAL.value,
            lambda: run_geospatial_intelligence(case, fraud_package, network_package), audit_trail, stage_seconds,
        ) or {}

    threat_fusion = _run_stage(
        "Threat Fusion Engine", EngineSource.REAL.value,
        lambda: fuse_threat_score(case.case_id, fraud_package, network_package, geo_package, counterfeit_package),
        audit_trail, stage_seconds,
    )

    confidence_fusion = _run_stage(
        "Confidence Fusion Engine", EngineSource.REAL.value,
        lambda: fuse_confidence(fraud_package, network_package, geo_package, counterfeit_package),
        audit_trail, stage_seconds,
    )

    validation = _run_stage(
        "Cross-Notebook Validation", EngineSource.REAL.value,
        lambda: validate_cross_notebook_consistency(fraud_package, network_package, geo_package),
        audit_trail, stage_seconds,
    )

    decision_package = _run_stage(
        "Notebook 3 - Decision Intelligence", EngineSource.REAL.value,
        lambda: run_decision_engine(fraud_package, network_package, counterfeit_package, threat_fusion, validation),
        audit_trail, stage_seconds,
    )

    explainability = _run_stage(
        "Explainability", EngineSource.REAL.value,
        lambda: build_explainability(fraud_package, network_package, geo_package, threat_fusion, validation, decision_package),
        audit_trail, stage_seconds,
    )

    # --- Step 2: base master dict ---
    case_dict = {
        "case_id": case.case_id, "citizen_name": case.citizen_name, "victim_id": case.victim_id,
        "timestamp": case.timestamp, "city": case.city, "state": case.state,
        "priority": case.priority, "source": case.source, "amount_involved": case.amount_involved,
    }

    master: Dict[str, Any] = {
        "package_id": f"DPSP-{datetime.now(timezone.utc).year}-{uuid.uuid4().hex[:6].upper()}",
        "case": case_dict,
        "evidence_routing_plan": routing_plan,
        "evidence": evidence_package,
        "fraud_intelligence": fraud_package,
        "counterfeit_intelligence": counterfeit_package,
        "fraud_network_intelligence": network_package,
        "geospatial_intelligence": geo_package,
        "threat_fusion": threat_fusion,
        "confidence_fusion": confidence_fusion,
        "cross_notebook_validation": validation,
        "decision_intelligence": decision_package,
        "explainability": explainability,
        "overall_threat_level": threat_fusion["overall_threat_score"],
        "overall_confidence": confidence_fusion["overall_confidence"],
        "engine_availability": {
            "notebook2_fraud_intelligence": _NOTEBOOK2_AVAILABLE,
            "notebook3_decision_intelligence": _NOTEBOOK3_AVAILABLE,
            "notebook4_evidence_intelligence": _NOTEBOOK4_AVAILABLE,
            "notebook5_counterfeit_intelligence": _NOTEBOOK5_AVAILABLE,
            "notebook6_network_intelligence": _NOTEBOOK6_AVAILABLE,
            "notebook7_geospatial_intelligence": _NOTEBOOK7_AVAILABLE,
        },
    }

    # --- Step 3: incident timeline ---
    master["incident_timeline"] = build_incident_timeline(
        case, evidence_package, fraud_package, network_package, geo_package, decision_package
    )

    # --- Step 4: audit_trail / execution_statistics (interim total) ---
    # THIS MUST HAPPEN BEFORE ANY DASHBOARD IS BUILT. This ordering is the
    # actual fix for the Revision 3 KeyError: 'audit_trail' bug.
    interim_total_seconds = round(time.perf_counter() - pipeline_start, 4)
    master["audit_trail"] = audit_trail
    master["execution_statistics"] = {
        "stage_seconds": stage_seconds,
        "total_seconds": interim_total_seconds,
    }

    # --- Step 5: engine_health (needs audit_trail, which now exists) ---
    master["engine_health"] = build_engine_health(audit_trail)

    # --- Step 6: executive summary and platform dashboard text ---
    master["executive_summary"] = build_executive_summary(master)
    master["platform_dashboard_text"] = build_platform_dashboard_text(master)

    # --- Step 7: all five audience dashboards, now safe to build ---
    master["citizen_response"] = build_citizen_dashboard(master)
    master["police_response"] = build_police_dashboard(master)
    master["bank_response"] = build_bank_dashboard(master)
    master["telecom_response"] = build_telecom_dashboard(master)
    master["administrator_response"] = build_administrator_dashboard(master)

    # --- Step 8: final report ---
    os.makedirs(report_dir, exist_ok=True)
    report_path = _run_stage(
        "Final Intelligence Report", EngineSource.REAL.value,
        lambda: generate_final_intelligence_report(master, os.path.join(report_dir, f"digital_public_safety_report_{uuid.uuid4().hex[:8]}.pdf")),
        audit_trail, stage_seconds,
    )
    master["final_report"] = report_path

    # --- Step 9: refresh audit-trail-derived fields after the report stage ---
    master["engine_health"] = build_engine_health(audit_trail)
    total_seconds = round(time.perf_counter() - pipeline_start, 4)
    master["execution_statistics"]["total_seconds"] = total_seconds
    # Keep the administrator dashboard's copies in sync with the refreshed values.
    master["administrator_response"]["engine_health"] = master["engine_health"]
    master["administrator_response"]["execution_statistics"] = master["execution_statistics"]

    case_digest = hashlib.sha256(case.case_id.encode("utf-8")).hexdigest()
    master["audit"] = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "case_id_hash": case_digest,
        "notebook_version": CONFIG.NOTEBOOK_VERSION,
    }

    CASE_REGISTRY.register_master_package(case.case_id, master)

    logger.info(
        "Case processing complete. case_id=%s decision=%s threat_level=%s confidence=%s total_seconds=%.3f",
        case.case_id, _decision_category_label(decision_package),
        master["overall_threat_level"], master["overall_confidence"], total_seconds,
    )
    return master

## 24. Synthetic Case Generation and Deterministic Test Suite

In [2]:
# ============================================================================
# 24. Synthetic Case Generation and Deterministic Test Suite
# ============================================================================


def _build_synthetic_cases() -> List[CaseIntake]:
    base_time = datetime(2026, 7, 1, tzinfo=timezone.utc)
    cases: List[CaseIntake] = []

    # --- Three linked "Digital Arrest" cases, sharing a phone number and
    # UPI id, reported from Kolhapur then Pune. ---
    shared_phone = "9876543210"
    shared_upi = "rahul@okaxis"
    cities = ["Kolhapur", "Kolhapur", "Pune"]
    for i in range(1, 4):
        cases.append(CaseIntake(
            case_id=f"DPSP-CASE-{i:03d}",
            citizen_name=f"Citizen {i}",
            city=cities[i - 1],
            state="Maharashtra",
            timestamp=base_time.replace(day=min(28, base_time.day + i)).isoformat(),
            amount_involved=45000.0 + i * 5000,
            evidence=[
                EvidenceItem(
                    evidence_type="call_recording",
                    content=(
                        f"Caller claimed to be from CBI and RBI, said there is an arrest warrant against me, "
                        f"demanded a video call and payment via UPI to {shared_upi} to avoid digital arrest. "
                        f"Caller phone number was {shared_phone}."
                    ),
                ),
            ],
        ))

    # --- One case with a currency image (no real file on disk in this
    # test, so the counterfeit stub path is exercised). ---
    cases.append(CaseIntake(
        case_id="DPSP-CASE-CURRENCY-001",
        citizen_name="Citizen 4",
        city="Mumbai",
        state="Maharashtra",
        timestamp=base_time.replace(day=15).isoformat(),
        amount_involved=500.0,
        evidence=[
            EvidenceItem(evidence_type="text", content="I received this 500 rupee note as change and it feels off."),
            EvidenceItem(
                evidence_type="currency_image",
                content="note_scan_001.jpg",
                metadata={"citizen_reported_suspicious": True},
            ),
        ],
    ))

    # --- One low-signal case with no fraud keywords, no location, and a
    # tiny amount, to exercise the No Action decision path. ---
    cases.append(CaseIntake(
        case_id="DPSP-CASE-LOWSIGNAL-001",
        citizen_name="Citizen 5",
        timestamp=base_time.replace(day=20).isoformat(),
        amount_involved=200.0,
        evidence=[EvidenceItem(evidence_type="text", content="Just wanted to ask a general question about safe UPI usage.")],
    ))

    return cases


def run_notebook8_test_suite() -> Dict[str, Any]:
    print("=== Notebook 8 Test Suite: end-to-end Digital Public Safety Platform (Revision 4) ===\n")

    cases = _build_synthetic_cases()
    checks: List[bool] = []
    results: List[Dict[str, Any]] = []

    def _check(label: str, actual: Any, expected: Any) -> None:
        ok = actual == expected
        checks.append(ok)
        print(f"    [{'PASS' if ok else 'FAIL'}] {label}: expected={expected!r} actual={actual!r}")

    def _check_true(label: str, condition: bool) -> None:
        checks.append(condition)
        print(f"    [{'PASS' if condition else 'FAIL'}] {label}")

    print("--- Processing cases sequentially through the orchestrator ---")
    for case in cases:
        print(f"\nProcessing {case.case_id} ...")
        master = process_case(case)
        results.append(master)
        print(master["platform_dashboard_text"])

    ring_result = results[2]
    currency_result = results[3]
    low_signal_result = results[4]

    # --- Case intake / routing ---
    _check("first case id preserved", results[0]["case"]["case_id"], "DPSP-CASE-001")
    _check_true("currency case routed counterfeit check on", currency_result["evidence_routing_plan"]["run_counterfeit_check"])
    _check_true("linked-ring case routed counterfeit check off", ring_result["evidence_routing_plan"]["run_counterfeit_check"] is False)

    # --- Evidence engine ---
    entities = ring_result["evidence"].get("metadata") or ring_result["evidence"].get("extracted_entities") or {}
    _check_true("phone number extracted from ring case evidence", "9876543210" in entities.get("phone_numbers", []))

    # --- Fraud intelligence ---
    _check_true("low-signal case classified without a hard crash",
                low_signal_result["fraud_intelligence"].get("fraud_type") is not None)

    # --- Threat fusion ---
    fusion = ring_result["threat_fusion"]
    _check_true("threat fusion score is within 0-100", 0.0 <= fusion["overall_threat_score"] <= 100.0)
    _check_true("threat fusion reports all four components",
                set(fusion["components"].keys()) == {"fraud_standalone_risk", "network_adjusted_risk", "geospatial_signal", "counterfeit_signal"})

    # --- Confidence fusion ---
    conf = ring_result["confidence_fusion"]
    _check_true("confidence fusion score is within 0-100", 0.0 <= conf["overall_confidence"] <= 100.0)

    # --- Cross-notebook validation ---
    _check_true("validation result carries a boolean consistency flag", isinstance(ring_result["cross_notebook_validation"]["is_consistent"], bool))

    # --- Counterfeit intelligence ---
    _check_true("counterfeit check ran for currency case", currency_result["counterfeit_intelligence"] is not None)
    _check_true("counterfeit check did not run for ring case", ring_result["counterfeit_intelligence"] is None)

    # --- Decision intelligence ---
    _check_true("decision category was produced for every case",
                all(_decision_category_label(r["decision_intelligence"]) != "Unknown" for r in results))

    # --- Explainability / timeline ---
    _check_true("explainability chain ends with the final decision",
                ring_result["explainability"][-1]["detail"] == _decision_category_label(ring_result["decision_intelligence"]))
    _check_true("incident timeline is non-empty", len(ring_result["incident_timeline"]) > 0)

    # --- Executive summary / platform dashboard ---
    _check_true("executive summary carries a threat_level", "threat_level" in ring_result["executive_summary"])
    _check_true("platform dashboard text is a non-empty string", isinstance(ring_result["platform_dashboard_text"], str) and len(ring_result["platform_dashboard_text"]) > 0)

    # --- Dashboards (the key regression check for this revision) ---
    citizen_dash = ring_result["citizen_response"]
    police_dash = ring_result["police_response"]
    bank_dash = ring_result["bank_response"]
    telecom_dash = ring_result["telecom_response"]
    admin_dash = ring_result["administrator_response"]

    _check_true("citizen dashboard excludes raw network graph internals", "fraud_network_intelligence" not in citizen_dash)
    _check_true("citizen dashboard carries the national helpline", citizen_dash["national_cyber_crime_helpline"] == "1930")
    _check_true("police dashboard includes full network intelligence", "fraud_network_intelligence" in police_dash)
    _check_true("bank dashboard is scoped to financial entities only",
                set(bank_dash.keys()) == {"case_id", "money_mule_accounts", "network_adjusted_risk", "recommended_actions"})
    _check_true("telecom dashboard is scoped to phone/campaign info only",
                set(telecom_dash.keys()) == {"case_id", "phone_numbers", "linked_campaigns", "recommended_actions"})
    _check_true("administrator dashboard was built without a KeyError (Revision 4 regression test)",
                admin_dash is not None and "engine_health" in admin_dash and "audit_trail" in admin_dash
                and "execution_statistics" in admin_dash and "total_cases_in_registry" in admin_dash
                and "cross_notebook_validation" in admin_dash)
    _check_true("administrator dashboard's execution_statistics total accounts for report generation time",
                admin_dash["execution_statistics"]["total_seconds"] >= admin_dash["execution_statistics"]["stage_seconds"].get("Final Intelligence Report", 0.0))

    # --- Case registry ---
    _check("case registry has processed all synthetic cases", CASE_REGISTRY.total_cases(), len(cases))

    # --- Master package structure ---
    expected_keys = {
        "package_id", "case", "evidence_routing_plan", "evidence", "fraud_intelligence",
        "counterfeit_intelligence", "fraud_network_intelligence", "geospatial_intelligence",
        "threat_fusion", "confidence_fusion", "cross_notebook_validation", "decision_intelligence",
        "explainability", "incident_timeline", "engine_health", "executive_summary",
        "platform_dashboard_text", "overall_threat_level", "overall_confidence", "engine_availability",
        "citizen_response", "police_response", "bank_response", "telecom_response", "administrator_response",
        "final_report", "audit_trail", "execution_statistics", "audit",
    }
    _check_true("master package contains all expected top-level keys", expected_keys.issubset(set(ring_result.keys())))

    # --- Audit trail / performance statistics ---
    _check_true("audit trail has an entry for every pipeline stage", len(ring_result["audit_trail"]) >= 10)
    _check_true("execution statistics report a positive total time", ring_result["execution_statistics"]["total_seconds"] > 0)

    # --- Final report ---
    _check_true("final intelligence report file was generated", ring_result["final_report"] is not None)
    _check_true("final intelligence report file exists on disk", os.path.exists(ring_result["final_report"]))

    print(f"\nSUMMARY: {sum(checks)}/{len(checks)} checks passed\n")

    print("Overall results by case:")
    for m in results:
        print(
            f"  {m['case']['case_id']:26s} threat={m['overall_threat_level']:6.1f} "
            f"decision={_decision_category_label(m['decision_intelligence']):20s} "
            f"confidence={m['overall_confidence']}%"
        )

    print(f"\nEngine availability: {json.dumps(ring_result['engine_availability'], indent=2)}")
    print(f"\nEngine health for {ring_result['case']['case_id']}:")
    for stage, health in ring_result["engine_health"].items():
        print(f"  {stage:36s} | {health['status']}")

    print(f"\nFinal report file: {ring_result['final_report']}")

    return {"results": results, "checks_passed": sum(checks), "checks_total": len(checks)}


if __name__ == "__main__":
    run_notebook8_test_suite()

NameError: name 'List' is not defined